# Stage 05-08 + RACAF: Joint Training (production notebook)

Trains Stage 05 (local features), Stage 06 (global features), Stage 07 (adaptive cross-attention), RACAF's trainable parameters and CORN **jointly, as one model**, through CORN's ordinal loss (`corn.corn_loss`) alone -- no auxiliary loss. Design: `JOINT_TRAINING_ARCHITECTURE.md` (locked, §27: this is the only joint-training notebook; do not create a second one).

Inputs: the committed APTOS 2019 split (`dataset_splits/aptos2019_train_val_split.csv`, 2929 train / 733 val), the frozen Stage 3 (vessel) and Stage 4 (lesion, Experiment 2C) checkpoints on Drive, and the frozen Stage 03/04/RACAF cache archive on Drive. Output: generation-based checkpoints under `experiments/FinalClassification/<timestamp>/checkpoints/` on Drive -- LAST (`gen_NNNNN/`, resumed from) and BEST (`best/`, global maximum `val_QWK`, the delivery model).

**As committed, running every cell trains nothing, creates no experiment and evaluates nothing** -- every switch in [S] is off.

## How to run a session

Use a **fresh runtime** for every session (Runtime -> Disconnect and delete runtime, then connect a T4 GPU runtime). Edit **only [S] Session configuration**, run the common cells **[S] through [8]** in order, then the cells for that kind of session. Every code cell starts with its label (`# ==== [P2] ...`); a cell the session does not need prints that it skipped, so running the whole notebook top to bottom is also safe.

| Session | Set in [S] | After [S] ... [8], run |
|---|---|---|
| Production, first session | `RUN_TRAINING = True`, `RESUME_EXPERIMENT_DIR = None` | [9] [P2] [T] [P3] |
| Production, every later session | `RUN_TRAINING = True`, `RESUME_EXPERIMENT_DIR = "<experiment root printed by [P3]>"` | [9] [P2] [T] [P3] |
| Post-run evaluation (training finished) | `RUN_POST_RUN_EVALUATION = True`, `POST_RUN_EXPERIMENT_DIR = "<experiment root>"` | [R1]-[R4], then [D2]-[D13], then [P3] |
| Cache maintenance | `MAINTENANCE_TASK = "<task>"` | the one [M] cell for that task |
| Optional overfit diagnostic | `RUN_OVERFIT_DIAGNOSTIC = True` | [D1] |

Common cells: [S] session configuration, [1] bootstrap, [2] setup, [3] safety toolkit, [P1] repository record + baseline-0 fingerprint, [4] frozen checkpoints, [5] split, [6] local cache, [7] joint model, [8] training configuration.

## Ending a training session

Interrupt (Runtime -> Interrupt execution) only once `Epoch N+1/50` has appeared, i.e. while an epoch is training. **Never interrupt while `Checkpoint written: ...` / `New global best ...` lines are being printed** -- LAST and BEST are being written then. Then run **[P3]**: it prints the exact `RESUME_EXPERIMENT_DIR` line for the next session. Optional insurance before disconnecting: `from google.colab import drive; drive.flush_and_unmount()`.

## What the notebook enforces

- **[T] is the only cell that trains** (`Trainer.fit`). It runs the launch gate [P2] itself first, so it cannot start if any check fails, if it already ran in this runtime, or if anything else (post-run evaluation, maintenance, a diagnostic) ran in this runtime.
- **Baseline-0** (`experiments/FinalClassification/2026-09-11_12-18-34`, LR 1e-3) is only ever listed and read: [P1] fingerprints it, [P2] and [P3] re-check it, and resuming it is refused.
- **One experiment per configuration**: with `RESUME_EXPERIMENT_DIR = None`, [P2] refuses to create a second experiment with this exact configuration and names the directory to resume instead (`ALLOW_ADDITIONAL_EXPERIMENT = True` deliberately starts a replicate).
- **Resume safety**: [P2] validates the experiment's checkpoints (config hash, LAST, BEST), refuses an experiment that has finished, and refuses if any Python module changed since the experiment was created.

Read `PROJECT_CODE.md`'s Development Workflow and `JOINT_TRAINING_ARCHITECTURE.md` in full before changing anything below.


In [ ]:
# ==== [S] SESSION CONFIGURATION -- the ONLY cell to edit before a session ====
# Everything else is fixed by the commit. Switch on at most ONE kind of session:
#
#   production, first session    RUN_TRAINING = True, RESUME_EXPERIMENT_DIR = None
#   production, later sessions   RUN_TRAINING = True, RESUME_EXPERIMENT_DIR = "<root printed by [P3]>"
#   post-run evaluation          RUN_POST_RUN_EVALUATION = True, POST_RUN_EXPERIMENT_DIR = "<root>"
#   cache maintenance            MAINTENANCE_TASK = one of MAINTENANCE_TASKS
#   overfit diagnostic           RUN_OVERFIT_DIAGNOSTIC = True
#
# Use a FRESH runtime for every session. With the values below nothing trains, no experiment
# directory is created and nothing is evaluated.
import posixpath

RUN_TRAINING = False                 # production training session (new or resumed experiment)
RESUME_EXPERIMENT_DIR = None         # None = first session of a NEW experiment; else the root to resume
ALLOW_ADDITIONAL_EXPERIMENT = False  # True ONLY to deliberately start a second experiment with this exact configuration

RUN_POST_RUN_EVALUATION = False      # read-only evaluation of a finished experiment
POST_RUN_EXPERIMENT_DIR = None       # the experiment root to evaluate

MAINTENANCE_TASK = None              # None, or one of MAINTENANCE_TASKS
RUN_OVERFIT_DIAGNOSTIC = False       # tiny-subset overfit test (optional diagnostic)

MAINTENANCE_TASKS = ("verify_raw_dataset", "precompute_cache", "flush_cache_to_drive",
                     "build_cache_archive")


def _drive_path(name, value):
    if not isinstance(value, str) or not value.strip().startswith("/content/drive/"):
        raise ValueError(f"{name} must be None or an absolute /content/drive/... path, not {value!r}")
    return posixpath.normpath(value.strip())


for _name, _value in (("RUN_TRAINING", RUN_TRAINING),
                      ("ALLOW_ADDITIONAL_EXPERIMENT", ALLOW_ADDITIONAL_EXPERIMENT),
                      ("RUN_POST_RUN_EVALUATION", RUN_POST_RUN_EVALUATION),
                      ("RUN_OVERFIT_DIAGNOSTIC", RUN_OVERFIT_DIAGNOSTIC)):
    if not isinstance(_value, bool):
        raise TypeError(f"{_name} must be True or False, not {_value!r}")
if MAINTENANCE_TASK is not None and MAINTENANCE_TASK not in MAINTENANCE_TASKS:
    raise ValueError(f"MAINTENANCE_TASK must be None or one of {MAINTENANCE_TASKS}, "
                     f"not {MAINTENANCE_TASK!r}")

_roles = [role for role, on in (("production training", RUN_TRAINING),
                                ("post-run evaluation", RUN_POST_RUN_EVALUATION),
                                ("maintenance", MAINTENANCE_TASK is not None),
                                ("overfit diagnostic", RUN_OVERFIT_DIAGNOSTIC)) if on]
if len(_roles) > 1:
    raise ValueError(f"More than one kind of session is switched on: {_roles}. Switch on exactly one.")
SESSION_ROLE = _roles[0] if _roles else "inspection only"

if RESUME_EXPERIMENT_DIR is not None:
    RESUME_EXPERIMENT_DIR = _drive_path("RESUME_EXPERIMENT_DIR", RESUME_EXPERIMENT_DIR)
if POST_RUN_EXPERIMENT_DIR is not None:
    POST_RUN_EXPERIMENT_DIR = _drive_path("POST_RUN_EXPERIMENT_DIR", POST_RUN_EXPERIMENT_DIR)
if RESUME_EXPERIMENT_DIR is not None and not RUN_TRAINING:
    raise ValueError("RESUME_EXPERIMENT_DIR is only used by a production training session "
                     "(RUN_TRAINING = True).")
if ALLOW_ADDITIONAL_EXPERIMENT and (not RUN_TRAINING or RESUME_EXPERIMENT_DIR is not None):
    raise ValueError("ALLOW_ADDITIONAL_EXPERIMENT only applies with RUN_TRAINING = True and "
                     "RESUME_EXPERIMENT_DIR = None.")
if RUN_POST_RUN_EVALUATION != (POST_RUN_EXPERIMENT_DIR is not None):
    raise ValueError("Set RUN_POST_RUN_EVALUATION = True and POST_RUN_EXPERIMENT_DIR together.")

NEEDS_LOCAL_CACHE = SESSION_ROLE in ("production training", "post-run evaluation", "overfit diagnostic")
NEEDS_JOINT_MODEL = SESSION_ROLE in ("production training", "post-run evaluation")

print(f"Session role: {SESSION_ROLE}")
if RUN_TRAINING:
    print("  experiment:", "NEW (first session)" if RESUME_EXPERIMENT_DIR is None
          else f"RESUME {RESUME_EXPERIMENT_DIR}")
if RUN_POST_RUN_EVALUATION:
    print("  evaluating:", POST_RUN_EXPERIMENT_DIR)
if MAINTENANCE_TASK is not None:
    print("  maintenance task:", MAINTENANCE_TASK)
print(f"  extracts/verifies the local cache: {NEEDS_LOCAL_CACHE} | builds joint_model: {NEEDS_JOINT_MODEL}")


### Bootstrap

Same minimal clone + `sys.path` setup every stage notebook needs -- see `colab/common/setup.py`'s module docstring for why this is intentionally duplicated.

In [ ]:
# ==== [1] BOOTSTRAP -- clone/pull the repository and set sys.path ====
import os
import subprocess
import sys

REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"

if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)

for path in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if path not in sys.path:
        sys.path.insert(0, path)

print("Bootstrap complete:", REPO_DIR)


### Imports

The reusable `colab/common/` infrastructure, the joint dataset loader (`joint_training_dataset.py`), and the joint model builder (`joint_training_model.py`) are all implemented -- see `JOINT_TRAINING_ARCHITECTURE.md`. `compile_joint_model()` compiles with `corn.corn_loss` as the sole training objective plus `corn.CORNQuadraticWeightedKappa` as a reported metric (never a second loss), so Keras produces `"QWK"`/`"val_QWK"` for this notebook's `monitor="val_QWK", mode="max"` checkpoint-selection policy below.


In [ ]:
# ==== [2] SETUP -- mount Drive, install requirements, verify the environment, imports ====
import setup

setup_info = setup.setup()

import colab_config
import verify_environment

env_report = verify_environment.verify_all(
    repo_dir=colab_config.REPO_DIR,
    drive_mount_point=colab_config.DRIVE_MOUNT_POINT,
    requirements_path=os.path.join(colab_config.REPO_DIR, "requirements.txt"),
    require_gpu=True,
)

import config
import corn
import joint_training_dataset as jtd
import joint_training_model as jtm
import racaf


### Session safety: toolkit [3] and baseline-0 fingerprint [P1]

[3] only defines the helpers the safety checks share ([P1], [P2], [P3], [R1]-[R4]); running it reads and writes nothing.

[P1] records the repository commit -- the runtime's clone must be clean and at `origin/main` -- and a read-only fingerprint of baseline-0 (`experiments/FinalClassification/2026-09-11_12-18-34`): every file's relative path, size and mtime, plus every directory, using directory listings and `os.stat` only. Nothing under baseline-0 is opened or written. The fingerprint is kept in memory and on local disk (`/content/session_safety/`) for [P2] and [P3]. The first [P1] ever also stores it as the cross-session reference in the Drive `logs/` folder (outside baseline-0), and every later [P1] compares against it, so a change made during an earlier session is caught as well.


In [ ]:
# ==== [3] SESSION SAFETY TOOLKIT -- definitions only; running this cell reads and writes nothing ====
import datetime
import hashlib
import json
import math
import os
import posixpath
import subprocess

import numpy as np
import tensorflow as tf

import training.checkpointing as ckpt
from training import config_hash, model_precision_policies
from training.trainer import DIAGNOSTIC_DIRTY_ATTRIBUTE

FINAL_CLASSIFICATION_EXPERIMENTS_DIR = colab_config.DRIVE.experiment_dir("FinalClassification")
# Baseline-0: the completed LR = 1e-3 run. Only ever listed and read -- never a training,
# resume or evaluation target, and nothing is ever written under it.
BASELINE_0_DIR = posixpath.join(FINAL_CLASSIFICATION_EXPERIMENTS_DIR, "2026-09-11_12-18-34")
# Written ONCE, by the first [P1] ever, outside baseline-0: lets every later session prove that
# baseline-0 did not change between sessions, not only within one.
BASELINE_0_REFERENCE_PATH = posixpath.join(colab_config.LOGS_ROOT, "baseline0_reference_fingerprint.json")
SESSION_SAFETY_DIR = "/content/session_safety"   # LOCAL disk: survives a kernel restart, not a new runtime
EXPECTED_OPTIMIZER_TYPE = "Adam"
EXPECTED_PRECISION_POLICY = "mixed_float16"
# Switches from earlier versions of this notebook; none may be set in a production runtime.
LEGACY_SWITCHES = ("VERIFY_RAW_DATASET", "RUN_CACHE_PRECOMPUTATION", "RUN_CACHE_DIAGNOSTIC",
                   "RUN_FLUSH_TO_DRIVE", "RUN_CACHE_ARCHIVE_BUILD", "RUN_CACHE_ARCHIVE_EXTRACT",
                   "RUN_STEP_PROFILER", "RUN_STEP_BREAKDOWN", "RUN_FIT_DIAGNOSTIC",
                   "RUN_CHECKPOINT_COST", "CHECKPOINT_COST_ON_DRIVE")

if "_RUNTIME_NON_TRAINING_WORK" not in globals():
    _RUNTIME_NON_TRAINING_WORK = set()   # non-training work done in this runtime; [P2] requires none


def git_output(*args):
    return subprocess.run(["git", "-C", colab_config.REPO_DIR, *args],
                          capture_output=True, text=True, check=True).stdout.strip()


def read_json(path):
    with open(path) as handle:
        return json.load(handle)


def is_baseline_0(path):
    """True for baseline-0 itself or anything beneath it, both as written and fully resolved."""
    written = posixpath.normpath(str(path))
    if written == BASELINE_0_DIR or written.startswith(BASELINE_0_DIR + "/"):
        return True
    resolved, root = os.path.realpath(str(path)), os.path.realpath(BASELINE_0_DIR)
    return resolved == root or resolved.startswith(root + os.sep)


def tree_fingerprint(root):
    """[relative path, size, mtime] for every file under `root` and [dir + "/", None, None] for
    every directory. Directory listings and os.stat only: nothing is opened or written."""
    rows = []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames.sort()
        relative = os.path.relpath(dirpath, root).replace(os.sep, "/")
        prefix = "" if relative == "." else relative + "/"
        rows.append([prefix or "./", None, None])
        for name in sorted(filenames):
            info = os.stat(os.path.join(dirpath, name))
            rows.append([prefix + name, int(info.st_size), int(info.st_mtime)])
    return rows


def describe_fingerprint_change(before, after, limit=8):
    old, new = {row[0]: row for row in before}, {row[0]: row for row in after}
    added = sorted(set(new) - set(old))
    removed = sorted(set(old) - set(new))
    changed = sorted(k for k in set(old) & set(new) if list(old[k]) != list(new[k]))
    parts = [f"{label} {len(names)}: {names[:limit]}" for label, names in
             (("added", added), ("removed", removed), ("changed", changed)) if names]
    return "; ".join(parts) or "no difference"


def model_weights_sha256(model):
    """SHA-256 over every model weight (trainable + BatchNorm statistics), in build order."""
    digest = hashlib.sha256()
    for array in model.get_weights():
        digest.update(np.ascontiguousarray(array).tobytes())
    return digest.hexdigest()


def verify_baseline_0_unchanged():
    """Failures (an empty list means unchanged): baseline-0 against the fingerprint [P1] recorded in
    this runtime and against the cross-session reference on Drive. Reads only."""
    if not os.path.isdir(BASELINE_0_DIR):
        return [f"baseline-0 is missing: {BASELINE_0_DIR}"]
    current = tree_fingerprint(BASELINE_0_DIR)
    failures = []
    recorded = globals().get("BASELINE_0_FINGERPRINT")
    local_copy = posixpath.join(SESSION_SAFETY_DIR, "baseline0_fingerprint.json")
    if recorded is None and os.path.exists(local_copy):
        recorded = read_json(local_copy)["rows"]        # after a kernel restart
    if recorded is None:
        failures.append("[P1] has not run in this runtime, so there is nothing to compare against")
    elif current != recorded:
        failures.append("baseline-0 CHANGED during this runtime: "
                        + describe_fingerprint_change(recorded, current))
    if not os.path.exists(BASELINE_0_REFERENCE_PATH):
        failures.append(f"the cross-session reference {BASELINE_0_REFERENCE_PATH} is missing")
    else:
        reference = read_json(BASELINE_0_REFERENCE_PATH)["rows"]
        if current != reference:
            failures.append("baseline-0 differs from the cross-session reference: "
                            + describe_fingerprint_change(reference, current))
    return failures


def experiment_config_hashes(experiment_root):
    """Every config hash recorded for one experiment: metadata.json (experiments created by this
    notebook record it) and every checkpoint manifest. An unreadable metadata.json raises, so a
    scan fails closed; an unreadable manifest -- a generation cut off mid-write -- is skipped."""
    hashes = set()
    metadata_path = posixpath.join(experiment_root, "metadata.json")
    if os.path.exists(metadata_path):
        value = read_json(metadata_path).get("config_hash")
        if value:
            hashes.add(value)
    checkpoint_dir = posixpath.join(experiment_root, "checkpoints")
    directories = [path for _, path in ckpt.list_generations(checkpoint_dir)]
    for directory in directories + [ckpt.best_dir(checkpoint_dir)]:
        try:
            value = read_json(os.path.join(directory, ckpt.MANIFEST_FILENAME)).get("config_hash")
        except (OSError, ValueError):
            continue
        if value:
            hashes.add(value)
    return hashes


def find_experiments_with_config(expected_hash):
    """(roots of experiments recording `expected_hash`, entries that could not be read)."""
    matches, problems = [], []
    if not os.path.isdir(FINAL_CLASSIFICATION_EXPERIMENTS_DIR):
        return matches, problems
    for name in sorted(os.listdir(FINAL_CLASSIFICATION_EXPERIMENTS_DIR)):
        root = posixpath.join(FINAL_CLASSIFICATION_EXPERIMENTS_DIR, name)
        if name.startswith((".", "_")) or not os.path.isdir(root):
            continue
        try:
            if expected_hash in experiment_config_hashes(root):
                matches.append(root)
        except (OSError, ValueError) as error:
            problems.append(f"{name}: {error!r}")
    return matches, problems


def python_changes_since(commit):
    """Tracked Python modules (tests excluded) and requirements.txt that differ between `commit`
    and HEAD -- code that could change training behaviour in the middle of an experiment."""
    output = git_output("diff", "--name-only", commit, "HEAD", "--",
                        "*.py", "requirements.txt", ":(exclude)tests")
    return [line for line in output.splitlines() if line.strip()]


def inspect_checkpoint_state(checkpoint_dir, expected):
    """Read-only inspection of one experiment's checkpoints/: the generation a resume would use,
    whether BEST agrees with it, and whether the run has finished. "failures" are real hazards (a
    resume would be refused, or BEST is missing/stale); "warnings" are the normal after-effects of
    an interrupted session, which the resume logic handles. Mode is "max" (val_QWK)."""
    report = {"checkpoint_dir": checkpoint_dir, "generations": [], "latest_pointer": None,
              "resumable": None, "state": None, "best": None, "finished": None,
              "failures": [], "warnings": [], "notes": []}
    fail, warn, note = report["failures"].append, report["warnings"].append, report["notes"].append
    if not os.path.isdir(checkpoint_dir):
        fail(f"{checkpoint_dir} does not exist -- not an experiment created by this notebook")
        return report

    report["latest_pointer"] = ckpt.read_latest_pointer(checkpoint_dir)
    resumable = ckpt.find_resumable_generation(checkpoint_dir, verbose=False)
    for number, path in ckpt.list_generations(checkpoint_dir):
        if path == resumable:
            valid, reason = True, None       # find_resumable_generation() has just validated it
        else:
            result = ckpt.validate_generation(path)
            valid, reason = result.ok, result.reason
        report["generations"].append({"name": os.path.basename(path), "number": number,
                                      "valid": valid, "reason": reason})

    if resumable is None:
        evidence = ckpt.checkpoint_evidence(checkpoint_dir)
        if evidence:
            fail(f"checkpoint state exists ({', '.join(evidence)}) but no generation validates, so "
                 "the next resume would be REFUSED (CheckpointResumeError). Inspect the generations "
                 "listed above before doing anything else.")
        else:
            note("no checkpoint yet (0 epochs completed): a resume starts this experiment at epoch 1")
        return report

    state = ckpt.read_state(resumable)
    manifest = read_json(os.path.join(resumable, ckpt.MANIFEST_FILENAME))
    name, number = os.path.basename(resumable), ckpt.generation_number(resumable)
    report["resumable"], report["state"] = name, state
    for field_name in ("config_hash", "optimizer_type", "precision_policy"):
        if manifest.get(field_name) != expected[field_name]:
            fail(f"{name}: {field_name} is {manifest.get(field_name)!r} but this notebook expects "
                 f"{expected[field_name]!r} -- the resume would be refused")
    if (state.monitor, state.monitor_mode) != (expected["monitor"], expected["mode"]):
        fail(f"{name}: monitors {state.monitor}/{state.monitor_mode}, expected "
             f"{expected['monitor']}/{expected['mode']}")
    early, reduce_lr = state.early_stopping or {}, state.reduce_lr or {}
    for label, got, want in (("EarlyStopping patience", early.get("patience"), expected["early_stopping_patience"]),
                             ("ReduceLROnPlateau patience", reduce_lr.get("patience"), expected["reduce_lr_patience"]),
                             ("ReduceLROnPlateau factor", reduce_lr.get("factor"), expected["reduce_lr_factor"]),
                             ("ReduceLROnPlateau min_lr", reduce_lr.get("min_lr"), expected["min_lr"])):
        if got is None or not math.isclose(float(got), float(want), rel_tol=1e-9):
            fail(f"{name}: {label} recorded as {got!r}, expected {want!r}")

    if report["latest_pointer"] != name:
        warn(f"latest.json points to {report['latest_pointer']!r}, which does not validate; a resume "
             f"falls back to {name}")
    for generation in report["generations"]:
        if generation["number"] > number and generation["valid"]:
            warn(f"{generation['name']} is valid but newer than latest.json's target; a resume uses "
                 f"{name} (Trainer's rule), so that epoch is trained again")
        elif generation["number"] > number:
            warn(f"{generation['name']} is incomplete ({generation['reason']}): the session ended while "
                 "writing it; it is ignored and that epoch is trained again")
        elif not generation["valid"]:
            warn(f"older {generation['name']} does not validate ({generation['reason']}); a resume "
                 "does not need it")

    best_path = ckpt.best_dir(checkpoint_dir)
    best = None
    if os.path.isdir(best_path):
        result = ckpt.validate_generation(best_path, required=ckpt.BEST_REQUIRED_FILES)
        best = {"valid": result.ok, "reason": result.reason}
        if result.ok:
            best_state = ckpt.read_state(best_path)
            best.update(best_epoch=best_state.best_epoch, best_metric=best_state.best_metric,
                        completed_epoch=best_state.completed_epoch,
                        source_generation=result.manifest.get("source_generation"),
                        config_hash=result.manifest.get("config_hash"))
    report["best"] = best
    claimed_epoch, claimed_metric = state.best_epoch, state.best_metric
    if claimed_epoch is not None and state.completed_epoch == claimed_epoch + 1:
        remedy = (f" The weights are still in {name}; republish BEST from it exactly as the epoch's "
                  f"own on_epoch_end would have: ckpt.save_best({checkpoint_dir!r}, {resumable!r}, "
                  f"ckpt.read_state({resumable!r}), staging_dir='/content/checkpoint_staging')")
    else:
        remedy = " The generation that earned it has been pruned; its weights are no longer on disk."
    if claimed_epoch is not None and (best is None or not best["valid"]):
        fail(f"BEST is {'missing' if best is None else 'invalid (' + str(best['reason']) + ')'} although "
             f"{name} records the global best {state.monitor}={claimed_metric} at epoch "
             f"{claimed_epoch + 1}." + remedy)
    elif claimed_epoch is not None:
        if best["config_hash"] != expected["config_hash"]:
            fail(f"best/ records config hash {best['config_hash']!r}, expected {expected['config_hash']!r}")
        if best["best_epoch"] is None or best["completed_epoch"] != best["best_epoch"] + 1:
            fail("best/state.json is internally inconsistent (BEST is saved at the epoch that earned it)")
        elif best["best_epoch"] == claimed_epoch and best["best_metric"] == claimed_metric:
            pass
        elif (best["best_metric"] is not None and claimed_metric is not None
              and best["best_metric"] > claimed_metric):
            warn(f"best/ (epoch {best['best_epoch'] + 1}) is ahead of {name}'s record (epoch "
                 f"{claimed_epoch + 1}): expected after a fallback to an older generation; the resume "
                 "adopts best/")
        else:
            fail(f"STALE BEST: {name} records the global best {state.monitor}={claimed_metric} at epoch "
                 f"{claimed_epoch + 1}, but best/ holds epoch {best['best_epoch'] + 1} "
                 f"({best['best_metric']}). The session ended between writing that epoch's generation "
                 "and publishing BEST." + remedy)
    elif best is not None:
        warn(f"best/ exists although {name} records no best epoch")

    stopped = int(early.get("stopped_epoch") or 0)
    if stopped > 0:
        report["finished"] = f"EarlyStopping stopped the run after epoch {stopped + 1}"
    elif state.completed_epoch >= expected["epochs"]:
        report["finished"] = f"the {expected['epochs']}-epoch cap was reached"
    return report


def print_checkpoint_report(report):
    print("checkpoints:", report["checkpoint_dir"])
    for generation in report["generations"]:
        print(f"  {generation['name']}: " + ("valid" if generation["valid"]
                                              else "NOT VALID -- " + str(generation["reason"])))
    print("  latest.json ->", report["latest_pointer"])
    state = report["state"]
    if state is not None:
        early, reduce_lr = state.early_stopping or {}, state.reduce_lr or {}
        best_epoch = None if state.best_epoch is None else state.best_epoch + 1
        print(f"  resume point: {report['resumable']} | epochs completed {state.completed_epoch} | "
              f"learning rate {state.learning_rate} | optimizer iterations "
              f"{state.optimizer.get('iterations')} | global best {state.monitor}={state.best_metric} "
              f"at epoch {best_epoch}")
        print(f"  EarlyStopping wait {early.get('wait')}/{early.get('patience')} | "
              f"ReduceLROnPlateau wait {reduce_lr.get('wait')}/{reduce_lr.get('patience')}")
    best = report["best"]
    if best is not None and best["valid"]:
        print(f"  best/: valid | epoch {best['best_epoch'] + 1} | {best['best_metric']} | "
              f"from {best['source_generation']}")
    elif best is not None:
        print("  best/: NOT VALID --", best["reason"])
    for line in report["notes"]:
        print("  note:", line)
    for line in report["warnings"]:
        print("  WARNING:", line)
    for line in report["failures"]:
        print("  FAILURE:", line)
    if report["finished"]:
        print("  FINISHED:", report["finished"])


In [ ]:
# ==== [P1] REPOSITORY RECORD + BASELINE-0 READ-ONLY FINGERPRINT ====
# Reads baseline-0 through directory listings and os.stat only. Writes to local disk and -- only
# the first time [P1] ever runs -- the cross-session reference in the Drive logs/ folder.
SESSION_COMMIT = git_output("rev-parse", "HEAD")
_origin = git_output("rev-parse", "origin/" + colab_config.REPO_BRANCH)
_modified = git_output("status", "--porcelain", "--untracked-files=no")
print("repository:", colab_config.REPO_DIR)
print("  commit:", SESSION_COMMIT)
if SESSION_COMMIT != _origin:
    raise RuntimeError(f"The runtime's clone is at {SESSION_COMMIT}, not origin/"
                       f"{colab_config.REPO_BRANCH} ({_origin}). Re-run [1].")
if _modified:
    raise RuntimeError("Tracked files were modified inside this runtime's clone:\n" + _modified
                       + "\nStart a fresh runtime.")
print(f"  clean, identical to origin/{colab_config.REPO_BRANCH}")

if not os.path.isdir(BASELINE_0_DIR):
    raise RuntimeError(f"baseline-0 not found at {BASELINE_0_DIR} -- is Drive mounted?")
BASELINE_0_FINGERPRINT = tree_fingerprint(BASELINE_0_DIR)
_files = [row for row in BASELINE_0_FINGERPRINT if row[1] is not None]
print("baseline-0:", BASELINE_0_DIR)
print(f"  fingerprinted read-only: {len(_files)} files, {sum(row[1] for row in _files):,} bytes, "
      f"{len(BASELINE_0_FINGERPRINT) - len(_files)} directories")

os.makedirs(SESSION_SAFETY_DIR, exist_ok=True)
with open(posixpath.join(SESSION_SAFETY_DIR, "baseline0_fingerprint.json"), "w") as handle:
    json.dump({"commit": SESSION_COMMIT, "rows": BASELINE_0_FINGERPRINT}, handle)

if os.path.exists(BASELINE_0_REFERENCE_PATH):
    _reference = read_json(BASELINE_0_REFERENCE_PATH)
    if _reference["rows"] != BASELINE_0_FINGERPRINT:
        raise RuntimeError(f"BASELINE-0 CHANGED since the reference recorded {_reference.get('recorded')}: "
                           + describe_fingerprint_change(_reference["rows"], BASELINE_0_FINGERPRINT))
    print(f"  identical to the cross-session reference recorded {_reference.get('recorded')}")
else:
    os.makedirs(posixpath.dirname(BASELINE_0_REFERENCE_PATH), exist_ok=True)
    with open(BASELINE_0_REFERENCE_PATH, "w") as handle:
        json.dump({"recorded": datetime.datetime.now().isoformat(timespec="seconds"),
                   "commit": SESSION_COMMIT, "baseline_0_dir": BASELINE_0_DIR,
                   "rows": BASELINE_0_FINGERPRINT}, handle)
    print("  cross-session reference recorded (first [P1] ever):", BASELINE_0_REFERENCE_PATH)


### Frozen Stage 1/3/4 checkpoint discovery

Stage 1 is resolved for completeness only -- it is **not** loaded into the downstream APTOS graph (`JOINT_TRAINING_ARCHITECTURE.md` §3.1, locked). Stage 3/Stage 4 checkpoints must already exist on Drive; this notebook never trains, retrains, moves, renames, or overwrites them.

In [ ]:
# ==== [4] FROZEN STAGE 1/3/4 CHECKPOINTS -- resolved and checked, never trained or written ====
print("Stage 1 (IQA) checkpoint  :", config.IQA_MODEL_DIR, "-- resolved only, NOT used in this graph")
print("Stage 3 (vessel) checkpoint:", config.VESSEL_SEG_MODEL_DIR)
print("Stage 4 (lesion) checkpoint:", config.LESION_SEG_MODEL_DIR)

vessel_checkpoint_path = os.path.join(config.VESSEL_SEG_MODEL_DIR, "best_model.pth")
lesion_checkpoint_path = os.path.join(config.LESION_SEG_MODEL_DIR, "best_model.keras")

for label, path in (("Stage 3 vessel", vessel_checkpoint_path), ("Stage 4 lesion (Experiment 2C)", lesion_checkpoint_path)):
    if not os.path.exists(path):
        raise FileNotFoundError(f"{label} checkpoint not found at {path} -- see JOINT_TRAINING_ARCHITECTURE.md \u00a78.")
    print(f"{label} checkpoint found: {path}")


### Authoritative split

The SAME split Stage 5/6 already use -- `downstream_split.get_authoritative_split()`, read from the committed manifest. No second split is computed here. [5] verifies it exactly: 2929 train / 733 val, the per-grade counts, no image in both, and a SHA-256 of the ordered entries.


In [ ]:
# ==== [5] AUTHORITATIVE SPLIT -- the committed manifest, verified exactly ====
import collections

train_entries, val_entries = jtd.split_train_val_ids()
EXPECTED_SPLIT_SHA256 = "f512a7a086ac7f53e1a5ef8b49a703ae0c5e70db37891253640c91306d965837"
EXPECTED_TRAIN_GRADES = {0: 1444, 1: 296, 2: 799, 3: 154, 4: 236}
EXPECTED_VAL_GRADES = {0: 361, 1: 74, 2: 200, 3: 39, 4: 59}

SPLIT_VERIFIED = False
_split_sha256 = hashlib.sha256(json.dumps(
    [[list(entry) for entry in train_entries], [list(entry) for entry in val_entries]],
    separators=(",", ":")).encode("utf-8")).hexdigest()
_train_grades = dict(sorted(collections.Counter(grade for _, grade in train_entries).items()))
_val_grades = dict(sorted(collections.Counter(grade for _, grade in val_entries).items()))
print(f"train: {len(train_entries)} (expected 2929) per grade {_train_grades}")
print(f"val:   {len(val_entries)} (expected 733) per grade {_val_grades}")
print("split sha256:", _split_sha256)
assert len(train_entries) == 2929 and len(val_entries) == 733, "authoritative split counts do not match the committed manifest"
assert _train_grades == EXPECTED_TRAIN_GRADES and _val_grades == EXPECTED_VAL_GRADES, "per-grade counts differ from the committed manifest"
assert not ({i for i, _ in train_entries} & {i for i, _ in val_entries}), "an image is in both train and val"
assert _split_sha256 == EXPECTED_SPLIT_SHA256, "the split's contents differ from the committed manifest"
SPLIT_VERIFIED = True


### Local cache: extract the archive and verify it (automatic when the session reads data)

Training, post-run evaluation and the overfit diagnostic read Stage 03/04 predictions, RACAF reliability and canonical RGB from LOCAL disk. For those sessions [6] streams the 8 archive shards from Drive into `/content/cache/` (measured on the real archive: 14,604 files / 28.53 GiB in ~8.8 min), after `plan_extraction()` has confirmed the disk budget without reading a byte, then verifies the result: 3,651 entries fully local, 0 needing Drive, 0 corrupt or zero-byte files, and exactly the 11 known empty-FOV entries missing everywhere (by design: Phase 1 cannot produce their vessel/lesion/reliability, and the training generator skips them). Any other result stops the session. Re-running [6] on the same machine does not read the archive again. The persistent Drive cache and the archive are only read here; rebuilding them is maintenance ([M2]-[M4]).


In [ ]:
# ==== [6] LOCAL CACHE -- locations; archive extraction + verification when the session reads data ====
LOCAL_CACHE_DIR = "/content/cache/local_feature_extraction"
LOCAL_RACAF_CACHE_DIR = "/content/cache/racaf"
# Deliberately empty/nonexistent -- makes _resolve_processed_rgb() always take its cheap,
# unmodified live Stage 02 fallback instead of one more per-image Drive lookup.
NO_PRECOMPUTED_STAGE02_DIR = "/content/cache/_no_precomputed_stage02_output"
LOCAL_RAW_IMAGE_DIR = "/content/cache/raw_images_for_uncached_entries"
ARCHIVE_DIR = os.path.join(os.path.dirname(config.LOCAL_FEATURE_RESULTS_DIR), "cache_archive")
LOCAL_CACHE_MARKER = "/content/cache/.archive_extracted_and_verified.json"   # local disk only
EXPECTED_FULLY_LOCAL_ENTRIES = 3651   # 3662 split entries minus the known empty-FOV images
EXPECTED_EMPTY_FOV_ENTRIES = 11       # no vessel/lesion/reliability anywhere, by design

print("Stage 3/4 canonical cache (Drive):", config.LOCAL_FEATURE_RESULTS_DIR)
print("RACAF reliability cache (Drive)  :", config.RACAF_RESULTS_DIR)
print("cache archive (Drive)            :", ARCHIVE_DIR)

LOCAL_CACHE_VERIFIED = False
if NEEDS_LOCAL_CACHE:
    import joint_cache_archive as jca
    import joint_cache_staging as jcs

    _entries = train_entries + val_entries
    # Local os.path.exists only -- never a per-file Drive stat.
    _incomplete = jcs.entries_missing_local_cache(_entries, LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR)
    if os.path.exists(LOCAL_CACHE_MARKER) and len(_incomplete) == EXPECTED_EMPTY_FOV_ENTRIES:
        print("Local cache already extracted and verified on this machine -- the archive is not read again.")
    else:
        # Measure the disk budget BEFORE reading a byte. Extraction is refused rather than allowed to
        # fill /content and fail somewhere unrelated later.
        _plan = jca.plan_extraction(archive_dir=ARCHIVE_DIR, cache_dir=LOCAL_CACHE_DIR,
                                    racaf_cache_dir=LOCAL_RACAF_CACHE_DIR)
        jca.print_extraction_plan(_plan)
        if not _plan["fits"] or _plan["drive_unreachable"]:
            raise RuntimeError(
                "Refusing to extract the cache archive: %s. Nothing was read, written or "
                "recomputed. Free local space first -- and do NOT delete the persistent Drive "
                "cache, the archive, or the local cache that is already correct."
                % ("Drive is unreachable" if _plan["drive_unreachable"]
                   else "short by %.2f GiB" % (_plan["shortfall_bytes"] / 1024 ** 3)))
        print("")
        _extract = jca.extract_archive(
            archive_dir=ARCHIVE_DIR, cache_dir=LOCAL_CACHE_DIR, racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
            # Second, independent guard: extract_archive re-checks free space itself.
            min_free_bytes=_plan["required_bytes"],
        )
        jca.print_extract(_extract)
        if _extract["corrupt"] or _extract["drive_unreachable"]:
            raise RuntimeError(f"Archive extraction did not complete (corrupt shards: {_extract['corrupt']}, "
                               f"Drive unreachable: {_extract['drive_unreachable']}). Re-run this cell; "
                               "files already extracted are kept.")

    print("")
    LOCAL_CACHE_REPORT = jcs.verify_local_cache(
        _entries, cache_dir=LOCAL_CACHE_DIR, racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
        persistent_cache_dir=config.LOCAL_FEATURE_RESULTS_DIR,
        persistent_racaf_cache_dir=config.RACAF_RESULTS_DIR,
        content_sample=5,
    )
    jcs.print_verification(LOCAL_CACHE_REPORT)
    _problems = [label for label, ok in (
        (f"fully local == {EXPECTED_FULLY_LOCAL_ENTRIES}",
         LOCAL_CACHE_REPORT["fully_local"] == EXPECTED_FULLY_LOCAL_ENTRIES),
        ("Drive fallback required == 0", LOCAL_CACHE_REPORT["drive_fallback_required"] == 0),
        (f"missing everywhere == {EXPECTED_EMPTY_FOV_ENTRIES} (the known empty-FOV images)",
         LOCAL_CACHE_REPORT["missing_everywhere"] == EXPECTED_EMPTY_FOV_ENTRIES),
        ("no corrupt local file", not LOCAL_CACHE_REPORT["corrupt_local"]),
        ("no zero-byte local file", not LOCAL_CACHE_REPORT["empty_local_files"]),
    ) if not ok]
    if _problems:
        if os.path.exists(LOCAL_CACHE_MARKER):
            os.remove(LOCAL_CACHE_MARKER)   # the next run of this cell extracts again
        raise RuntimeError("Local cache verification FAILED: " + "; ".join(_problems) + ". Nothing was "
                           "trained. Do NOT delete the persistent Drive cache or the archive.")
    with open(LOCAL_CACHE_MARKER, "w") as handle:
        json.dump({"verified": datetime.datetime.now().isoformat(timespec="seconds"),
                   "fully_local": LOCAL_CACHE_REPORT["fully_local"]}, handle)
    LOCAL_CACHE_VERIFIED = True
    print("Local cache verified: every split entry is local except the known empty-FOV images.")
else:
    print(f"Session role '{SESSION_ROLE}' does not read the local cache -- nothing was extracted or verified.")


### Joint model construction (precision policy FIRST)

Composes each stage's own, unmodified `build_*()` function -- `joint_training_model.build_and_compile_joint_model()` -- into one functional `keras.Model` and compiles it with `corn.corn_loss` + `CORNQuadraticWeightedKappa`.

**Why this cell sets the precision policy before building.** Keras 3 captures a layer's dtype policy in the layer's CONSTRUCTOR and decides whether to wrap the optimizer in a `LossScaleOptimizer` when the model is COMPILED. Previously this notebook built and compiled the model here, and the dtype policy was only set later, inside `Trainer.prepare()` -- by which point it could no longer affect anything. Verified on TF 2.21.0 / Keras 3.15.1: a model built under `float32` and then exposed to a `mixed_float16` global policy still reports `compute=float32` on every weighted layer and still carries a bare `Adam`, so a run configured with `MIXED_PRECISION = True` trained entirely in float32, with no loss scaling and no float16 tensor-core throughput, while every flag reported mixed precision as enabled.

`build_and_compile_joint_model()` does policy -> build -> compile in that order and raises if the result is not what was asked for. `Trainer` re-checks it (`precision_check="error"` in the training cell below), so the two can never silently disagree again.

Note: under `mixed_float16`, trainable VARIABLES stay float32 by design (`compute_dtype=float16, variable_dtype=float32`) and so do the gradients applied to them -- that is correct mixed precision, not a symptom. What matters is that the layers' compute dtype is float16 and the optimizer is a `LossScaleOptimizer`; both are asserted.

[7] then verifies the result -- `mixed_float16`, `LossScaleOptimizer(inner=Adam)`, learning rate 1e-4, 409 trainable tensors, 43,338,506 trainable parameters, optimizer iterations 0 -- and records a SHA-256 over every weight, so [P2] can prove the model handed to training is byte-for-byte the one built here. (`count_params()` prints 43,342,346 because it also counts the 3,844 BatchNorm statistics.) The model is built only in sessions that use it (training and post-run evaluation).


In [ ]:
# ==== [7] JOINT MODEL -- precision policy FIRST, then build, compile and verify ====
# Precision policy is established HERE, before a single layer is constructed -- see the
# markdown above for why the previous ordering made MIXED_PRECISION inert.
import tensorflow as tf

MIXED_PRECISION = True
# Initial learning rate. Baseline-0 (experiments/FinalClassification/2026-09-11_12-18-34) ran with
# Adam()'s default of 1e-3. The optimizer TYPE is unchanged (Adam), and compile() still wraps it in
# a LossScaleOptimizer under mixed_float16. On resume, the optimizer state restored from the
# checkpoint (learning rate included, with any ReduceLROnPlateau reduction) replaces this value.
LEARNING_RATE = 1e-4
EXPECTED_TRAINABLE_TENSORS = 409
EXPECTED_TRAINABLE_PARAMETERS = 43_338_506

if NEEDS_JOINT_MODEL:
    joint_model = jtm.build_and_compile_joint_model(
        mixed_precision=MIXED_PRECISION,
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    )

    _optimizer = joint_model.optimizer
    _inner = getattr(_optimizer, "inner_optimizer", None)
    _lr = float(tf.keras.backend.get_value(_optimizer.learning_rate))
    _parameters = sum(int(np.prod(v.shape)) for v in joint_model.trainable_variables)
    _model_checks = {
        "global policy mixed_float16": tf.keras.mixed_precision.global_policy().name == EXPECTED_PRECISION_POLICY,
        "layers built under mixed_float16": EXPECTED_PRECISION_POLICY in model_precision_policies(joint_model),
        "LossScaleOptimizer(inner=Adam)": (type(_optimizer).__name__ == "LossScaleOptimizer"
                                           and type(_inner).__name__ == EXPECTED_OPTIMIZER_TYPE),
        f"learning rate {LEARNING_RATE:g}": abs(_lr - LEARNING_RATE) < 1e-9,
        f"{EXPECTED_TRAINABLE_TENSORS} trainable tensors": len(joint_model.trainable_variables) == EXPECTED_TRAINABLE_TENSORS,
        f"{EXPECTED_TRAINABLE_PARAMETERS:,} trainable parameters": _parameters == EXPECTED_TRAINABLE_PARAMETERS,
        "optimizer never stepped (iterations 0)": int(tf.keras.backend.get_value(_optimizer.iterations)) == 0,
    }
    for _label, _ok in _model_checks.items():
        print(("PASS  " if _ok else "FAIL  ") + _label)
    if not all(_model_checks.values()):
        raise RuntimeError("Model verification failed -- do not train this model.")
    JOINT_MODEL_WEIGHTS_SHA256 = model_weights_sha256(joint_model)

    print("Optimizer learning rate:", _lr)
    print(f"Trainable parameters: {_parameters:,} | all weights incl. BatchNorm statistics: "
          f"{joint_model.count_params():,}")
    print("Weights SHA-256 at construction:", JOINT_MODEL_WEIGHTS_SHA256[:16], "(re-checked by [P2])")
    print("Loss:", joint_model.loss.__name__)
    joint_model.summary(line_length=120)
else:
    print(f"Session role '{SESSION_ROLE}' does not use joint_model -- it was not built.")


### Training configuration

Initial T4 configuration (`JOINT_TRAINING_ARCHITECTURE.md` §24): `batch_size=2`, `mixed_precision=True` (set in [7], before the model is built). `EPOCHS=50` is a cap; EarlyStopping decides the actual end. Best-checkpoint selection: `monitor="val_QWK"`, `mode="max"`. Learning rate 1e-4 (set in [7]); EarlyStopping patience 12; ReduceLROnPlateau patience 4, factor 0.5, `min_lr` 1e-6.

All of these go into `config_hash`, which every checkpoint records and every resume must match. [8] stops the session unless the hash equals the committed production hash `3f549e1638d9409f7862f1e799bd7052` (baseline-0's was `0099aabcab487a5e56792f34b2efdb28`, so neither can resume the other).


In [ ]:
# ==== [8] TRAINING CONFIGURATION + CONFIG HASH ====
BATCH_SIZE = 2
EPOCHS = 50
MONITOR_METRIC = "val_QWK"
MONITOR_MODE = "max"
# Callback schedule. EarlyStopping patience is 12; baseline-0 used TrainingConfig's default of 8.
# With ReduceLROnPlateau patience 4, a patience of 8 stops on the same epoch as the second LR
# reduction, so only one reduced rate is ever trained; 12 (= 3 x 4) gives each of two reductions
# a full 4-epoch window. EarlyStopping does not alter training before it stops, so the patience-8
# outcome is an exact prefix of this run. The ReduceLROnPlateau values are TrainingConfig's
# unchanged defaults, made explicit so config_hash can pin all of them.
EARLY_STOPPING_PATIENCE = 12
REDUCE_LR_PATIENCE = 4
REDUCE_LR_FACTOR = 0.5
MIN_LR = 1e-6
# MIXED_PRECISION is deliberately NOT set here: it must be established before the model is
# constructed, so it lives in the model-construction cell above. Setting it here would be too
# late to affect anything (Keras 3 captures the dtype policy per layer at construction time).

# Everything a later session must match to be allowed to continue this run. A resume whose
# hash differs is REFUSED rather than silently continued -- changing the batch size, the
# monitored metric, the precision policy, the initial learning rate or the EarlyStopping /
# ReduceLROnPlateau schedule mid-run would make the optimizer state and the global best
# incomparable. The callback schedule is not stored in the checkpoint (every session uses
# the values above), so this hash is what keeps sessions consistent. EXPECTED_CONFIG_HASH pins
# it to the committed production configuration.
EXPECTED_CONFIG_HASH = "3f549e1638d9409f7862f1e799bd7052"


def production_config_mapping():
    return {
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "monitor": MONITOR_METRIC,
        "mode": MONITOR_MODE,
        "mixed_precision": MIXED_PRECISION,
        "model": "joint_stage05_08_racaf",
        "loss": "corn_loss",
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "reduce_lr_patience": REDUCE_LR_PATIENCE,
        "reduce_lr_factor": REDUCE_LR_FACTOR,
        "min_lr": MIN_LR,
    }


def production_expectations():
    """What a checkpoint of THIS configuration must record ([P2], [P3] and [R1] compare against it)."""
    return {
        "config_hash": EXPECTED_CONFIG_HASH, "optimizer_type": EXPECTED_OPTIMIZER_TYPE,
        "precision_policy": EXPECTED_PRECISION_POLICY, "monitor": MONITOR_METRIC, "mode": MONITOR_MODE,
        "epochs": EPOCHS, "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "reduce_lr_patience": REDUCE_LR_PATIENCE, "reduce_lr_factor": REDUCE_LR_FACTOR, "min_lr": MIN_LR,
    }


TRAINING_CONFIG_HASH = config_hash(production_config_mapping())
print(f"batch_size={BATCH_SIZE}  epochs={EPOCHS}  mixed_precision={MIXED_PRECISION}  "
      f"learning_rate={LEARNING_RATE:g}  monitor={MONITOR_METRIC}/{MONITOR_MODE}")
print(f"EarlyStopping patience={EARLY_STOPPING_PATIENCE}  ReduceLROnPlateau patience={REDUCE_LR_PATIENCE} "
      f"factor={REDUCE_LR_FACTOR} min_lr={MIN_LR:g}")
print("config hash:", TRAINING_CONFIG_HASH)
if TRAINING_CONFIG_HASH != EXPECTED_CONFIG_HASH:
    raise RuntimeError(f"Config hash {TRAINING_CONFIG_HASH} is not the committed production hash "
                       f"{EXPECTED_CONFIG_HASH}: a training setting differs from the commit.")


### Production datasets (RUN_TRAINING only) -- cache-backed, no raw dataset staging

Points the training pipeline at the LOCAL cache directories, with the persistent Drive cache as a
read-only fallback for any entry not yet mirrored locally.

**The raw APTOS dataset is no longer staged.** With a complete local cache
`_build_joint_sample`'s `if not (frozen_outputs_cached and rgb_cached)` guard short-circuits, so
`lfed._load_raw_bgr` is never reached -- measured directly: with the image directory *deleted*,
the real generator yielded every sample with 0 raw reads, 0 Stage 03 calls and 0 Stage 04 calls
(`JOINT_TRAINING_ARCHITECTURE.md` Sec 44). Staging all 3,662 images was ~9.5 GiB of local disk and
~3,663 Drive file opens spent on files nothing reads.

What *is* still needed is the raw image for any entry whose **local** cache is incomplete. For
the 11 known empty-FOV images that is permanent by design: Phase 1 caches their canonical RGB but
cannot produce vessel/lesion/reliability, so every epoch re-enters the compute branch, reads the
raw image, and Stage 03 raises `EmptyFieldOfViewError` -- which the generator catches and skips.
Remove that raw image and the graceful skip becomes a hard `FileNotFoundError` that kills the run
(also measured). So exactly those entries' images are staged, and nothing else: a scope
reduction, not a behavior change. Every entry still takes the identical code path.


In [ ]:
# ==== [9] PRODUCTION DATASETS (RUN_TRAINING only) -- cache-backed, no raw dataset staging ====
MIN_FREE_FOR_TRAINING_BYTES = 3 * 1024 ** 3  # /content headroom a 50-epoch run must keep

PRODUCTION_DATASETS_BUILT = False
if RUN_TRAINING:
    import shutil

    import joint_cache_staging as jcs

    if not globals().get("LOCAL_CACHE_VERIFIED"):
        raise RuntimeError("Run [6] first: the local cache has not been verified in this runtime.")

    # Checkpoints and TensorBoard logs are written to Drive (experiment_manager resolves
    # experiment.root under the Drive mount), so training's own LOCAL footprint is small -- this
    # floor exists so the runtime itself does not run /content dry mid-epoch, which surfaces as
    # unrelated-looking failures hours in. Refuse up front rather than discover it at epoch 12.
    _free = shutil.disk_usage("/content").free
    if _free < MIN_FREE_FOR_TRAINING_BYTES:
        raise RuntimeError(
            "Refusing to start: %.2f GiB free on /content, %.2f GiB required. Nothing was "
            "staged, built or trained. Free space first -- do NOT delete the persistent Drive "
            "cache, the archive, or the local cache."
            % (_free / 1024 ** 3, MIN_FREE_FOR_TRAINING_BYTES / 1024 ** 3))
    print("local disk before training: %.2f GiB free on /content" % (_free / 1024 ** 3))

    # Copies raw images ONLY for entries without a complete local cache -- the 11 known empty-FOV
    # ones (~25 MiB, ~11 Drive opens), versus 9.52 GiB / 3,663 opens for the whole dataset.
    # Read-only with respect to Drive; loads no model; writes no cache file, so nothing frozen can
    # be regenerated here. A missing raw image would turn the generator's graceful empty-FOV skip
    # into a run-killing FileNotFoundError, so incomplete staging stops the session here.
    raw_staging = jcs.stage_raw_images_for_uncached_entries(
        train_entries + val_entries,
        cache_dir=LOCAL_CACHE_DIR,
        racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
        source_image_dir=os.path.join(colab_config.APTOS2019_RAW_DIR, "train_images"),
        local_image_dir=LOCAL_RAW_IMAGE_DIR,
    )
    jcs.print_raw_image_staging(raw_staging)
    if (raw_staging["entries_needing_raw"] != EXPECTED_EMPTY_FOV_ENTRIES
            or raw_staging["missing_at_source"] or raw_staging["drive_unreachable"]):
        raise RuntimeError("Raw-image staging for the empty-FOV entries did not complete. Nothing was trained.")
    print("")

    train_ds, val_ds = jtd.load_joint_training_datasets(
        batch_size=BATCH_SIZE,
        image_dir=LOCAL_RAW_IMAGE_DIR,
        cache_dir=LOCAL_CACHE_DIR,
        racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
        persistent_cache_dir=config.LOCAL_FEATURE_RESULTS_DIR,
        persistent_racaf_cache_dir=config.RACAF_RESULTS_DIR,
        processed_dir=NO_PRECOMPUTED_STAGE02_DIR,
    )
    PRODUCTION_DATASETS_BUILT = True
    print("Joint train/val tf.data pipelines built.")
else:
    print("RUN_TRAINING is False -- dataset pipelines were NOT built.")


### Production launch gate [P2]

Runs immediately before training and again as the first action of [T]. Read-only. If any check fails it raises before anything is created or trained:

- **Session**: `RUN_TRAINING` is True and no post-run / maintenance / diagnostic switch is on; no switch from older versions of this notebook is set; nothing else has run in this runtime; [T] has not already run in it.
- **Repository and baseline-0**: the clone is still at the commit [P1] recorded, with no tracked file modified; baseline-0 is identical to [P1]'s fingerprint and to the cross-session reference.
- **Data**: split verified by [5] (2929 / 733, per-grade counts, SHA-256); local cache verified by [6]; `train_ds` / `val_ds` built by [9] in this runtime.
- **Model**: built and verified by [7] in this runtime; not diagnostic-dirty; optimizer never stepped; weights byte-identical to construction; `mixed_float16`; `LossScaleOptimizer(inner=Adam)`; learning rate 1e-4; 409 trainable tensors; 43,338,506 trainable parameters.
- **Configuration**: batch size 2, 50 epochs, `val_QWK` / max, EarlyStopping patience 12, ReduceLROnPlateau 4 / 0.5 / 1e-6, and the config hash equal to the committed production hash.
- **New experiment** (`RESUME_EXPERIMENT_DIR = None`): no experiment with this configuration exists yet. Otherwise it names the directory to resume; `ALLOW_ADDITIONAL_EXPERIMENT = True` deliberately starts a replicate.
- **Resume**: the directory is not baseline-0, sits directly under `experiments/FinalClassification`, was created with this config hash, has checkpoints that are safe to resume from (LAST validates with the right hash, optimizer and precision; BEST present and not stale), has not finished, and no Python module or `requirements.txt` changed since it was created.


In [ ]:
# ==== [P2] PRODUCTION LAUNCH GATE -- read-only; [T] runs it again as its first action ====
def production_launch_gate():
    """Every check must pass before [T] may create or resume an experiment; raises otherwise."""
    g = globals()
    checks = []

    def check(label, ok, detail=""):
        checks.append((label, bool(ok), "" if detail is None else str(detail)))

    check("RUN_TRAINING is True", RUN_TRAINING is True)
    check("session role is production training only", SESSION_ROLE == "production training", SESSION_ROLE)
    legacy = [name for name in LEGACY_SWITCHES if g.get(name)]
    check("no legacy diagnostic/maintenance switch is set in this runtime", not legacy, ", ".join(legacy))
    check("nothing but production work has run in this runtime", not _RUNTIME_NON_TRAINING_WORK,
          ", ".join(sorted(_RUNTIME_NON_TRAINING_WORK)))
    check("[T] has not already run in this runtime", not g.get("_PRODUCTION_TRAINING_LAUNCHED"))

    baseline = verify_baseline_0_unchanged()
    check("baseline-0 unchanged since [P1] and vs the cross-session reference", not baseline,
          "; ".join(baseline))
    head = git_output("rev-parse", "HEAD")
    check("repository still at the commit [P1] recorded", head == g.get("SESSION_COMMIT"), head)
    modified = git_output("status", "--porcelain", "--untracked-files=no")
    check("no tracked file modified in the runtime's clone", not modified, modified)

    check("split verified by [5] (2929 / 733, per-grade counts, sha256)",
          g.get("SPLIT_VERIFIED") is True
          and (len(g.get("train_entries", ())), len(g.get("val_entries", ()))) == (2929, 733))
    check("local cache verified by [6] in this runtime", g.get("LOCAL_CACHE_VERIFIED") is True)
    model = g.get("joint_model")
    built = model is not None and bool(g.get("JOINT_MODEL_WEIGHTS_SHA256"))
    check("joint_model built and verified by [7] in this runtime", built)
    if built:
        optimizer = model.optimizer
        inner = getattr(optimizer, "inner_optimizer", None)
        lr = float(tf.keras.backend.get_value(optimizer.learning_rate))
        tensors = len(model.trainable_variables)
        parameters = sum(int(np.prod(v.shape)) for v in model.trainable_variables)
        check("model is not diagnostic-dirty", not getattr(model, DIAGNOSTIC_DIRTY_ATTRIBUTE, False))
        check("optimizer never stepped (iterations == 0)",
              int(tf.keras.backend.get_value(optimizer.iterations)) == 0)
        check("weights byte-identical to construction (SHA-256)",
              model_weights_sha256(model) == JOINT_MODEL_WEIGHTS_SHA256)
        check("global policy mixed_float16",
              tf.keras.mixed_precision.global_policy().name == EXPECTED_PRECISION_POLICY)
        check("layers built under mixed_float16", EXPECTED_PRECISION_POLICY in model_precision_policies(model))
        check("LossScaleOptimizer(inner=Adam)", type(optimizer).__name__ == "LossScaleOptimizer"
              and type(inner).__name__ == EXPECTED_OPTIMIZER_TYPE,
              f"{type(optimizer).__name__}(inner={type(inner).__name__})")
        check("learning rate exactly 1e-4", abs(lr - 1e-4) < 1e-9, repr(lr))
        check("409 trainable tensors", tensors == 409, tensors)
        check("43,338,506 trainable parameters", parameters == 43_338_506, f"{parameters:,}")

    check("MIXED_PRECISION True, LEARNING_RATE 1e-4", MIXED_PRECISION is True and LEARNING_RATE == 1e-4)
    check("BATCH_SIZE 2, EPOCHS 50", (BATCH_SIZE, EPOCHS) == (2, 50))
    check("monitor val_QWK, mode max", (MONITOR_METRIC, MONITOR_MODE) == ("val_QWK", "max"))
    check("EarlyStopping patience 12", EARLY_STOPPING_PATIENCE == 12)
    check("ReduceLROnPlateau patience 4, factor 0.5, min_lr 1e-6",
          (REDUCE_LR_PATIENCE, REDUCE_LR_FACTOR, MIN_LR) == (4, 0.5, 1e-6))
    live_hash = config_hash(production_config_mapping())
    check("config hash is the committed production hash",
          live_hash == TRAINING_CONFIG_HASH == EXPECTED_CONFIG_HASH, live_hash)
    check("train_ds and val_ds built by [9] in this runtime",
          g.get("PRODUCTION_DATASETS_BUILT") is True and "train_ds" in g and "val_ds" in g)

    if RESUME_EXPERIMENT_DIR is None:
        matches, problems = find_experiments_with_config(EXPECTED_CONFIG_HASH)
        check("every existing FinalClassification experiment could be read", not problems,
              "; ".join(problems))
        check("NEW experiment: none with this configuration exists yet",
              not matches or ALLOW_ADDITIONAL_EXPERIMENT,
              (f"found {matches}. To continue it, set RESUME_EXPERIMENT_DIR = {matches[-1]!r} in [S]; "
               "to deliberately start another experiment with this exact configuration, set "
               "ALLOW_ADDITIONAL_EXPERIMENT = True.") if matches else "")
        if matches and ALLOW_ADDITIONAL_EXPERIMENT:
            print(f"NOTE: ALLOW_ADDITIONAL_EXPERIMENT = True -- a new experiment is created beside {matches}")
    else:
        target = RESUME_EXPERIMENT_DIR
        check("RESUME_EXPERIMENT_DIR is not baseline-0", not is_baseline_0(target), target)
        check("RESUME_EXPERIMENT_DIR is directly under experiments/FinalClassification",
              posixpath.dirname(target) == FINAL_CLASSIFICATION_EXPERIMENTS_DIR, target)
        metadata_path = posixpath.join(target, "metadata.json")
        exists = os.path.isfile(metadata_path) and os.path.isdir(posixpath.join(target, "checkpoints"))
        check("the experiment exists (metadata.json and checkpoints/)", exists, target)
        if exists and not is_baseline_0(target):
            metadata = read_json(metadata_path)
            check("it was created with this configuration (metadata.json config_hash)",
                  metadata.get("config_hash") == EXPECTED_CONFIG_HASH, metadata.get("config_hash"))
            report = inspect_checkpoint_state(posixpath.join(target, "checkpoints"), production_expectations())
            print_checkpoint_report(report)
            check("its checkpoints are safe to resume from", not report["failures"],
                  " | ".join(report["failures"]))
            check("it has not finished", not report["finished"], report["finished"])
            created = metadata.get("git_commit_hash")
            try:
                changed = (python_changes_since(created) if created
                           else ["metadata.json records no git_commit_hash"])
            except subprocess.CalledProcessError as error:
                changed = [f"cannot compare with {created}: {error.stderr.strip()}"]
            check("no Python module or requirements.txt changed since it was created", not changed,
                  ", ".join(changed))

    width = max(len(label) for label, _, _ in checks)
    for label, ok, detail in checks:
        print(f"{'PASS' if ok else 'FAIL'}  {label:<{width}}" + (f"  -- {detail}" if detail and not ok else ""))
    failed = [label for label, ok, _ in checks if not ok]
    if failed:
        raise RuntimeError(f"LAUNCH GATE FAILED ({len(failed)} check(s)); nothing was created or trained: "
                           + "; ".join(failed))
    print("Launch gate PASSED --", "NEW experiment" if RESUME_EXPERIMENT_DIR is None
          else f"RESUME {RESUME_EXPERIMENT_DIR}")
    return checks


if RUN_TRAINING:
    production_launch_gate()
else:
    print("RUN_TRAINING is False -- the launch gate runs only in a production training session.")


### [T] Experiment + training -- the only cell that trains

Runs the launch gate [P2] itself first, then uses the existing `experiment_manager.py` / `training.Trainer` infrastructure -- no new timestamp/run system. With `RESUME_EXPERIMENT_DIR = None` it creates a new timestamped experiment and records this configuration's hash in `metadata.json` (that is how [P2] recognises the experiment in later sessions); otherwise it resumes the given experiment. It records the experiment directory for [P3] and prints the `RESUME_EXPERIMENT_DIR` line for the next session.

Passing a `CheckpointOptions` switches `Trainer` to the generation-based checkpoints in `training/checkpointing.py`, which persist the model, the full Adam state, the effective learning rate, the GLOBAL best `val_QWK` and `EarlyStopping`/`ReduceLROnPlateau`'s counters. This run spans many independent Colab sessions; weights-only checkpoints would restart Adam with zeroed moments, reset both callbacks' counters and let the first epoch of every session overwrite BEST.

**LAST vs BEST.** `checkpoints/gen_NNNNN/` is LAST: the trajectory the next session continues from, written every epoch. `checkpoints/best/` is BEST: the globally best `val_QWK` epoch across the whole experiment, the intended delivery model, never resumed from. `EarlyStopping(restore_best_weights=True)` rewinds the in-memory model at `on_train_end`; LAST is written at `on_epoch_end`, so it always holds the real end-of-epoch trajectory. [T] does not evaluate at the end: after a resume, the in-memory restore is only that session's best, so BEST and LAST are evaluated from disk by the POST-RUN cells.

A resume whose `config_hash` differs from the checkpoint's is refused by `training.checkpointing` itself; [P2] refuses it earlier, before anything is created.


In [ ]:
# ==== [T] PRODUCTION TRAINING -- the ONLY cell in this notebook that trains ====
if RUN_TRAINING:
    production_launch_gate()               # hard stop on any failed check (a NameError means [P2] was skipped)
    _PRODUCTION_TRAINING_LAUNCHED = True   # from here on, [P2] refuses to run [T] again in this runtime

    import experiment_manager
    from training import Trainer, TrainingConfig
    from training.checkpointing import CheckpointOptions

    experiment = experiment_manager.resolve_experiment(
        colab_config.DRIVE.experiment_dir("FinalClassification"), colab_config.REPO_DIR,
        resume_from=RESUME_EXPERIMENT_DIR,
        batch_size=BATCH_SIZE, epochs=EPOCHS, monitor=MONITOR_METRIC, mode=MONITOR_MODE,
        learning_rate=LEARNING_RATE, early_stopping_patience=EARLY_STOPPING_PATIENCE,
        reduce_lr_patience=REDUCE_LR_PATIENCE, reduce_lr_factor=REDUCE_LR_FACTOR, min_lr=MIN_LR,
        config_hash=TRAINING_CONFIG_HASH,   # metadata.json only (ignored on resume); [P2] matches on it
    )
    SESSION_EXPERIMENT_DIR = experiment.root
    if is_baseline_0(SESSION_EXPERIMENT_DIR):   # unreachable after [P2]; the last line of defence
        raise RuntimeError(f"Refusing to train into baseline-0: {SESSION_EXPERIMENT_DIR}")
    os.makedirs(SESSION_SAFETY_DIR, exist_ok=True)
    with open(posixpath.join(SESSION_SAFETY_DIR, "session_experiment.json"), "w") as handle:
        json.dump({"experiment_dir": SESSION_EXPERIMENT_DIR, "commit": SESSION_COMMIT}, handle)

    training_config = TrainingConfig(
        run_dir=experiment.root, epochs=EPOCHS, monitor=MONITOR_METRIC, mode=MONITOR_MODE,
        mixed_precision=MIXED_PRECISION, resume=RESUME_EXPERIMENT_DIR is not None,
        early_stopping_patience=EARLY_STOPPING_PATIENCE, reduce_lr_patience=REDUCE_LR_PATIENCE,
        reduce_lr_factor=REDUCE_LR_FACTOR, min_lr=MIN_LR,
        precision_check="error",   # refuse to train a model built under the wrong dtype policy
        repo_dir=colab_config.REPO_DIR,
        checkpoint_options=CheckpointOptions(
            experiment_id=os.path.basename(experiment.root.rstrip("/")),
            config_hash=TRAINING_CONFIG_HASH,
            dataset_version="aptos2019-joint-cache-v1",
            # Checkpoints are BUILT and VALIDATED on local disk, then copied to Drive and
            # re-validated there before the READY marker is written. Never point this at Drive.
            staging_dir="/content/checkpoint_staging",
            keep_generations=2,
        ),
    )
    trainer = Trainer(training_config)

    print("Experiment resolved:", experiment.root)
    print("Config hash:", TRAINING_CONFIG_HASH)
    print(f"Next session: RESUME_EXPERIMENT_DIR = {experiment.root!r}")
    print(f"Training joint_model: {EPOCHS} epochs, batch_size={BATCH_SIZE}, "
          f"monitor={MONITOR_METRIC}/{MONITOR_MODE}, resume={training_config.resume}")

    # The ONLY real training call in this notebook. joint_model was already built and compiled
    # above (jtm.build_and_compile_joint_model() -- dtype policy first, then build, then compile;
    # loss=joint_corn_loss only, metrics=[corn.CORNQuadraticWeightedKappa()], no auxiliary loss,
    # no class weighting); train_ds/val_ds were built above from the authoritative APTOS2019
    # split (2929 train / 733 val, jtd.load_joint_training_datasets()). Trainer.fit() handles
    # precision verification, checkpointing (generation-based, on Drive under
    # experiment.root/checkpoints/), early stopping, LR reduction, TensorBoard, and resume
    # (restoring model weights AND the full optimizer AND the callback counters) internally.
    # No class_weight is passed (Trainer.fit()'s default, None).
    history = trainer.fit(joint_model, train_ds, val_ds)

    # tensorboard/ has no natural producer under Trainer's fixed logs/ convention -- archived
    # once training completes, per experiment_manager.py's own documented convention.
    experiment_manager.archive_tensorboard_logs(experiment)

    print("Training complete.")
    print("  LAST (trajectory, resume from this):",
          ckpt.find_resumable_generation(training_config.checkpoint_dir, verbose=False))
    print("  BEST (global max val_QWK, delivery):", trainer.best_weights_path())
    print("Run [P3] now. Evaluate BEST and LAST with the POST-RUN cells in a fresh runtime.")
else:
    print("RUN_TRAINING is False -- no experiment directory was created, no Trainer was built, "
          "no dataset was iterated, no training was started.")


### End-of-session safety check [P3]

Run after [T] stops -- because training ended, or because you interrupted it -- and before disconnecting. Read-only. It fails only on real hazards, never merely because a session was interrupted:

- baseline-0 differs from [P1]'s fingerprint or from the cross-session reference;
- the session's experiment directory is baseline-0;
- the experiment holds checkpoint state of which nothing validates (the next resume would be refused), or LAST records another config hash, optimizer, precision policy or callback schedule;
- BEST is missing, invalid or stale: the session ended between writing an epoch's generation and publishing BEST. [P3] then prints the exact one-line command that republishes BEST from the generation holding that epoch, while it still exists.

An incomplete newest generation, a `latest.json` that lags by one generation, or no checkpoint yet are reported as warnings or notes: the resume logic handles them, at the cost of re-training at most one epoch. On success it prints the `RESUME_EXPERIMENT_DIR` line for the next session or, once the run has finished, the post-run instructions. It also works after a kernel restart (it reads the experiment directory and [P1]'s fingerprint back from `/content/session_safety/`).


In [ ]:
# ==== [P3] END-OF-SESSION SAFETY CHECK -- read-only; after [T] stops, before disconnecting ====
def end_of_session_check():
    failures, warnings = [], []
    baseline = verify_baseline_0_unchanged()
    failures += baseline
    print("baseline-0:", "unchanged" if not baseline else " | ".join(baseline))

    target = globals().get("SESSION_EXPERIMENT_DIR")
    recorded = posixpath.join(SESSION_SAFETY_DIR, "session_experiment.json")
    if target is None and os.path.exists(recorded):
        target = read_json(recorded)["experiment_dir"]        # after a kernel restart
    if target is None:
        target = RESUME_EXPERIMENT_DIR
    if target is None:
        print("No experiment was resolved in this runtime -- only baseline-0 was checked.")
    elif is_baseline_0(target):
        failures.append(f"the session's experiment directory is baseline-0: {target}")
    else:
        print("experiment:", target)
        report = inspect_checkpoint_state(posixpath.join(target, "checkpoints"), production_expectations())
        print_checkpoint_report(report)
        failures += report["failures"]
        warnings += report["warnings"]
        if report["finished"] and not report["failures"]:
            print(f"FINISHED ({report['finished']}). Do not resume it. Next: a fresh runtime with "
                  f"RUN_POST_RUN_EVALUATION = True, POST_RUN_EXPERIMENT_DIR = {target!r}")
        elif not report["failures"]:
            print(f"Next session: RESUME_EXPERIMENT_DIR = {target!r}")
    if failures:
        raise RuntimeError(f"END-OF-SESSION CHECK FAILED ({len(failures)}): " + " | ".join(failures))
    print(f"End-of-session check PASSED ({len(warnings)} warning(s)). Optional before disconnecting: "
          "from google.colab import drive; drive.flush_and_unmount()")


end_of_session_check()


## POST-RUN evaluation (read-only)

Run after the experiment has finished (EarlyStopping stopped it, or epoch 50), in a **fresh runtime**: set `RUN_POST_RUN_EVALUATION = True` and `POST_RUN_EXPERIMENT_DIR` in [S], run [S] to [8], then [R1] to [R4], then the diagnostics [D2] to [D13], then [P3]. Never during training: these cells do nothing unless `RUN_POST_RUN_EVALUATION` is on, and [S] refuses to switch it on together with `RUN_TRAINING`.

Nothing here trains, updates an optimizer or writes to Drive. Weights are copied to local disk and verified against their manifests; evaluation is `predict_on_batch` only; baseline-0 is only read. [R3] loads checkpoint weights into `joint_model` and marks it diagnostic-dirty, so this runtime can never train afterwards.

| Cell | Covers |
|---|---|
| [R1] | 1 checkpoint integrity; 8 BEST_full; 9 BEST_p8; 11 technical validity (checkpoints and training log) |
| [R2] | verified local copies of the BEST and LAST weights; the production validation pipeline |
| [R3] | 2 BEST vs LAST; 3-6 BEST/LAST confusion matrices and prediction histograms; 7 recorded vs recomputed loss/QWK |
| [R4] | 10 baseline-0 comparison and the pre-registered decision rule; 11 technical-validity summary |

Pre-registered endpoints (LR = 1e-4 plan, section G). **Primary:** BEST_p8 -- the best `val_QWK` up to the epoch a patience-8 EarlyStopping would have stopped at (EarlyStopping does not alter training before it stops, so that run is an exact prefix of this one) -- against baseline-0's 0.1796. **Secondary:** BEST_full (the global BEST), BEST's validation loss against the constant class-prior reference 0.6548, BEST's confusion matrix and prediction histogram, and the number of epochs with `val_QWK < 0.01` (baseline-0: 5). **Rule:** materially better if BEST_p8 >= 0.2796 and BEST's val_loss <= 0.6048 and BEST predicts >= 3 grades (confirmed if the bootstrap 95% CI of BEST's QWK has its lower bound above 0.1796); not materially better if BEST_full < 0.2296; otherwise inconclusive.


In [ ]:
# ==== [R1] POST-RUN: checkpoint integrity, training-log validity, BEST_full, BEST_p8 -- read-only ====
def replay_early_stopping(values, patience):
    """Keras EarlyStopping replayed over a val_QWK log (mode max, min_delta 0, no baseline):
    (best value, best index, index at which it stops -- or None if it never stops)."""
    best, best_index = None, None
    for index, value in enumerate(values):
        if best is None or value > best:
            best, best_index = value, index
        elif index - best_index >= patience:
            return best, best_index, index
    return best, best_index, None


if RUN_POST_RUN_EVALUATION:
    import csv

    EXPECTED_TRAIN_BATCHES = 1461   # 2929 train entries minus the 7-8 empty-FOV ones, batch 2 (baseline-0: every epoch)
    EXPECTED_VAL_BATCHES = 365      # 733 val entries minus the 3-4 empty-FOV ones, batch 2
    POST_RUN_CHECKS = {}            # technical validity: name -> (ok, detail); summarised by [R4]

    EVAL_EXPERIMENT_DIR = POST_RUN_EXPERIMENT_DIR
    if is_baseline_0(EVAL_EXPERIMENT_DIR):
        raise RuntimeError("POST_RUN_EXPERIMENT_DIR is baseline-0: it is the comparison reference [R4] "
                           "reads, not an evaluation target.")
    if posixpath.dirname(EVAL_EXPERIMENT_DIR) != FINAL_CLASSIFICATION_EXPERIMENTS_DIR:
        raise RuntimeError(f"{EVAL_EXPERIMENT_DIR} is not directly under {FINAL_CLASSIFICATION_EXPERIMENTS_DIR}")
    EVAL_CHECKPOINT_DIR = posixpath.join(EVAL_EXPERIMENT_DIR, "checkpoints")
    eval_metadata = read_json(posixpath.join(EVAL_EXPERIMENT_DIR, "metadata.json"))

    print("=== 1. CHECKPOINT INTEGRITY ===")
    eval_report = inspect_checkpoint_state(EVAL_CHECKPOINT_DIR, production_expectations())
    print_checkpoint_report(eval_report)
    if eval_report["state"] is None or not (eval_report["best"] or {}).get("valid"):
        raise RuntimeError("There is no valid LAST generation and BEST checkpoint to evaluate.")
    LAST_DIR = posixpath.join(EVAL_CHECKPOINT_DIR, eval_report["resumable"])
    BEST_DIR = ckpt.best_dir(EVAL_CHECKPOINT_DIR)
    last_state, best_state = eval_report["state"], ckpt.read_state(BEST_DIR)
    EVAL_FINISHED = eval_report["finished"]
    POST_RUN_CHECKS["checkpoint integrity: no failure"] = (not eval_report["failures"],
                                                           " | ".join(eval_report["failures"]))
    _hashes = {g["name"]: read_json(posixpath.join(EVAL_CHECKPOINT_DIR, g["name"], ckpt.MANIFEST_FILENAME)).get("config_hash")
               for g in eval_report["generations"] if g["valid"]}
    _hashes["best"] = read_json(posixpath.join(BEST_DIR, ckpt.MANIFEST_FILENAME)).get("config_hash")
    _hashes["metadata.json"] = eval_metadata.get("config_hash")
    POST_RUN_CHECKS["production config hash in metadata.json and every checkpoint"] = (
        set(_hashes.values()) == {EXPECTED_CONFIG_HASH}, _hashes)
    _intended = {"batch_size": BATCH_SIZE, "epochs": EPOCHS, "monitor": MONITOR_METRIC, "mode": MONITOR_MODE,
                 "learning_rate": LEARNING_RATE, "early_stopping_patience": EARLY_STOPPING_PATIENCE,
                 "reduce_lr_patience": REDUCE_LR_PATIENCE, "reduce_lr_factor": REDUCE_LR_FACTOR, "min_lr": MIN_LR}
    _recorded = {key: eval_metadata.get(key) for key in _intended}
    POST_RUN_CHECKS["metadata.json records the production configuration"] = (_recorded == _intended, _recorded)

    print("\n=== training log: checkpoints/metrics.csv ===")
    _rows, _superseded = {}, 0
    with open(posixpath.join(EVAL_CHECKPOINT_DIR, "metrics.csv"), newline="") as handle:
        for row in csv.DictReader(handle):
            epoch = int(row["epoch"])
            _superseded += epoch in _rows
            _rows[epoch] = row   # an epoch re-run after an interrupted session: the later row is the checkpointed one
    EPOCH_INDICES = sorted(_rows)

    def _logged(key):
        values = []
        for epoch in EPOCH_INDICES:
            try:
                values.append(float(_rows[epoch][key]))
            except (KeyError, TypeError, ValueError):
                values.append(float("nan"))
        return values

    VAL_QWK, VAL_LOSS, TRAIN_LOSS, TRAIN_QWK, LOGGED_LR = (
        _logged(key) for key in ("val_QWK", "val_loss", "loss", "QWK", "learning_rate"))
    print(f"{len(EPOCH_INDICES)} epoch(s) logged; {_superseded} row(s) superseded by a re-run of the same epoch")
    for i, epoch in enumerate(EPOCH_INDICES):
        print(f"  epoch {epoch + 1:2d}: loss {TRAIN_LOSS[i]:.4f} QWK {TRAIN_QWK[i]:.4f} | "
              f"val_loss {VAL_LOSS[i]:.4f} val_QWK {VAL_QWK[i]:.4f} | lr {LOGGED_LR[i]:.3g}")

    POST_RUN_CHECKS["epochs logged are 1..N, N = LAST's completed epochs"] = (
        EPOCH_INDICES == list(range(last_state.completed_epoch)),
        f"logged {len(EPOCH_INDICES)}, LAST completed {last_state.completed_epoch}")
    POST_RUN_CHECKS["no NaN/inf in loss, QWK, val_loss, val_QWK"] = (
        all(math.isfinite(v) for v in TRAIN_LOSS + TRAIN_QWK + VAL_LOSS + VAL_QWK), "")
    _lr_ok, _cuts = bool(LOGGED_LR) and abs(LOGGED_LR[0] - LEARNING_RATE) < 1e-9, 0
    for _previous, _current in zip(LOGGED_LR, LOGGED_LR[1:] + [last_state.learning_rate]):
        if math.isclose(_current, _previous, rel_tol=1e-6):
            continue
        if math.isclose(_current, max(_previous * REDUCE_LR_FACTOR, MIN_LR), rel_tol=1e-5):
            _cuts += 1
            continue
        _lr_ok = False
    POST_RUN_CHECKS["LR starts at 1e-4 and changes only by ReduceLROnPlateau halvings"] = (
        _lr_ok, f"{_cuts} reduction(s); logged {[f'{v:.3g}' for v in LOGGED_LR]}, next {last_state.learning_rate:.3g}")
    _iterations = int(last_state.optimizer.get("iterations"))
    _batches = EXPECTED_TRAIN_BATCHES * last_state.completed_epoch
    POST_RUN_CHECKS["optimizer iterations = 1461 x epochs minus loss-scale skips (< 1%)"] = (
        0 <= _batches - _iterations <= 0.01 * _batches,
        f"{_iterations:,} of {_batches:,}: {_batches - _iterations} step(s) skipped by the loss scaler")
    _best_i = max(range(len(VAL_QWK)), key=lambda i: (VAL_QWK[i], -i))
    POST_RUN_CHECKS["best/ = the first epoch with the maximum logged val_QWK"] = (
        best_state.best_epoch == EPOCH_INDICES[_best_i] and abs(best_state.best_metric - VAL_QWK[_best_i]) < 1e-9,
        f"best/ epoch {best_state.best_epoch + 1} = {best_state.best_metric}; "
        f"log epoch {EPOCH_INDICES[_best_i] + 1} = {VAL_QWK[_best_i]}")
    _stopped = int((last_state.early_stopping or {}).get("stopped_epoch") or 0)
    if _stopped > 0:
        _replay_stop = replay_early_stopping(VAL_QWK, EARLY_STOPPING_PATIENCE)[2]
        POST_RUN_CHECKS["EarlyStopping stopped where a patience-12 replay of the log stops"] = (
            _replay_stop is not None and EPOCH_INDICES[_replay_stop] == _stopped,
            f"replay: {None if _replay_stop is None else EPOCH_INDICES[_replay_stop] + 1}; recorded: {_stopped + 1}")

    print("\n=== 8. BEST_full   9. BEST_p8 ===")
    BEST_FULL, BEST_FULL_EPOCH = VAL_QWK[_best_i], EPOCH_INDICES[_best_i]
    BEST_P8, _p8_index, _p8_stop = replay_early_stopping(VAL_QWK, 8)
    BEST_P8_EPOCH = EPOCH_INDICES[_p8_index]
    print(f"BEST_full = {BEST_FULL:.6f} at epoch {BEST_FULL_EPOCH + 1} "
          f"(checkpoints/best: {best_state.best_metric:.6f} at epoch {best_state.best_epoch + 1})")
    print(f"BEST_p8   = {BEST_P8:.6f} at epoch {BEST_P8_EPOCH + 1}; a patience-8 EarlyStopping would have stopped "
          + (f"at epoch {EPOCH_INDICES[_p8_stop] + 1}" if _p8_stop is not None
             else "-- not within the logged epochs (so BEST_p8 = BEST_full)"))
    LOW_QWK_EPOCHS = [epoch + 1 for epoch, q in zip(EPOCH_INDICES, VAL_QWK) if q < 0.01]
    print(f"epochs with val_QWK < 0.01: {len(LOW_QWK_EPOCHS)} {LOW_QWK_EPOCHS} (baseline-0: 5, epochs 4-8)")
    print("status:", EVAL_FINISHED or "NOT FINISHED -- every figure here is interim")
    print("\n=== 1 + 11. technical validity: checkpoints and training log ===")
    for _name, (_ok, _detail) in POST_RUN_CHECKS.items():
        print(("PASS  " if _ok else "FAIL  ") + _name + (f"  -- {_detail}" if _detail != "" else ""))
else:
    print("RUN_POST_RUN_EVALUATION is False -- nothing was read or evaluated.")


In [ ]:
# ==== [R2] POST-RUN: verified local copies of BEST/LAST weights + the production validation pipeline ====
if RUN_POST_RUN_EVALUATION:
    import shutil

    import joint_cache_staging as jcs

    if not globals().get("LOCAL_CACHE_VERIFIED") or globals().get("joint_model") is None:
        raise RuntimeError("Run [6] and [7] first (local cache verified, joint_model built).")
    POST_RUN_LOCAL_DIR = "/content/post_run_eval"   # LOCAL only -- nothing is written under the experiment

    def copy_verified_weights(label, directory):
        """Drive -> local copy of model.weights.h5, checked against that checkpoint's own manifest."""
        manifest = read_json(posixpath.join(directory, ckpt.MANIFEST_FILENAME))
        local = posixpath.join(POST_RUN_LOCAL_DIR, label, ckpt.MODEL_WEIGHTS_FILENAME)
        os.makedirs(posixpath.dirname(local), exist_ok=True)
        shutil.copyfile(posixpath.join(directory, ckpt.MODEL_WEIGHTS_FILENAME), local)
        if ckpt.sha256_file(local) != manifest["files"][ckpt.MODEL_WEIGHTS_FILENAME]["sha256"]:
            raise RuntimeError(f"{label}: the local copy does not match its manifest sha256")
        print(f"{label}: {directory} -> {local} (sha256 verified)")
        return local

    BEST_WEIGHTS_LOCAL = copy_verified_weights("BEST", BEST_DIR)
    LAST_WEIGHTS_LOCAL = copy_verified_weights("LAST", LAST_DIR)

    # The production pipeline, from the SAME calls [9] makes; only its validation half is used.
    # Validation is unshuffled and unaugmented, so these are exactly the batches Keras validated on.
    _raw = jcs.stage_raw_images_for_uncached_entries(
        train_entries + val_entries,
        cache_dir=LOCAL_CACHE_DIR,
        racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
        source_image_dir=os.path.join(colab_config.APTOS2019_RAW_DIR, "train_images"),
        local_image_dir=LOCAL_RAW_IMAGE_DIR,
    )
    jcs.print_raw_image_staging(_raw)
    if _raw["missing_at_source"] or _raw["drive_unreachable"]:
        raise RuntimeError("Raw-image staging for the empty-FOV entries did not complete.")
    _unused_train_ds, POST_RUN_VAL_DS = jtd.load_joint_training_datasets(
        batch_size=BATCH_SIZE,
        image_dir=LOCAL_RAW_IMAGE_DIR,
        cache_dir=LOCAL_CACHE_DIR,
        racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
        persistent_cache_dir=config.LOCAL_FEATURE_RESULTS_DIR,
        persistent_racaf_cache_dir=config.RACAF_RESULTS_DIR,
        processed_dir=NO_PRECOMPUTED_STAGE02_DIR,
    )
    del _unused_train_ds   # never iterated
    print("Validation pipeline built (the production call; only the validation half is used).")
else:
    print("RUN_POST_RUN_EVALUATION is False -- nothing was copied or built.")


In [ ]:
# ==== [R3] POST-RUN: BEST vs LAST on the validation split -- inference only, no optimizer, no writes ====
if RUN_POST_RUN_EVALUATION:
    import corn

    _RUNTIME_NON_TRAINING_WORK.add("post-run evaluation")
    # Checkpoint weights are loaded into joint_model next: from here on Trainer.fit() and [P2] refuse it.
    setattr(joint_model, DIAGNOSTIC_DIRTY_ATTRIBUTE, True)

    GRADES = list(range(corn.NUM_GRADES))
    QWK_WEIGHTS = np.subtract.outer(GRADES, GRADES) ** 2 / float((corn.NUM_GRADES - 1) ** 2)
    _train_grades = np.array([grade for _, grade in train_entries])
    # Reference only: constant logits = logit(P(grade > k | grade >= k)) from the TRAIN split.
    PRIOR_LOGITS = np.array(
        [np.log(p / (1 - p)) for p in
         (np.mean(_train_grades[_train_grades >= k] > k) for k in range(corn.NUM_THRESHOLDS))],
        dtype=np.float32)

    def qwk_from_confusion(confusion):
        """training.metrics.QuadraticWeightedKappa.result(), in NumPy."""
        confusion = np.asarray(confusion, dtype=np.float64)
        total = confusion.sum()
        if total == 0:
            return 0.0
        expected = np.outer(confusion.sum(1), confusion.sum(0)) / total
        return float(1.0 - (QWK_WEIGHTS * confusion).sum() / max((QWK_WEIGHTS * expected).sum(), 1e-7))

    def evaluate_weights(label, weights_path, recorded_logs):
        ckpt.load_model_weights_only(joint_model, weights_path)     # model variables only; optimizer detached
        metric = corn.CORNQuadraticWeightedKappa()                   # the production val_QWK metric, fresh state
        loss_sum = prior_sum = 0.0
        batches, y_true, y_pred = 0, [], []
        for (stage5, stage6, reliability), grades in POST_RUN_VAL_DS:
            logits = joint_model.predict_on_batch([stage5, stage6, tf.reshape(reliability, (-1, 1))])
            n = int(grades.shape[0])
            loss_sum += float(jtm.joint_corn_loss(grades, logits)) * n   # Keras: per-batch loss weighted by batch size
            prior_sum += float(jtm.joint_corn_loss(grades, np.tile(PRIOR_LOGITS, (n, 1)))) * n
            metric.update_state(grades, logits)
            p_cum = tf.math.cumprod(tf.sigmoid(tf.cast(logits, tf.float32)), axis=-1)   # the metric's own decode
            y_pred += tf.reduce_sum(tf.cast(p_cum > 0.5, tf.int32), axis=-1).numpy().tolist()
            y_true += [int(grade) for grade in grades.numpy()]
            batches += 1
        samples = len(y_true)
        confusion = metric.confusion.numpy().astype(np.int64)          # rows = true grade, columns = predicted
        per_sample = np.zeros_like(confusion)
        np.add.at(per_sample, (np.array(y_true), np.array(y_pred)), 1)
        result = {"label": label, "batches": batches, "samples": samples, "loss": loss_sum / samples,
                  "qwk": float(metric.result()), "qwk_from_matrix": qwk_from_confusion(confusion),
                  "prior_loss": prior_sum / samples, "confusion": confusion,
                  "y_true": np.array(y_true), "y_pred": np.array(y_pred),
                  "decode_matches_metric": bool(np.array_equal(per_sample, confusion)),
                  "recorded_loss": recorded_logs.get("val_loss"), "recorded_qwk": recorded_logs.get("val_QWK")}
        print(f"\n=== {label} ===")
        print(f"batches: {batches} (expected {EXPECTED_VAL_BATCHES})   samples: {samples}")
        print(f"val_loss recomputed {result['loss']:.6f} | recorded {result['recorded_loss']}")
        print(f"val_QWK  recomputed {result['qwk']:.6f} | recorded {result['recorded_qwk']} | "
              f"from the matrix {result['qwk_from_matrix']:.6f}")
        print(f"constant class-prior loss on the same batches: {result['prior_loss']:.6f}")
        print("confusion matrix (rows = true grade, columns = predicted grade):")
        print("        " + "".join(f"{'pred ' + str(g):>9}" for g in GRADES))
        for g in GRADES:
            print(f"true {g} " + "".join(f"{v:>9d}" for v in confusion[g]))
        predicted, actual = confusion.sum(0), confusion.sum(1)
        print("prediction histogram: " + "  ".join(
            f"{g}: {predicted[g]} ({100.0 * predicted[g] / samples:.1f}%)" for g in GRADES))
        print("true-label histogram: " + "  ".join(
            f"{g}: {actual[g]} ({100.0 * actual[g] / samples:.1f}%)" for g in GRADES))
        return result

    print("=== 2. BEST vs LAST  (3-6 confusion matrices and histograms, 7 recorded vs recomputed) ===")
    BEST_RESULT = evaluate_weights(f"BEST  checkpoints/best (epoch {best_state.best_epoch + 1})",
                                   BEST_WEIGHTS_LOCAL, (best_state.extra or {}).get("epoch_logs", {}))
    LAST_RESULT = evaluate_weights(f"LAST  checkpoints/{eval_report['resumable']} (epoch {last_state.completed_epoch})",
                                   LAST_WEIGHTS_LOCAL, (last_state.extra or {}).get("epoch_logs", {}))

    print(f"\n{'':6}{'val_loss':>12}{'val_QWK':>12}{'grades predicted':>18}")
    for _tag, _result in (("BEST", BEST_RESULT), ("LAST", LAST_RESULT)):
        print(f"{_tag:6}{_result['loss']:>12.6f}{_result['qwk']:>12.6f}"
              f"{int((_result['confusion'].sum(0) > 0).sum()):>18d}")
        POST_RUN_CHECKS[f"{_tag}: {EXPECTED_VAL_BATCHES} validation batches evaluated"] = (
            _result["batches"] == EXPECTED_VAL_BATCHES, _result["batches"])
        POST_RUN_CHECKS[f"{_tag}: per-sample decode reproduces the metric's confusion matrix"] = (
            _result["decode_matches_metric"], "")
        _dl = None if _result["recorded_loss"] is None else abs(_result["loss"] - _result["recorded_loss"])
        POST_RUN_CHECKS[f"{_tag}: recomputed val_loss = recorded (|diff| < 1e-3)"] = (
            _dl is not None and _dl < 1e-3, f"|diff| {_dl}")
        # One borderline float16 decode flip moves QWK by ~0.003 on ~730 images: hence the wider tolerance.
        _dq = None if _result["recorded_qwk"] is None else abs(_result["qwk"] - _result["recorded_qwk"])
        POST_RUN_CHECKS[f"{_tag}: recomputed val_QWK = recorded (|diff| < 0.01)"] = (
            _dq is not None and _dq < 0.01, f"|diff| {_dq}")
else:
    print("RUN_POST_RUN_EVALUATION is False -- no weights were loaded and nothing was evaluated.")


In [ ]:
# ==== [R4] POST-RUN: baseline-0 comparison, pre-registered decision rule, technical validity ====
if RUN_POST_RUN_EVALUATION:
    import csv

    import corn

    # Pre-registered in the LR = 1e-4 plan (section G) before the run started -- do not tune.
    BASELINE_0_BEST_VAL_QWK = 0.17963171   # baseline-0 checkpoints/best (epoch 3)
    PRIOR_REFERENCE_VAL_LOSS = 0.6548      # constant class-prior model, Keras-style val_loss
    MATERIALLY_BETTER_QWK = 0.2796         # baseline-0 + 0.10, applied to BEST_p8
    NOT_BETTER_QWK = 0.2296                # baseline-0 + 0.05, applied to BEST_full
    LOSS_MARGIN = 0.05                     # BEST's val_loss must be at least this far below the reference
    MIN_PREDICTED_GRADES = 3

    print("=== 10. BASELINE-0 COMPARISON (baseline-0 is only read) ===")
    _b0_checkpoints = posixpath.join(BASELINE_0_DIR, "checkpoints")
    _b0_best = ckpt.read_state(ckpt.best_dir(_b0_checkpoints))
    if abs(_b0_best.best_metric - BASELINE_0_BEST_VAL_QWK) > 1e-6:
        raise RuntimeError(f"baseline-0's BEST on disk ({_b0_best.best_metric}) is not the pre-registered "
                           f"{BASELINE_0_BEST_VAL_QWK}")
    with open(posixpath.join(_b0_checkpoints, "metrics.csv"), newline="") as handle:
        _b0_log = {int(row["epoch"]): float(row["val_QWK"]) for row in csv.DictReader(handle)}
    _b0_low = [epoch + 1 for epoch, q in sorted(_b0_log.items()) if q < 0.01]

    _rng = np.random.default_rng(20260911)     # fixed, so the interval is reproducible
    _boot = []
    for _ in range(2000):
        _idx = _rng.integers(0, BEST_RESULT["samples"], BEST_RESULT["samples"])
        _conf = np.zeros((corn.NUM_GRADES, corn.NUM_GRADES))
        np.add.at(_conf, (BEST_RESULT["y_true"][_idx], BEST_RESULT["y_pred"][_idx]), 1)
        _boot.append(qwk_from_confusion(_conf))
    CI_LOW, CI_HIGH = (float(v) for v in np.percentile(_boot, [2.5, 97.5]))
    PREDICTED_GRADES = int((BEST_RESULT["confusion"].sum(0) > 0).sum())

    _table = [
        ("BEST_p8 val_QWK (primary)", f"{BEST_P8:.4f} (epoch {BEST_P8_EPOCH + 1})",
         f"{_b0_best.best_metric:.4f} (epoch {_b0_best.best_epoch + 1})", f">= {MATERIALLY_BETTER_QWK}: materially better"),
        ("BEST_full val_QWK", f"{BEST_FULL:.4f} (epoch {BEST_FULL_EPOCH + 1})",
         f"{_b0_best.best_metric:.4f}", f"< {NOT_BETTER_QWK}: not materially better"),
        ("BEST val_loss (recomputed)", f"{BEST_RESULT['loss']:.4f}",
         f"prior {PRIOR_REFERENCE_VAL_LOSS} (here {BEST_RESULT['prior_loss']:.4f})",
         f"<= {PRIOR_REFERENCE_VAL_LOSS - LOSS_MARGIN:.4f}"),
        ("grades BEST predicts", str(PREDICTED_GRADES), "--", f">= {MIN_PREDICTED_GRADES}"),
        ("BEST QWK bootstrap 95% CI", f"[{CI_LOW:.4f}, {CI_HIGH:.4f}]", "--",
         f"lower bound > {BASELINE_0_BEST_VAL_QWK:.4f} confirms"),
        ("epochs with val_QWK < 0.01", str(len(LOW_QWK_EPOCHS)), f"{len(_b0_low)} {_b0_low}", "secondary"),
    ]
    print(f"{'':28}{'LR 1e-4':>24}{'baseline-0':>34}   rule")
    for _name, _new, _old, _rule in _table:
        print(f"{_name:28}{_new:>24}{_old:>34}   {_rule}")

    print("\n=== 11. TECHNICAL VALIDITY SUMMARY ===")
    _baseline = verify_baseline_0_unchanged()
    POST_RUN_CHECKS["baseline-0 unchanged"] = (not _baseline, " | ".join(_baseline))
    for _name, (_ok, _detail) in POST_RUN_CHECKS.items():
        print(("PASS  " if _ok else "FAIL  ") + _name + ("" if _ok else f"  -- {_detail}"))
    TECHNICALLY_VALID = all(ok for ok, _ in POST_RUN_CHECKS.values())

    if not TECHNICALLY_VALID:
        VERDICT = "INVALID -- a technical-validity check failed, so the comparison is not interpretable."
    elif not EVAL_FINISHED:
        VERDICT = "INTERIM -- the experiment has not finished; the pre-registered rule is not applied."
    elif (BEST_P8 >= MATERIALLY_BETTER_QWK and BEST_RESULT["loss"] <= PRIOR_REFERENCE_VAL_LOSS - LOSS_MARGIN
          and PREDICTED_GRADES >= MIN_PREDICTED_GRADES):
        VERDICT = "MATERIALLY BETTER than baseline-0 -- " + (
            f"confirmed: bootstrap lower bound {CI_LOW:.4f} > {BASELINE_0_BEST_VAL_QWK:.4f}"
            if CI_LOW > BASELINE_0_BEST_VAL_QWK else
            f"NOT confirmed: bootstrap lower bound {CI_LOW:.4f} <= {BASELINE_0_BEST_VAL_QWK:.4f}")
    elif BEST_FULL < NOT_BETTER_QWK:
        VERDICT = "NOT MATERIALLY BETTER -- the learning rate alone does not explain baseline-0."
    else:
        VERDICT = ("INCONCLUSIVE -- between the pre-registered thresholds; separating the learning-rate "
                   "effect from run-to-run variation needs a repeat run.")
    print("\nPRE-REGISTERED VERDICT:", VERDICT)
    print("Run [P3] before disconnecting. Do not train in this runtime.")
else:
    print("RUN_POST_RUN_EVALUATION is False -- nothing was compared.")


## POST-RUN diagnostics and final-model finalization [D2]-[D13] (read-only until [D11])

Run after [R1]-[R4] in the same post-run session, in order. Every cell here is gated on `RUN_POST_RUN_EVALUATION`, uses inference only (`predict_on_batch`), and never trains, never touches an optimizer, and never modifies baseline-0, the experiment's `checkpoints/`, the training data or the frozen Stage 03/04 artifacts. The only writes are into this experiment's own `evaluation/` folder (diagnostic tables, plots and reports) and, in [D11], a NEW final-model directory under `exported_models/FinalClassification/`, which is never overwritten.

| Cell | What it produces |
|---|---|
| [D2] | The evaluation manifest: the exact validation population (and the explicitly reported skipped empty-FOV ids) every later cell uses |
| [D3] | Accuracy, QWK, macro/weighted P/R/F1, balanced accuracy, MAE and the per-class table, cross-checked against [R3]/[R4] |
| [D4] | Confusion matrix, ordinal error-distance distribution, adjacent vs non-adjacent errors, every off-diagonal pair |
| [D5] | Per-sample CORN logits, cumulative probabilities, reconstructed class distribution, margins -- as CSV and JSON |
| [D6] | Confidence, uncertainty and calibration diagnostics (threshold-level and, from the valid reconstruction, Brier/ECE) with plots |
| [D7] | Deterministic misclassification case lists, each with its recorded sorting rule |
| [D8] | Image panels covering every part of the error structure, from the cached canonical RGB the model actually consumed |
| [D9] | The full BEST vs LAST comparison on the identical population |
| [D10] | Final-artifact verification of the BEST checkpoint before anything is archived |
| [D11] | The final model archive (a new `<experiment>_BEST` directory) with full provenance |
| [D12] | The machine-readable JSON report and the human-readable Markdown summary |
| [D13] | Reloads the archived weights, re-verifies QWK and SHA-256, and re-checks that nothing else changed |

**Cost.** [D3] and [D13] each run one full validation pass, in addition to the two [R3] already ran, so budget roughly four validation passes for the whole post-run session.

**BEST is the delivery model.** LAST is only the resume trajectory; [D9] compares them and [D11] archives BEST alone.

**Not to be confused with [D1]**, the optional tiny-subset overfit test at the end of the notebook, which is a training-pipeline diagnostic and unrelated to this chain.

These are diagnostics on ONE validation split: not an external test set, not a claim of clinical validity or generalization.


In [ ]:
# ==== [D2] POST-RUN: evaluation manifest -- the exact validation population every diagnostic uses ====
if RUN_POST_RUN_EVALUATION:
    import collections
    import csv

    import downstream_split
    import joint_cache_staging as jcs

    DIAG_DIR = posixpath.join(EVAL_EXPERIMENT_DIR, "evaluation")   # created by create_experiment()
    DIAG_PLOTS_DIR = posixpath.join(DIAG_DIR, "plots")
    os.makedirs(DIAG_PLOTS_DIR, exist_ok=True)
    DIAG_CHECKS = {}          # name -> (ok, detail); summarised by [D12] and [D13]
    DIAG_OUTPUTS = []         # every file these cells write


    def diag_path(name):
        return posixpath.join(DIAG_DIR, name)


    def _json_default(value):
        if isinstance(value, np.generic):
            return value.item()
        if isinstance(value, np.ndarray):
            return value.tolist()
        return str(value)


    def save_json(name, payload):
        path = diag_path(name)
        with open(path, "w") as handle:
            json.dump(payload, handle, indent=2, default=_json_default)
        DIAG_OUTPUTS.append(path)
        return path


    def save_csv(name, rows, fieldnames=None):
        rows = [dict(row) for row in rows]
        path = diag_path(name)
        with open(path, "w", newline="") as handle:
            writer = csv.DictWriter(handle, fieldnames=fieldnames or (list(rows[0]) if rows else ["empty"]))
            writer.writeheader()
            writer.writerows(rows)
        DIAG_OUTPUTS.append(path)
        return path


    def diag_check(name, ok, detail=""):
        DIAG_CHECKS[name] = (bool(ok), str(detail))
        print(("PASS  " if ok else "FAIL  ") + name + (f"  -- {detail}" if detail else ""))
        return bool(ok)


    # The generator skips exactly the entries whose local cache is incomplete, because Stage 03
    # raises EmptyFieldOfViewError for them ([6] proved none of them is recoverable from Drive).
    _incomplete_val = {id_code for id_code, _ in jcs.entries_missing_local_cache(
        val_entries, LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR, jtd.STAGE5_IMAGE_SIZE)}
    _missing_everywhere = {id_code for id_code, _ in LOCAL_CACHE_REPORT["missing_everywhere_ids"]}
    SKIPPED_VAL_IDS = sorted(_incomplete_val)
    EXPECTED_EVALUATED_ENTRIES = [entry for entry in val_entries if entry[0] not in _incomplete_val]
    EVALUATED_IDS = [id_code for id_code, _ in EXPECTED_EVALUATED_ENTRIES]
    EVALUATED_GRADES = np.array([grade for _, grade in EXPECTED_EVALUATED_ENTRIES], dtype=int)
    # Fingerprints taken BEFORE any diagnostic runs; [D13] re-checks them.
    CHECKPOINTS_FINGERPRINT_BEFORE = tree_fingerprint(EVAL_CHECKPOINT_DIR)

    _best_manifest = read_json(posixpath.join(BEST_DIR, ckpt.MANIFEST_FILENAME))
    EVAL_MANIFEST = {
        "experiment_dir": EVAL_EXPERIMENT_DIR,
        "experiment_id": posixpath.basename(EVAL_EXPERIMENT_DIR),
        "checkpoint_evaluated": BEST_DIR,
        "checkpoint_source_generation": _best_manifest.get("source_generation"),
        "checkpoint_sha256": _best_manifest["files"][ckpt.MODEL_WEIGHTS_FILENAME]["sha256"],
        "last_checkpoint": LAST_DIR,
        "config_hash": _best_manifest.get("config_hash"),
        "git_commit_hash": _best_manifest.get("git_commit_hash"),
        "split_manifest": downstream_split.DEFAULT_SPLIT_MANIFEST,
        "split_sha256": EXPECTED_SPLIT_SHA256,
        "split_seed": downstream_split.DEFAULT_SEED,
        "split_val_fraction": downstream_split.DEFAULT_VAL_SPLIT,
        "train_entries": len(train_entries),
        "validation_entries": len(val_entries),
        "skipped_entries": len(SKIPPED_VAL_IDS),
        "skipped_ids": SKIPPED_VAL_IDS,
        "skipped_reason": "empty field of view: Stage 03 finds no fundus disk, so the generator skips the image",
        "evaluated_entries": len(EXPECTED_EVALUATED_ENTRIES),
        "batch_size": BATCH_SIZE,
        "expected_batches": EXPECTED_VAL_BATCHES,
        "dataset_version": _best_manifest.get("dataset_version"),
        "image_size": list(jtd.STAGE5_IMAGE_SIZE),
        "stage5_input_shape": list(jtm.STAGE5_INPUT_SHAPE),
        "stage6_input_shape": list(jtm.STAGE6_INPUT_SHAPE),
        "augmentation": False,          # validation branch of load_joint_training_datasets
        "shuffle": False,
        "grades": GRADES,
        "class_counts_all_validation": {int(g): int(c) for g, c in
                                        sorted(collections.Counter(g for _, g in val_entries).items())},
        "class_counts_evaluated": {int(g): int(c) for g, c in
                                   sorted(collections.Counter(EVALUATED_GRADES.tolist()).items())},
        "local_cache_fully_local": LOCAL_CACHE_REPORT["fully_local"],
        "local_cache_drive_fallback": LOCAL_CACHE_REPORT["drive_fallback_required"],
        "recorded_best_epoch": best_state.best_epoch + 1,
        "recorded_best_val_qwk": best_state.best_metric,
        "recorded_last_epoch": last_state.completed_epoch,
    }
    print(json.dumps({k: v for k, v in EVAL_MANIFEST.items() if k != "skipped_ids"}, indent=2, default=_json_default))
    print("skipped ids (reported, never silently dropped):", SKIPPED_VAL_IDS)
    save_json("evaluation_manifest.json", EVAL_MANIFEST)

    diag_check("skipped validation entries are exactly the known empty-FOV ones",
               _incomplete_val <= _missing_everywhere,
               f"{len(_incomplete_val)} skipped, {len(_missing_everywhere)} missing everywhere in the split")
    # [5] already pinned the split's size, per-grade counts and sha256; this only checks that the
    # evaluated population plus the reported skips accounts for all of it.
    diag_check("evaluated + skipped = the whole validation split",
               len(EXPECTED_EVALUATED_ENTRIES) + len(SKIPPED_VAL_IDS) == len(val_entries),
               f"{len(EXPECTED_EVALUATED_ENTRIES)} + {len(SKIPPED_VAL_IDS)} of {len(val_entries)}")
    diag_check("[R3] evaluated exactly this population",
               BEST_RESULT["samples"] == len(EXPECTED_EVALUATED_ENTRIES)
               and LAST_RESULT["samples"] == len(EXPECTED_EVALUATED_ENTRIES),
               f"[R3] BEST {BEST_RESULT['samples']}, LAST {LAST_RESULT['samples']}, "
               f"manifest {len(EXPECTED_EVALUATED_ENTRIES)}")
    diag_check("[R3]'s true-label histogram matches the manifest's class counts",
               np.array_equal(np.bincount(BEST_RESULT["y_true"], minlength=corn.NUM_GRADES),
                              np.bincount(EVALUATED_GRADES, minlength=corn.NUM_GRADES)))
    print(f"\nEvery [D3]-[D13] diagnostic uses these {len(EXPECTED_EVALUATED_ENTRIES)} validation images.")
    print("diagnostic output directory:", DIAG_DIR)
else:
    print("RUN_POST_RUN_EVALUATION is False -- no manifest was built.")


In [ ]:
# ==== [D3] POST-RUN: comprehensive classification metrics for the BEST checkpoint (inference only) ====
if RUN_POST_RUN_EVALUATION:
    from evaluation import metrics as evm

    def evaluate_samples(label, weights_path):
        """One deterministic inference pass over the validation pipeline, aligned id-by-id to the
        [D2] manifest. predict_on_batch only: no optimizer, no training op, nothing written.
        Decoding is corn.decode_logits -- the production decoder, not a second implementation."""
        ckpt.load_model_weights_only(joint_model, weights_path)      # model variables only
        logit_batches, grade_batches, loss_sum, batches = [], [], 0.0, 0
        for (stage5, stage6, reliability), grades in POST_RUN_VAL_DS:
            batch_logits = joint_model.predict_on_batch([stage5, stage6, tf.reshape(reliability, (-1, 1))])
            loss_sum += float(jtm.joint_corn_loss(grades, batch_logits)) * int(grades.shape[0])
            logit_batches.append(np.asarray(batch_logits, dtype=np.float32))
            grade_batches.append(grades.numpy().astype(int))
            batches += 1
        logits = np.concatenate(logit_batches)
        y_true = np.concatenate(grade_batches)
        if len(y_true) != len(EVALUATED_IDS) or not np.array_equal(y_true, EVALUATED_GRADES):
            raise RuntimeError(
                f"{label}: the evaluated population does not match the [D2] manifest "
                f"({len(y_true)} samples vs {len(EVALUATED_IDS)}; label sequence "
                f"{'differs' if len(y_true) == len(EVALUATED_IDS) else 'not comparable'}). "
                "Diagnostics would be attributed to the wrong images -- stopping.")
        decoded = corn.decode_logits(logits)                          # p_cond, p_cum, grade, class probs
        return {"label": label, "weights_path": weights_path, "batches": batches, "samples": len(y_true),
                "ids": list(EVALUATED_IDS), "y_true": y_true,
                "y_pred": decoded["predicted_grade"].astype(int), "logits": logits,
                "p_cond": decoded["p_cond"], "p_cum": decoded["p_cum"],
                "class_probabilities": decoded["class_probabilities"], "loss": loss_sum / len(y_true)}


    def classification_metrics(y_true, y_pred):
        """Every metric from the repository's own evaluation.metrics (sklearn-backed); QWK is
        additionally recomputed with the project's training-metric formula as a cross-check."""
        confusion = evm.confusion_matrix(y_true, y_pred, num_classes=corn.NUM_GRADES)
        per_class = {}
        for grade in GRADES:
            tp = int(confusion[grade, grade])
            fn = int(confusion[grade, :].sum() - tp)
            fp = int(confusion[:, grade].sum() - tp)
            precision_g = tp / (tp + fp) if tp + fp else 0.0
            recall_g = tp / (tp + fn) if tp + fn else 0.0
            f1_g = 2 * precision_g * recall_g / (precision_g + recall_g) if precision_g + recall_g else 0.0
            per_class[grade] = {"true_count": int(confusion[grade, :].sum()),
                                "predicted_count": int(confusion[:, grade].sum()), "TP": tp, "FP": fp, "FN": fn,
                                "precision": precision_g, "recall": recall_g, "f1": f1_g, "support": tp + fn}
        return {
            "samples": int(len(y_true)),
            "accuracy": evm.accuracy(y_true, y_pred),
            "qwk": qwk_from_confusion(confusion),                      # the project's training formula
            "qwk_sklearn": evm.quadratic_weighted_kappa(y_true, y_pred),
            "macro_precision": float(evm.precision(y_true, y_pred, average="macro")),
            "weighted_precision": float(evm.precision(y_true, y_pred, average="weighted")),
            "macro_recall": float(evm.recall(y_true, y_pred, average="macro")),
            "weighted_recall": float(evm.recall(y_true, y_pred, average="weighted")),
            "macro_f1": float(evm.f1_score(y_true, y_pred, average="macro")),
            "weighted_f1": float(evm.f1_score(y_true, y_pred, average="weighted")),
            # balanced accuracy is the mean per-class recall, i.e. macro recall
            "balanced_accuracy": float(evm.recall(y_true, y_pred, average="macro")),
            "mae": float(np.mean(np.abs(y_true.astype(float) - y_pred.astype(float)))),
            "per_class": per_class,
            "confusion": confusion.astype(int),
            "true_histogram": np.bincount(y_true, minlength=corn.NUM_GRADES).astype(int),
            "prediction_histogram": np.bincount(y_pred, minlength=corn.NUM_GRADES).astype(int),
        }


    print(f"BEST inference pass over {len(EVALUATED_IDS)} validation images ...")
    BEST_DIAG = evaluate_samples("BEST", BEST_WEIGHTS_LOCAL)
    BEST_METRICS = classification_metrics(BEST_DIAG["y_true"], BEST_DIAG["y_pred"])
    BEST_METRICS["loss"] = BEST_DIAG["loss"]

    print(f"\n=== BEST (checkpoints/best, epoch {best_state.best_epoch + 1}) on "
          f"{BEST_METRICS['samples']} evaluated validation images ===")
    for _name in ("loss", "qwk", "accuracy", "balanced_accuracy", "mae", "macro_f1", "weighted_f1",
                  "macro_precision", "weighted_precision", "macro_recall", "weighted_recall"):
        print(f"  {_name:20} {BEST_METRICS[_name]:.6f}")
    print(f"  {'qwk (sklearn)':20} {BEST_METRICS['qwk_sklearn']:.6f}")
    print(f"\n{'grade':>6}{'true':>8}{'pred':>8}{'TP':>7}{'FP':>7}{'FN':>7}"
          f"{'precision':>12}{'recall':>10}{'F1':>10}{'support':>9}")
    for grade in GRADES:
        row = BEST_METRICS["per_class"][grade]
        print(f"{grade:>6}{row['true_count']:>8}{row['predicted_count']:>8}{row['TP']:>7}{row['FP']:>7}"
              f"{row['FN']:>7}{row['precision']:>12.4f}{row['recall']:>10.4f}{row['f1']:>10.4f}{row['support']:>9}")

    diag_check("[D3] reproduces [R3]'s BEST val_QWK (|diff| < 1e-6)",
               abs(BEST_METRICS["qwk"] - BEST_RESULT["qwk"]) < 1e-6,
               f"{BEST_METRICS['qwk']:.8f} vs {BEST_RESULT['qwk']:.8f}")
    diag_check("[D3] reproduces [R3]'s BEST val_loss (|diff| < 1e-4)",
               abs(BEST_METRICS["loss"] - BEST_RESULT["loss"]) < 1e-4,
               f"{BEST_METRICS['loss']:.8f} vs {BEST_RESULT['loss']:.8f}")
    diag_check("QWK agrees with [R1]'s recorded best_metric (|diff| < 0.01)",
               abs(BEST_METRICS["qwk"] - float(best_state.best_metric)) < 0.01,
               f"recomputed {BEST_METRICS['qwk']:.6f} vs recorded {best_state.best_metric}")
    diag_check("the project's QWK and sklearn's quadratic kappa agree (|diff| < 1e-6)",
               abs(BEST_METRICS["qwk"] - BEST_METRICS["qwk_sklearn"]) < 1e-6)
    diag_check("corn.decode_logits reproduces the metric's own decode used by [R3]",
               np.array_equal(BEST_DIAG["y_pred"], BEST_RESULT["y_pred"]))
    diag_check("confusion matrix sums to the evaluated sample count",
               int(BEST_METRICS["confusion"].sum()) == len(EVALUATED_IDS))
    diag_check("class supports match the manifest's true-label histogram",
               np.array_equal(BEST_METRICS["true_histogram"],
                              np.bincount(EVALUATED_GRADES, minlength=corn.NUM_GRADES)))
    diag_check("prediction histogram matches the decoded predictions",
               np.array_equal(BEST_METRICS["prediction_histogram"],
                              np.bincount(BEST_DIAG["y_pred"], minlength=corn.NUM_GRADES)))
    diag_check("every metric is finite",
               all(math.isfinite(BEST_METRICS[k]) for k in
                   ("loss", "qwk", "accuracy", "balanced_accuracy", "mae", "macro_f1", "weighted_f1",
                    "macro_precision", "weighted_precision", "macro_recall", "weighted_recall")))
    diag_check("logits and probabilities are finite",
               bool(np.isfinite(BEST_DIAG["logits"]).all() and np.isfinite(BEST_DIAG["p_cum"]).all()))
    save_json("best_metrics.json", {k: v for k, v in BEST_METRICS.items()})
    save_csv("best_per_class_metrics.csv",
             [dict(grade=grade, **BEST_METRICS["per_class"][grade]) for grade in GRADES])
else:
    print("RUN_POST_RUN_EVALUATION is False -- no metrics were computed.")


In [ ]:
# ==== [D4] POST-RUN: confusion matrix + ordinal error-distance analysis ====
if RUN_POST_RUN_EVALUATION:
    import matplotlib.pyplot as plt

    from evaluation import visualization as evv

    CONFUSION = BEST_METRICS["confusion"]
    _true, _pred = BEST_DIAG["y_true"], BEST_DIAG["y_pred"]
    ERROR_DISTANCE = np.abs(_true - _pred)
    _total = int(len(_true))
    _errors = int((ERROR_DISTANCE > 0).sum())

    print(f"=== BEST confusion matrix ({_total} evaluated validation images) ===")
    print("        " + "".join(f"{'pred ' + str(g):>9}" for g in GRADES) + f"{'true total':>12}")
    for grade in GRADES:
        print(f"true {grade} " + "".join(f"{v:>9d}" for v in CONFUSION[grade])
              + f"{CONFUSION[grade].sum():>12d}")
    print("pred tot" + "".join(f"{v:>9d}" for v in CONFUSION.sum(axis=0)) + f"{CONFUSION.sum():>12d}")

    print("\n=== ordinal error distance |true - predicted| ===")
    ORDINAL_ERRORS = {}
    print(f"{'distance':>10}{'count':>8}{'% of evaluated':>17}{'% of errors':>14}")
    for distance in range(corn.NUM_GRADES):
        count = int((ERROR_DISTANCE == distance).sum())
        ORDINAL_ERRORS[distance] = {
            "count": count,
            "percent_of_evaluated": 100.0 * count / _total,
            "percent_of_errors": (100.0 * count / _errors) if (_errors and distance > 0) else 0.0,
        }
        print(f"{distance:>10}{count:>8}{ORDINAL_ERRORS[distance]['percent_of_evaluated']:>16.2f}%"
              + (f"{ORDINAL_ERRORS[distance]['percent_of_errors']:>13.2f}%" if distance else f"{'-':>14}"))

    ADJACENT = ORDINAL_ERRORS[1]["count"]
    NON_ADJACENT = int((ERROR_DISTANCE >= 2).sum())
    ERROR_SUMMARY = {
        "evaluated": _total,
        "correct": int((ERROR_DISTANCE == 0).sum()),
        "errors": _errors,
        "adjacent_errors": ADJACENT,
        "non_adjacent_errors": NON_ADJACENT,
        "severe_errors_distance_ge_2": NON_ADJACENT,
        "max_error_distance": int(ERROR_DISTANCE.max()),
        "mean_error_distance": float(ERROR_DISTANCE.mean()),
        "adjacent_share_of_errors": (100.0 * ADJACENT / _errors) if _errors else 0.0,
        "by_distance": ORDINAL_ERRORS,
    }
    print(f"\ncorrect {ERROR_SUMMARY['correct']} ({100.0 * ERROR_SUMMARY['correct'] / _total:.2f}%) | "
          f"errors {_errors} | adjacent {ADJACENT} | non-adjacent (>= 2) {NON_ADJACENT} | "
          f"max distance {ERROR_SUMMARY['max_error_distance']}")
    print(f"adjacent errors are {ERROR_SUMMARY['adjacent_share_of_errors']:.1f}% of all errors -- "
          "the ordering only supports reading this as local versus distant confusion, nothing clinical.")

    print(f"\n=== every off-diagonal pair ===\n{'true':>6}{'predicted':>11}{'count':>8}{'% of true class':>18}")
    OFF_DIAGONAL = []
    for true_grade in GRADES:
        for predicted_grade in GRADES:
            count = int(CONFUSION[true_grade, predicted_grade])
            if true_grade == predicted_grade or count == 0:
                continue
            support = int(CONFUSION[true_grade].sum())
            row = {"true_grade": true_grade, "predicted_grade": predicted_grade, "count": count,
                   "error_distance": abs(true_grade - predicted_grade),
                   "percent_of_true_class": 100.0 * count / support if support else 0.0}
            OFF_DIAGONAL.append(row)
            print(f"{true_grade:>6}{predicted_grade:>11}{count:>8}{row['percent_of_true_class']:>17.2f}%")
    if not OFF_DIAGONAL:
        print("  (none: every evaluated image was classified correctly)")

    _figure = evv.plot_confusion_matrix(CONFUSION, class_names=[f"grade {g}" for g in GRADES],
                                        output_path=posixpath.join(DIAG_PLOTS_DIR, "confusion_matrix.png"),
                                        title=f"BEST confusion matrix (epoch {best_state.best_epoch + 1})")
    DIAG_OUTPUTS.append(posixpath.join(DIAG_PLOTS_DIR, "confusion_matrix.png"))
    _figure_normalized = evv.plot_confusion_matrix(
        CONFUSION, class_names=[f"grade {g}" for g in GRADES], normalize=True,
        output_path=posixpath.join(DIAG_PLOTS_DIR, "confusion_matrix_normalized.png"),
        title="BEST confusion matrix (row-normalized)")
    DIAG_OUTPUTS.append(posixpath.join(DIAG_PLOTS_DIR, "confusion_matrix_normalized.png"))
    plt.show()

    save_csv("confusion_matrix.csv",
             [dict(true_grade=g, **{f"predicted_{p}": int(CONFUSION[g, p]) for p in GRADES}) for g in GRADES])
    save_csv("off_diagonal_pairs.csv", OFF_DIAGONAL or [{"true_grade": "", "predicted_grade": "", "count": 0,
                                                          "error_distance": 0, "percent_of_true_class": 0.0}])
    save_json("ordinal_error_analysis.json", ERROR_SUMMARY)

    diag_check("error distances cover every evaluated sample",
               sum(ORDINAL_ERRORS[d]["count"] for d in range(corn.NUM_GRADES)) == _total)
    diag_check("correct + errors = evaluated", ERROR_SUMMARY["correct"] + _errors == _total)
    diag_check("accuracy from the error distances matches [D3]",
               abs(ERROR_SUMMARY["correct"] / _total - BEST_METRICS["accuracy"]) < 1e-12)
    diag_check("MAE from the error distances matches [D3]",
               abs(ERROR_SUMMARY["mean_error_distance"] - BEST_METRICS["mae"]) < 1e-12)
else:
    print("RUN_POST_RUN_EVALUATION is False -- no confusion or error analysis was produced.")


In [ ]:
# ==== [D5] POST-RUN: per-sample CORN logits, cumulative probabilities and margins ====
if RUN_POST_RUN_EVALUATION:
    # corn.decode_logits already returned everything below; nothing is re-implemented here.
    #   p_cond_k = sigmoid(z_k) = P(y > k | y >= k)
    #   p_cum_k  = prod(i<=k) p_cond_i = P(y > k)          <- the four numbers CORN actually predicts
    #   grade    = count(p_cum > 0.5)
    # p_cum is non-increasing in k (a cumulative product of values in (0,1)), so the finite
    # differences P(y=0) = 1 - p_cum_0, P(y=k) = p_cum_(k-1) - p_cum_k, P(y=4) = p_cum_3 are
    # non-negative and sum to 1: a valid class distribution, asserted below. That is what makes the
    # Brier score and ECE in [D6] defensible rather than invented.
    P_CUM = BEST_DIAG["p_cum"].astype(np.float64)
    CLASS_PROBABILITIES = BEST_DIAG["class_probabilities"].astype(np.float64)
    _true, _pred = BEST_DIAG["y_true"], BEST_DIAG["y_pred"]
    _rows_index = np.arange(len(_true))

    RECONSTRUCTED_DECODE = np.sum(P_CUM > 0.5, axis=-1).astype(int)
    PREDICTED_CLASS_PROBABILITY = CLASS_PROBABILITIES[_rows_index, _pred]
    ARGMAX_CLASS = CLASS_PROBABILITIES.argmax(axis=-1).astype(int)
    # How close the sample is to flipping one of the four ordinal decisions, in probability space.
    THRESHOLD_MARGIN = np.min(np.abs(P_CUM - 0.5), axis=-1)
    _safe = np.clip(CLASS_PROBABILITIES, 1e-12, 1.0)
    CLASS_ENTROPY = -np.sum(CLASS_PROBABILITIES * np.log(_safe), axis=-1)   # nats, from the valid reconstruction
    ERROR_DISTANCE = np.abs(_true - _pred)

    PER_SAMPLE = []
    for index, image_id in enumerate(BEST_DIAG["ids"]):
        row = {"image_id": image_id, "true_grade": int(_true[index]), "predicted_grade": int(_pred[index]),
               "correct": bool(_true[index] == _pred[index]), "error_distance": int(ERROR_DISTANCE[index])}
        row.update({f"logit_{k}": float(BEST_DIAG["logits"][index, k]) for k in range(corn.NUM_THRESHOLDS)})
        row.update({f"p_gt_{k}": float(P_CUM[index, k]) for k in range(corn.NUM_THRESHOLDS)})
        row.update({f"p_cond_{k}": float(BEST_DIAG["p_cond"][index, k]) for k in range(corn.NUM_THRESHOLDS)})
        row.update({f"class_prob_{g}": float(CLASS_PROBABILITIES[index, g]) for g in GRADES})
        row.update({"predicted_class_probability": float(PREDICTED_CLASS_PROBABILITY[index]),
                    "argmax_class": int(ARGMAX_CLASS[index]),
                    "argmax_matches_decode": bool(ARGMAX_CLASS[index] == _pred[index]),
                    "nearest_threshold_margin": float(THRESHOLD_MARGIN[index]),
                    "class_entropy_nats": float(CLASS_ENTROPY[index])})
        PER_SAMPLE.append(row)

    PER_SAMPLE_CSV = save_csv("per_sample_corn_predictions.csv", PER_SAMPLE)
    save_json("per_sample_corn_predictions.json", PER_SAMPLE)
    _argmax_disagreements = int((ARGMAX_CLASS != _pred).sum())

    print(f"per-sample table: {len(PER_SAMPLE)} rows -> {PER_SAMPLE_CSV}")
    print("columns:", ", ".join(PER_SAMPLE[0]))
    print(f"\nmean P(y>k) by threshold: "
          + "  ".join(f"k={k}: {P_CUM[:, k].mean():.4f}" for k in range(corn.NUM_THRESHOLDS)))
    print(f"probability of the decoded grade: mean {PREDICTED_CLASS_PROBABILITY.mean():.4f}, "
          f"median {np.median(PREDICTED_CLASS_PROBABILITY):.4f}, min {PREDICTED_CLASS_PROBABILITY.min():.4f}")
    print(f"nearest threshold margin |P(y>k) - 0.5|: mean {THRESHOLD_MARGIN.mean():.4f}, "
          f"median {np.median(THRESHOLD_MARGIN):.4f}, min {THRESHOLD_MARGIN.min():.6f}")
    print(f"class entropy (nats): mean {CLASS_ENTROPY.mean():.4f}, max {CLASS_ENTROPY.max():.4f} "
          f"(uniform over 5 grades would be {math.log(corn.NUM_GRADES):.4f})")
    print(f"argmax of the reconstructed distribution differs from the CORN decode on "
          f"{_argmax_disagreements} of {len(PER_SAMPLE)} samples "
          f"({100.0 * _argmax_disagreements / len(PER_SAMPLE):.2f}%) -- the decode rule is the "
          "cumulative-threshold count, not an argmax, so the two need not agree.")
    print("\nfirst five rows:")
    for row in PER_SAMPLE[:5]:
        print(f"  {row['image_id']} true {row['true_grade']} pred {row['predicted_grade']} | "
              + " ".join(f"P(y>{k})={row[f'p_gt_{k}']:.4f}" for k in range(corn.NUM_THRESHOLDS))
              + f" | margin {row['nearest_threshold_margin']:.4f}")

    diag_check("the decoded grade recomputed from P(y>k) equals the production decode",
               np.array_equal(RECONSTRUCTED_DECODE, _pred))
    diag_check("P(y>k) is non-increasing in k for every sample",
               bool(np.all(np.diff(P_CUM, axis=-1) <= 1e-12)))
    diag_check("the reconstructed class distribution is non-negative and sums to 1",
               bool((CLASS_PROBABILITIES >= -1e-9).all()
                    and np.allclose(CLASS_PROBABILITIES.sum(axis=-1), 1.0, atol=1e-6)),
               f"max |sum - 1| = {np.abs(CLASS_PROBABILITIES.sum(axis=-1) - 1.0).max():.2e}")
    diag_check("one row per evaluated image, ids unique and in manifest order",
               len(PER_SAMPLE) == len(EVALUATED_IDS)
               and [row["image_id"] for row in PER_SAMPLE] == list(EVALUATED_IDS)
               and len({row["image_id"] for row in PER_SAMPLE}) == len(EVALUATED_IDS))
else:
    print("RUN_POST_RUN_EVALUATION is False -- no per-sample table was produced.")


In [ ]:
# ==== [D6] POST-RUN: confidence, uncertainty and calibration diagnostics (no retraining) ====
if RUN_POST_RUN_EVALUATION:
    import matplotlib.pyplot as plt

    from evaluation import metrics as evm
    from evaluation import visualization as evv

    CALIBRATION_BINS = 10
    _correct = (_true == _pred)

    def _describe(values, label):
        values = np.asarray(values, dtype=float)
        if values.size == 0:
            return {"label": label, "count": 0}
        return {"label": label, "count": int(values.size), "mean": float(values.mean()),
                "median": float(np.median(values)), "min": float(values.min()), "max": float(values.max())}

    def _print_table(rows, title):
        print(f"\n{title}")
        print(f"{'group':>26}{'count':>8}{'mean':>10}{'median':>10}{'min':>10}{'max':>10}")
        for row in rows:
            if row["count"] == 0:
                print(f"{row['label']:>26}{0:>8}{'-':>10}{'-':>10}{'-':>10}{'-':>10}")
                continue
            print(f"{row['label']:>26}{row['count']:>8}{row['mean']:>10.4f}{row['median']:>10.4f}"
                  f"{row['min']:>10.4f}{row['max']:>10.4f}")

    # ---- 1. threshold-level CORN calibration: always well defined for this model -------------------
    print("=== CORN threshold calibration: predicted P(y>k) vs the empirical rate of (true > k) ===")
    print(f"{'threshold':>10}{'mean P(y>k)':>14}{'empirical':>12}{'gap':>10}{'n(true>k)':>12}")
    THRESHOLD_CALIBRATION = []
    for k in range(corn.NUM_THRESHOLDS):
        predicted = float(P_CUM[:, k].mean())
        empirical = float((_true > k).mean())
        THRESHOLD_CALIBRATION.append({"threshold": k, "mean_predicted": predicted,
                                      "empirical_rate": empirical, "gap": predicted - empirical,
                                      "positives": int((_true > k).sum())})
        print(f"{k:>10}{predicted:>14.4f}{empirical:>12.4f}{predicted - empirical:>10.4f}"
              f"{int((_true > k).sum()):>12}")

    THRESHOLD_RELIABILITY = {}
    _edges = np.linspace(0.0, 1.0, CALIBRATION_BINS + 1)
    for k in range(corn.NUM_THRESHOLDS):
        probabilities, outcomes = P_CUM[:, k], (_true > k).astype(float)
        bins = np.clip(np.digitize(probabilities, _edges[1:-1], right=True), 0, CALIBRATION_BINS - 1)
        rows = []
        for b in range(CALIBRATION_BINS):
            mask = bins == b
            rows.append({"bin": b, "lower": float(_edges[b]), "upper": float(_edges[b + 1]),
                         "count": int(mask.sum()),
                         "mean_predicted": float(probabilities[mask].mean()) if mask.any() else None,
                         "empirical_rate": float(outcomes[mask].mean()) if mask.any() else None})
        THRESHOLD_RELIABILITY[k] = rows

    # ---- 2. multiclass calibration from the reconstruction [D5] verified ---------------------------
    _one_hot = np.eye(corn.NUM_GRADES)[_true]
    BRIER_SCORE = float(np.mean(np.sum((CLASS_PROBABILITIES - _one_hot) ** 2, axis=1)))
    ECE_ARGMAX = float(evm.expected_calibration_error(_true, CLASS_PROBABILITIES, n_bins=CALIBRATION_BINS))
    _bins = np.clip(np.digitize(PREDICTED_CLASS_PROBABILITY, _edges[1:-1], right=True), 0, CALIBRATION_BINS - 1)
    RELIABILITY_DECODE, _ece_decode = [], 0.0
    for b in range(CALIBRATION_BINS):
        mask = _bins == b
        row = {"bin": b, "lower": float(_edges[b]), "upper": float(_edges[b + 1]), "count": int(mask.sum()),
               "mean_confidence": float(PREDICTED_CLASS_PROBABILITY[mask].mean()) if mask.any() else None,
               "accuracy": float(_correct[mask].mean()) if mask.any() else None}
        RELIABILITY_DECODE.append(row)
        if mask.any():
            _ece_decode += mask.mean() * abs(row["mean_confidence"] - row["accuracy"])
    ECE_DECODE = float(_ece_decode)
    print(f"\n=== multiclass calibration, from the reconstructed class distribution verified in [D5] ===")
    print(f"Brier score (multiclass, 0 = perfect): {BRIER_SCORE:.6f}")
    print(f"ECE, CORN decode convention (confidence = P(decoded grade), accuracy = decode correct): {ECE_DECODE:.6f}")
    print(f"ECE, standard argmax convention (evaluation.metrics.expected_calibration_error): {ECE_ARGMAX:.6f}")
    print("Both use the finite-difference reconstruction, which [D5] verified is a proper distribution; "
          "the CORN decode is a cumulative-threshold count, so the two conventions differ where argmax "
          "and decode disagree.")

    # ---- 3. confidence and margin structure --------------------------------------------------------
    _print_table([_describe(PREDICTED_CLASS_PROBABILITY, "all"),
                  _describe(PREDICTED_CLASS_PROBABILITY[_correct], "correct"),
                  _describe(PREDICTED_CLASS_PROBABILITY[~_correct], "incorrect")],
                 "confidence = probability of the decoded grade")
    _print_table([_describe(THRESHOLD_MARGIN, "all"), _describe(THRESHOLD_MARGIN[_correct], "correct"),
                  _describe(THRESHOLD_MARGIN[~_correct], "incorrect")],
                 "nearest threshold margin |P(y>k) - 0.5|")
    _print_table([_describe(CLASS_ENTROPY, "all"), _describe(CLASS_ENTROPY[_correct], "correct"),
                  _describe(CLASS_ENTROPY[~_correct], "incorrect")], "class entropy (nats)")
    _print_table([_describe(PREDICTED_CLASS_PROBABILITY[_true == g], f"true grade {g}") for g in GRADES],
                 "confidence by TRUE grade")
    _print_table([_describe(PREDICTED_CLASS_PROBABILITY[_pred == g], f"predicted grade {g}") for g in GRADES],
                 "confidence by PREDICTED grade")
    _print_table([_describe(PREDICTED_CLASS_PROBABILITY[ERROR_DISTANCE == d], f"error distance {d}")
                  for d in range(corn.NUM_GRADES)], "confidence by error distance")
    _print_table([_describe(THRESHOLD_MARGIN[ERROR_DISTANCE == d], f"error distance {d}")
                  for d in range(corn.NUM_GRADES)], "margin by error distance")

    # ---- 4. plots ----------------------------------------------------------------------------------
    _calibration = evm.calibration_curve_data(_true, CLASS_PROBABILITIES, n_bins=CALIBRATION_BINS)
    _path = posixpath.join(DIAG_PLOTS_DIR, "calibration_curve.png")
    evv.plot_calibration_curve(_calibration, output_path=_path,
                               title="BEST reliability (argmax convention)")
    DIAG_OUTPUTS.append(_path)

    _figure, _axes = plt.subplots(2, 2, figsize=(11, 8))
    for k, axis in zip(range(corn.NUM_THRESHOLDS), _axes.ravel()):
        rows = [row for row in THRESHOLD_RELIABILITY[k] if row["count"] > 0]
        axis.plot([0, 1], [0, 1], "k--", lw=1)
        axis.plot([row["mean_predicted"] for row in rows], [row["empirical_rate"] for row in rows], "o-")
        axis.set_title(f"P(y > {k}) vs observed"); axis.set_xlabel("predicted"); axis.set_ylabel("observed")
        axis.grid(alpha=0.3)
    _figure.suptitle("CORN threshold reliability")
    _figure.tight_layout()
    _path = posixpath.join(DIAG_PLOTS_DIR, "threshold_reliability.png")
    _figure.savefig(_path, bbox_inches="tight")
    DIAG_OUTPUTS.append(_path)

    _figure, _axes = plt.subplots(1, 2, figsize=(12, 4))
    _axes[0].hist([PREDICTED_CLASS_PROBABILITY[_correct], PREDICTED_CLASS_PROBABILITY[~_correct]],
                  bins=20, label=["correct", "incorrect"], color=["seagreen", "indianred"])
    _axes[0].set_title("confidence of the decoded grade"); _axes[0].legend(); _axes[0].grid(alpha=0.3)
    _axes[1].hist([THRESHOLD_MARGIN[_correct], THRESHOLD_MARGIN[~_correct]], bins=20,
                  label=["correct", "incorrect"], color=["seagreen", "indianred"])
    _axes[1].set_title("nearest threshold margin"); _axes[1].legend(); _axes[1].grid(alpha=0.3)
    _figure.tight_layout()
    _path = posixpath.join(DIAG_PLOTS_DIR, "confidence_distributions.png")
    _figure.savefig(_path, bbox_inches="tight")
    DIAG_OUTPUTS.append(_path)
    plt.show()

    CALIBRATION_DIAGNOSTICS = {
        "bins": CALIBRATION_BINS, "brier_score": BRIER_SCORE, "ece_decode_convention": ECE_DECODE,
        "ece_argmax_convention": ECE_ARGMAX, "threshold_calibration": THRESHOLD_CALIBRATION,
        "threshold_reliability": {str(k): rows for k, rows in THRESHOLD_RELIABILITY.items()},
        "reliability_decode_convention": RELIABILITY_DECODE,
        "confidence": {"all": _describe(PREDICTED_CLASS_PROBABILITY, "all"),
                       "correct": _describe(PREDICTED_CLASS_PROBABILITY[_correct], "correct"),
                       "incorrect": _describe(PREDICTED_CLASS_PROBABILITY[~_correct], "incorrect"),
                       "by_true_grade": {str(g): _describe(PREDICTED_CLASS_PROBABILITY[_true == g], str(g)) for g in GRADES},
                       "by_predicted_grade": {str(g): _describe(PREDICTED_CLASS_PROBABILITY[_pred == g], str(g)) for g in GRADES},
                       "by_error_distance": {str(d): _describe(PREDICTED_CLASS_PROBABILITY[ERROR_DISTANCE == d], str(d))
                                             for d in range(corn.NUM_GRADES)}},
        "margin": {"all": _describe(THRESHOLD_MARGIN, "all"),
                   "correct": _describe(THRESHOLD_MARGIN[_correct], "correct"),
                   "incorrect": _describe(THRESHOLD_MARGIN[~_correct], "incorrect"),
                   "by_error_distance": {str(d): _describe(THRESHOLD_MARGIN[ERROR_DISTANCE == d], str(d))
                                         for d in range(corn.NUM_GRADES)}},
        "entropy_nats": {"all": _describe(CLASS_ENTROPY, "all"),
                         "correct": _describe(CLASS_ENTROPY[_correct], "correct"),
                         "incorrect": _describe(CLASS_ENTROPY[~_correct], "incorrect")},
    }
    save_json("calibration_diagnostics.json", CALIBRATION_DIAGNOSTICS)
    diag_check("Brier score and both ECE values are finite and in range",
               all(math.isfinite(v) for v in (BRIER_SCORE, ECE_DECODE, ECE_ARGMAX))
               and 0.0 <= ECE_DECODE <= 1.0 and 0.0 <= ECE_ARGMAX <= 1.0 and 0.0 <= BRIER_SCORE <= 2.0)
    diag_check("every evaluated sample falls in exactly one reliability bin",
               sum(row["count"] for row in RELIABILITY_DECODE) == len(_true))
else:
    print("RUN_POST_RUN_EVALUATION is False -- no calibration diagnostics were produced.")


In [ ]:
# ==== [D7] POST-RUN: deterministic misclassification case lists ====
if RUN_POST_RUN_EVALUATION:
    CASE_LIST_MAX = 25          # rows printed and kept per ranked list; the full lists are saved as CSV

    # Every list below is produced by an explicit sort with image_id as the final tie-break, so it is
    # reproducible; nothing is hand-picked and nothing is sampled.
    _by_id = {row["image_id"]: row for row in PER_SAMPLE}
    _incorrect = [row for row in PER_SAMPLE if not row["correct"]]
    _correct_rows = [row for row in PER_SAMPLE if row["correct"]]

    CASE_SORT_RULES = {
        "all_misclassified": "error_distance desc, predicted_class_probability desc, image_id asc",
        "largest_ordinal_errors": "error_distance desc, nearest_threshold_margin desc, image_id asc",
        "adjacent_errors": "predicted_class_probability desc, image_id asc (error_distance == 1)",
        "confusion_pairs": "true_grade asc, predicted_grade asc, image_id asc (off-diagonal only)",
        "lowest_confidence_correct": "predicted_class_probability asc, image_id asc",
        "highest_confidence_incorrect": "predicted_class_probability desc, image_id asc",
        "widest_margin_incorrect": "nearest_threshold_margin desc, image_id asc",
        "most_ambiguous": "nearest_threshold_margin asc, image_id asc",
    }
    CASE_LISTS = {
        "all_misclassified": sorted(_incorrect, key=lambda r: (-r["error_distance"],
                                                               -r["predicted_class_probability"], r["image_id"])),
        "largest_ordinal_errors": sorted(_incorrect, key=lambda r: (-r["error_distance"],
                                                                    -r["nearest_threshold_margin"], r["image_id"])),
        "adjacent_errors": sorted([r for r in _incorrect if r["error_distance"] == 1],
                                  key=lambda r: (-r["predicted_class_probability"], r["image_id"])),
        "confusion_pairs": sorted(_incorrect, key=lambda r: (r["true_grade"], r["predicted_grade"], r["image_id"])),
        "lowest_confidence_correct": sorted(_correct_rows, key=lambda r: (r["predicted_class_probability"], r["image_id"])),
        "highest_confidence_incorrect": sorted(_incorrect, key=lambda r: (-r["predicted_class_probability"], r["image_id"])),
        "widest_margin_incorrect": sorted(_incorrect, key=lambda r: (-r["nearest_threshold_margin"], r["image_id"])),
        "most_ambiguous": sorted(PER_SAMPLE, key=lambda r: (r["nearest_threshold_margin"], r["image_id"])),
    }

    _columns = (["image_id", "true_grade", "predicted_grade", "error_distance", "correct"]
                + [f"p_gt_{k}" for k in range(corn.NUM_THRESHOLDS)]
                + [f"class_prob_{g}" for g in GRADES]
                + ["predicted_class_probability", "nearest_threshold_margin", "class_entropy_nats",
                   "argmax_matches_decode"])

    print(f"{len(_incorrect)} misclassified of {len(PER_SAMPLE)} evaluated "
          f"({100.0 * len(_incorrect) / len(PER_SAMPLE):.2f}%)")
    for name, rows in CASE_LISTS.items():
        path = save_csv(f"cases_{name}.csv", [{column: row[column] for column in _columns} for row in rows]
                        or [{column: "" for column in _columns}], fieldnames=_columns)
        print(f"\n--- {name} ({len(rows)} rows; sort: {CASE_SORT_RULES[name]}) -> {posixpath.basename(path)}")
        for row in rows[:min(CASE_LIST_MAX, 8)]:
            print(f"    {row['image_id']} true {row['true_grade']} -> pred {row['predicted_grade']} "
                  f"(d={row['error_distance']}) conf {row['predicted_class_probability']:.4f} "
                  f"margin {row['nearest_threshold_margin']:.4f} "
                  + " ".join(f"P(y>{k})={row[f'p_gt_{k}']:.3f}" for k in range(corn.NUM_THRESHOLDS)))
        if not rows:
            print("    (none)")

    # Per confusion pair, so no single grade is privileged.
    PAIR_CASES = {}
    for row in CASE_LISTS["confusion_pairs"]:
        PAIR_CASES.setdefault((row["true_grade"], row["predicted_grade"]), []).append(row["image_id"])
    print("\n--- per confusion pair ---")
    for (true_grade, predicted_grade), ids in sorted(PAIR_CASES.items()):
        print(f"    true {true_grade} -> pred {predicted_grade}: {len(ids)} image(s); first: {ids[:5]}")
    save_json("case_lists_index.json",
              {"sort_rules": CASE_SORT_RULES,
               "counts": {name: len(rows) for name, rows in CASE_LISTS.items()},
               "confusion_pairs": {f"{t}->{p}": ids for (t, p), ids in sorted(PAIR_CASES.items())},
               "case_list_max_printed": CASE_LIST_MAX})

    diag_check("case lists cover every misclassified sample exactly once",
               len(CASE_LISTS["all_misclassified"]) == int((~_correct).sum())
               and len({row["image_id"] for row in CASE_LISTS["all_misclassified"]}) == int((~_correct).sum()))
    diag_check("adjacent + non-adjacent case counts match the [D4] error analysis",
               len(CASE_LISTS["adjacent_errors"]) == ERROR_SUMMARY["adjacent_errors"]
               and len(_incorrect) - len(CASE_LISTS["adjacent_errors"]) == ERROR_SUMMARY["non_adjacent_errors"])
    diag_check("confusion-pair case counts match the confusion matrix",
               all(len(ids) == int(CONFUSION[true_grade, predicted_grade])
                   for (true_grade, predicted_grade), ids in PAIR_CASES.items()))
else:
    print("RUN_POST_RUN_EVALUATION is False -- no case lists were produced.")


In [ ]:
# ==== [D8] POST-RUN: validation image inspection across the whole error structure ====
if RUN_POST_RUN_EVALUATION:
    import matplotlib.pyplot as plt

    IMAGES_PER_CATEGORY = 4     # per panel; raise for more, every selection stays deterministic
    IMAGE_SELECTION_SEED = None  # no sampling is used anywhere in this cell: pure deterministic sorts

    # The cached canonical RGB is exactly what the model consumed (channels 0-2 of stage5_input):
    # Stage 02-processed, resized to 512x512, [0,1] float32. Read-only; the raw dataset is untouched.
    def load_canonical_rgb(image_id):
        local = jtd._canonical_rgb_cache_path(image_id, LOCAL_CACHE_DIR, jtd.STAGE5_IMAGE_SIZE)
        persistent = jtd._canonical_rgb_cache_path(image_id, config.LOCAL_FEATURE_RESULTS_DIR,
                                                   jtd.STAGE5_IMAGE_SIZE)
        for path in (local, persistent):
            if os.path.exists(path):
                return np.clip(np.asarray(np.load(path), dtype=np.float32), 0.0, 1.0), path
        return None, None

    def show_panel(title, rows, filename):
        """One figure of up to IMAGES_PER_CATEGORY images; reports any image it cannot load."""
        rows = rows[:IMAGES_PER_CATEGORY]
        if not rows:
            print(f"  {title}: no sample in this category")
            return
        figure, axes = plt.subplots(1, len(rows), figsize=(3.6 * len(rows), 4.4), squeeze=False)
        missing = []
        for axis, row in zip(axes[0], rows):
            image, source = load_canonical_rgb(row["image_id"])
            if image is None:
                missing.append(row["image_id"])
                axis.text(0.5, 0.5, "canonical RGB\nnot cached", ha="center", va="center")
                axis.set_facecolor("0.92")
            else:
                axis.imshow(image)
            axis.set_xticks([]); axis.set_yticks([])
            axis.set_title(f"{row['image_id']}\ntrue {row['true_grade']} -> pred {row['predicted_grade']}"
                           f"  (d={row['error_distance']})\nconf {row['predicted_class_probability']:.3f}"
                           f"  margin {row['nearest_threshold_margin']:.3f}\n"
                           + " ".join(f"{row[f'p_gt_{k}']:.2f}" for k in range(corn.NUM_THRESHOLDS)),
                           fontsize=8, color="seagreen" if row["correct"] else "indianred")
        figure.suptitle(title + "   (P(y>0..3) under each image)", fontsize=10)
        figure.tight_layout()
        path = posixpath.join(DIAG_IMAGE_DIR, filename)
        figure.savefig(path, bbox_inches="tight", dpi=110)
        DIAG_OUTPUTS.append(path)
        plt.show()
        if missing:
            print(f"  {title}: canonical RGB unavailable for {missing} -- reported, not substituted")
        return missing

    DIAG_IMAGE_DIR = posixpath.join(DIAG_PLOTS_DIR, "images")
    os.makedirs(DIAG_IMAGE_DIR, exist_ok=True)
    IMAGE_PANELS, UNAVAILABLE_IMAGES = {}, []
    _selection_rules = {
        "A_correct_by_true_grade": "correct samples of the grade, predicted_class_probability desc, image_id asc",
        "B_misclassified_by_true_grade": "misclassified samples of the grade, error_distance desc, "
                                          "predicted_class_probability desc, image_id asc",
        "C_confusion_pair": "samples of the pair, predicted_class_probability desc, image_id asc",
        "D_largest_ordinal_errors": CASE_SORT_RULES["largest_ordinal_errors"],
        "E_highest_confidence_incorrect": CASE_SORT_RULES["highest_confidence_incorrect"],
        "F_lowest_confidence_correct": CASE_SORT_RULES["lowest_confidence_correct"],
    }

    print("=== A. correctly classified, one panel per true grade ===")
    for grade in GRADES:
        rows = sorted([r for r in PER_SAMPLE if r["correct"] and r["true_grade"] == grade],
                      key=lambda r: (-r["predicted_class_probability"], r["image_id"]))
        IMAGE_PANELS[f"A_correct_grade_{grade}"] = [r["image_id"] for r in rows[:IMAGES_PER_CATEGORY]]
        UNAVAILABLE_IMAGES += show_panel(f"A. correct, true grade {grade}", rows,
                                         f"A_correct_grade_{grade}.png") or []

    print("\n=== B. misclassified, one panel per true grade ===")
    for grade in GRADES:
        rows = sorted([r for r in PER_SAMPLE if not r["correct"] and r["true_grade"] == grade],
                      key=lambda r: (-r["error_distance"], -r["predicted_class_probability"], r["image_id"]))
        IMAGE_PANELS[f"B_misclassified_grade_{grade}"] = [r["image_id"] for r in rows[:IMAGES_PER_CATEGORY]]
        UNAVAILABLE_IMAGES += show_panel(f"B. misclassified, true grade {grade}", rows,
                                         f"B_misclassified_grade_{grade}.png") or []

    print("\n=== C. every observed off-diagonal confusion pair ===")
    for (true_grade, predicted_grade) in sorted(PAIR_CASES):
        rows = sorted([r for r in PER_SAMPLE
                       if r["true_grade"] == true_grade and r["predicted_grade"] == predicted_grade],
                      key=lambda r: (-r["predicted_class_probability"], r["image_id"]))
        key = f"C_pair_true{true_grade}_pred{predicted_grade}"
        IMAGE_PANELS[key] = [r["image_id"] for r in rows[:IMAGES_PER_CATEGORY]]
        UNAVAILABLE_IMAGES += show_panel(
            f"C. true {true_grade} predicted {predicted_grade} ({len(rows)} image(s))", rows, key + ".png") or []

    print("\n=== D/E/F. error-structure extremes ===")
    for key, rows in (("D_largest_ordinal_errors", CASE_LISTS["largest_ordinal_errors"]),
                      ("E_highest_confidence_incorrect", CASE_LISTS["highest_confidence_incorrect"]),
                      ("F_lowest_confidence_correct", CASE_LISTS["lowest_confidence_correct"])):
        IMAGE_PANELS[key] = [r["image_id"] for r in rows[:IMAGES_PER_CATEGORY]]
        UNAVAILABLE_IMAGES += show_panel(key.replace("_", " "), rows, key + ".png") or []

    save_json("image_panels.json", {"images_per_category": IMAGES_PER_CATEGORY,
                                    "selection_seed": IMAGE_SELECTION_SEED,
                                    "selection_rules": _selection_rules,
                                    "panels": IMAGE_PANELS,
                                    "unavailable_images": sorted(set(UNAVAILABLE_IMAGES)),
                                    "image_source": "cached canonical RGB (Stage 02 processed, 512x512, [0,1])",
                                    "directory": DIAG_IMAGE_DIR})
    print(f"\npanels saved under {DIAG_IMAGE_DIR}")
    diag_check("every off-diagonal confusion pair has an image panel",
               all(f"C_pair_true{t}_pred{p}" in IMAGE_PANELS for (t, p) in PAIR_CASES),
               f"{len(PAIR_CASES)} pair(s)")
    diag_check("every selected image was displayed from the cache (none substituted)",
               not UNAVAILABLE_IMAGES, sorted(set(UNAVAILABLE_IMAGES)))
    diag_check("no skipped empty-FOV image was selected (they are not evaluated at all)",
               not (set(SKIPPED_VAL_IDS) & {image_id for ids in IMAGE_PANELS.values() for image_id in ids}))
else:
    print("RUN_POST_RUN_EVALUATION is False -- no images were inspected.")


In [ ]:
# ==== [D9] POST-RUN: BEST vs LAST on the identical validation population ====
if RUN_POST_RUN_EVALUATION:
    # BEST is the delivery model (global maximum val_QWK). LAST is only the trajectory a resume
    # continues from. This cell compares them; it never promotes LAST and never touches either file.
    if not (LAST_RESULT["samples"] == len(EVALUATED_IDS)
            and np.array_equal(LAST_RESULT["y_true"], EVALUATED_GRADES)):
        raise RuntimeError("[R3]'s LAST evaluation does not match the [D2] manifest population.")

    LAST_METRICS = classification_metrics(LAST_RESULT["y_true"], LAST_RESULT["y_pred"])
    LAST_METRICS["loss"] = LAST_RESULT["loss"]
    _last_distance = np.abs(LAST_RESULT["y_true"] - LAST_RESULT["y_pred"])
    LAST_ERROR_SUMMARY = {
        "evaluated": int(len(_last_distance)), "correct": int((_last_distance == 0).sum()),
        "errors": int((_last_distance > 0).sum()), "adjacent_errors": int((_last_distance == 1).sum()),
        "non_adjacent_errors": int((_last_distance >= 2).sum()),
        "max_error_distance": int(_last_distance.max()), "mean_error_distance": float(_last_distance.mean()),
        "by_distance": {d: int((_last_distance == d).sum()) for d in range(corn.NUM_GRADES)},
    }

    _scalar_metrics = ("loss", "qwk", "accuracy", "balanced_accuracy", "mae", "macro_f1", "weighted_f1",
                       "macro_precision", "weighted_precision", "macro_recall", "weighted_recall")
    print(f"BEST = checkpoints/best, epoch {best_state.best_epoch + 1} (delivery model)")
    print(f"LAST = checkpoints/{eval_report['resumable']}, epoch {last_state.completed_epoch} "
          "(trajectory / resume checkpoint)")
    print(f"both evaluated on the same {len(EVALUATED_IDS)} validation images\n")
    print(f"{'metric':>22}{'BEST':>12}{'LAST':>12}{'BEST - LAST':>14}")
    COMPARISON_ROWS = []
    for name in _scalar_metrics:
        best_value, last_value = float(BEST_METRICS[name]), float(LAST_METRICS[name])
        COMPARISON_ROWS.append({"metric": name, "best": best_value, "last": last_value,
                                "difference": best_value - last_value})
        print(f"{name:>22}{best_value:>12.6f}{last_value:>12.6f}{best_value - last_value:>14.6f}")
    for grade in GRADES:
        for metric in ("recall", "f1", "precision"):
            best_value = float(BEST_METRICS["per_class"][grade][metric])
            last_value = float(LAST_METRICS["per_class"][grade][metric])
            COMPARISON_ROWS.append({"metric": f"grade_{grade}_{metric}", "best": best_value,
                                    "last": last_value, "difference": best_value - last_value})
    print(f"\n{'per-class recall':>22}" + "".join(f"{'grade ' + str(g):>12}" for g in GRADES))
    for label, metrics in (("BEST", BEST_METRICS), ("LAST", LAST_METRICS)):
        print(f"{label:>22}" + "".join(f"{metrics['per_class'][g]['recall']:>12.4f}" for g in GRADES))
    print(f"\n{'per-class F1':>22}" + "".join(f"{'grade ' + str(g):>12}" for g in GRADES))
    for label, metrics in (("BEST", BEST_METRICS), ("LAST", LAST_METRICS)):
        print(f"{label:>22}" + "".join(f"{metrics['per_class'][g]['f1']:>12.4f}" for g in GRADES))
    print(f"\n{'prediction histogram':>22}" + "".join(f"{'grade ' + str(g):>12}" for g in GRADES))
    for label, metrics in (("BEST", BEST_METRICS), ("LAST", LAST_METRICS)):
        print(f"{label:>22}" + "".join(f"{int(v):>12}" for v in metrics["prediction_histogram"]))
    print(f"{'true histogram':>22}" + "".join(f"{int(v):>12}" for v in BEST_METRICS["true_histogram"]))
    print(f"\n{'error distance':>22}" + "".join(f"{d:>12}" for d in range(corn.NUM_GRADES)))
    print(f"{'BEST':>22}" + "".join(f"{ERROR_SUMMARY['by_distance'][d]['count']:>12}"
                                    for d in range(corn.NUM_GRADES)))
    print(f"{'LAST':>22}" + "".join(f"{LAST_ERROR_SUMMARY['by_distance'][d]:>12}"
                                    for d in range(corn.NUM_GRADES)))
    print("\nLAST confusion matrix (rows = true grade, columns = predicted grade):")
    print("        " + "".join(f"{'pred ' + str(g):>9}" for g in GRADES))
    for grade in GRADES:
        print(f"true {grade} " + "".join(f"{v:>9d}" for v in LAST_METRICS["confusion"][grade]))

    BEST_VS_LAST = {
        "population": {"evaluated": len(EVALUATED_IDS), "skipped": len(SKIPPED_VAL_IDS)},
        "best": {"role": "delivery model (global maximum val_QWK)", "checkpoint": BEST_DIR,
                 "epoch": best_state.best_epoch + 1, "metrics": BEST_METRICS,
                 "error_summary": ERROR_SUMMARY},
        "last": {"role": "trajectory / resume checkpoint", "checkpoint": LAST_DIR,
                 "epoch": last_state.completed_epoch, "metrics": LAST_METRICS,
                 "error_summary": LAST_ERROR_SUMMARY},
        "comparison": COMPARISON_ROWS,
    }
    save_json("best_vs_last.json", BEST_VS_LAST)
    save_csv("best_vs_last.csv", COMPARISON_ROWS, fieldnames=["metric", "best", "last", "difference"])

    diag_check("LAST was evaluated on exactly the manifest population",
               LAST_METRICS["samples"] == len(EVALUATED_IDS))
    # [R3] reports the float32 Keras metric; this recomputes it in float64 from the confusion matrix,
    # so they agree to about 1e-8 rather than exactly.
    diag_check("[D9] reproduces [R3]'s LAST val_QWK (|diff| < 1e-6)",
               abs(LAST_METRICS["qwk"] - LAST_RESULT["qwk"]) < 1e-6,
               f"{LAST_METRICS['qwk']:.8f} vs {LAST_RESULT['qwk']:.8f}")
    diag_check("BEST's val_QWK is at least LAST's (BEST is selected on this metric)",
               BEST_METRICS["qwk"] >= LAST_METRICS["qwk"] - 1e-9,
               f"BEST {BEST_METRICS['qwk']:.6f} vs LAST {LAST_METRICS['qwk']:.6f}")
    print("\nBEST remains the delivery model; LAST is kept only as the resume trajectory.")
else:
    print("RUN_POST_RUN_EVALUATION is False -- no BEST/LAST comparison was produced.")


In [ ]:
# ==== [D10] POST-RUN: final-model artifact verification (read-only; runs before any archival) ====
if RUN_POST_RUN_EVALUATION:
    BEST_WEIGHTS_DRIVE = posixpath.join(BEST_DIR, ckpt.MODEL_WEIGHTS_FILENAME)
    _manifest = read_json(posixpath.join(BEST_DIR, ckpt.MANIFEST_FILENAME))
    _recorded_logs = (best_state.extra or {}).get("epoch_logs", {})
    _validation = ckpt.validate_generation(BEST_DIR, required=ckpt.BEST_REQUIRED_FILES)

    # The decoder must be able to produce every grade: a property of corn.decode_logits itself,
    # checked on constructed logits rather than hoping the validation set happens to show all five.
    _reachable = corn.decode_logits(np.array([[-9.0, -9.0, -9.0, -9.0], [9.0, -9.0, -9.0, -9.0],
                                              [9.0, 9.0, -9.0, -9.0], [9.0, 9.0, 9.0, -9.0],
                                              [9.0, 9.0, 9.0, 9.0]], dtype=np.float32))["predicted_grade"]
    _batch_inputs, _batch_grades = next(iter(POST_RUN_VAL_DS))
    _probe = joint_model.predict_on_batch([_batch_inputs[0], _batch_inputs[1],
                                           tf.reshape(_batch_inputs[2], (-1, 1))])
    _parameters = sum(int(np.prod(v.shape)) for v in joint_model.trainable_variables)
    _weights_sha = ckpt.sha256_file(BEST_WEIGHTS_LOCAL)      # the verified local copy made by [R2]

    ARTIFACT_VERIFICATION = {
        "best_checkpoint_dir": BEST_DIR,
        "best_weights_drive": BEST_WEIGHTS_DRIVE,
        "best_weights_local_copy": BEST_WEIGHTS_LOCAL,
        "sha256": _weights_sha,
        "manifest_sha256": _manifest["files"][ckpt.MODEL_WEIGHTS_FILENAME]["sha256"],
        "best_epoch": best_state.best_epoch + 1,
        "best_val_qwk_recorded": best_state.best_metric,
        "best_val_qwk_recomputed": BEST_METRICS["qwk"],
        "best_val_loss_recorded": _recorded_logs.get("val_loss"),
        "best_val_loss_recomputed": BEST_METRICS["loss"],
        "config_hash_manifest": _manifest.get("config_hash"),
        "config_hash_metadata": eval_metadata.get("config_hash"),
        "git_commit_hash": _manifest.get("git_commit_hash"),
        "dataset_version": _manifest.get("dataset_version"),
        "precision_policy": _manifest.get("precision_policy"),
        "optimizer_type": _manifest.get("optimizer_type"),
        "optimizer_wrapper": _manifest.get("optimizer_wrapper"),
        "trainable_parameters": _parameters,
        "output_shape": list(np.asarray(_probe).shape),
        "decodable_grades": sorted(int(g) for g in set(_reachable.tolist())),
    }

    diag_check("BEST checkpoint validates (READY, manifest, sizes, SHA-256)", _validation.ok,
               _validation.reason or "")
    diag_check("BEST weights file exists on Drive", os.path.isfile(BEST_WEIGHTS_DRIVE))
    diag_check("BEST weights SHA-256 matches its manifest",
               _weights_sha == ARTIFACT_VERIFICATION["manifest_sha256"], _weights_sha[:16] + "...")
    diag_check("weights load into the reconstructed production architecture",
               bool(getattr(joint_model, "built", True)) and _parameters == EXPECTED_TRAINABLE_PARAMETERS)
    diag_check(f"trainable parameter count is the production {EXPECTED_TRAINABLE_PARAMETERS:,}",
               _parameters == EXPECTED_TRAINABLE_PARAMETERS, f"{_parameters:,}")
    diag_check(f"model output shape is (batch, {corn.NUM_THRESHOLDS}) CORN logits",
               tuple(np.asarray(_probe).shape[1:]) == (corn.NUM_THRESHOLDS,),
               str(np.asarray(_probe).shape))
    diag_check("all five grades are reachable through the production decoder",
               ARTIFACT_VERIFICATION["decodable_grades"] == GRADES,
               str(ARTIFACT_VERIFICATION["decodable_grades"]))
    diag_check("inference output is finite", bool(np.isfinite(np.asarray(_probe)).all()))
    diag_check("recomputed val_QWK matches the recorded BEST metric (|diff| < 0.01)",
               abs(BEST_METRICS["qwk"] - float(best_state.best_metric)) < 0.01,
               f"{BEST_METRICS['qwk']:.6f} vs {best_state.best_metric}")
    diag_check("recomputed val_loss matches the recorded value (|diff| < 1e-3)",
               _recorded_logs.get("val_loss") is not None
               and abs(BEST_METRICS["loss"] - float(_recorded_logs["val_loss"])) < 1e-3,
               f"{BEST_METRICS['loss']:.6f} vs {_recorded_logs.get('val_loss')}")
    diag_check("config hash matches the manifest, metadata.json and this notebook",
               _manifest.get("config_hash") == eval_metadata.get("config_hash") == EXPECTED_CONFIG_HASH)
    diag_check("git commit metadata is preserved in the checkpoint",
               bool(_manifest.get("git_commit_hash")), str(_manifest.get("git_commit_hash"))[:12])
    diag_check("optimizer and precision metadata are preserved",
               _manifest.get("optimizer_type") == EXPECTED_OPTIMIZER_TYPE
               and _manifest.get("precision_policy") == EXPECTED_PRECISION_POLICY)
    diag_check("dataset split metadata is preserved and still verifies",
               SPLIT_VERIFIED is True and EVAL_MANIFEST["split_sha256"] == EXPECTED_SPLIT_SHA256)
    diag_check("the experiment's checkpoints/ is byte-for-byte unchanged by the diagnostics",
               tree_fingerprint(EVAL_CHECKPOINT_DIR) == CHECKPOINTS_FINGERPRINT_BEFORE)

    ARTIFACT_VERIFICATION["checks"] = {name: {"ok": ok, "detail": detail}
                                       for name, (ok, detail) in DIAG_CHECKS.items()}
    D10_PASSED = all(ok for ok, _ in DIAG_CHECKS.values())
    ARTIFACT_VERIFICATION["passed"] = D10_PASSED
    save_json("artifact_verification.json", ARTIFACT_VERIFICATION)
    print(f"\nARTIFACT VERIFICATION: {'PASS' if D10_PASSED else 'FAIL'} "
          f"({sum(1 for ok, _ in DIAG_CHECKS.values() if ok)}/{len(DIAG_CHECKS)} checks across [D2]-[D10])")
    if not D10_PASSED:
        print("Failing checks:", [name for name, (ok, _) in DIAG_CHECKS.items() if not ok])
        print("[D11] will refuse to archive until these pass.")
else:
    print("RUN_POST_RUN_EVALUATION is False -- no artifact verification was run.")


In [ ]:
# ==== [D11] POST-RUN: final model archival (the only cell that writes outside evaluation/) ====
if RUN_POST_RUN_EVALUATION:
    import shutil

    FINAL_MODEL_ROOT = colab_config.FINAL_CLASSIFICATION_EXPORTED_DIR
    FINAL_MODEL_DIR = posixpath.join(FINAL_MODEL_ROOT, f"{EVAL_MANIFEST['experiment_id']}_BEST")
    ARCHIVE_STAGING = "/content/final_model_staging"      # LOCAL: built and verified before any Drive write
    ARCHIVE_READY_FILENAME = "ARCHIVE_READY"

    if not D10_PASSED:
        raise RuntimeError("[D10] did not pass -- refusing to archive. Nothing was written.")
    if os.path.exists(FINAL_MODEL_DIR):
        raise RuntimeError(
            f"{FINAL_MODEL_DIR} already exists. This cell never overwrites or deletes an archive: "
            "inspect the existing directory and, if a new archive is really wanted, choose a new "
            "destination by editing FINAL_MODEL_DIR above. Nothing was written.")

    _components = sorted({layer.name for layer in joint_model.layers})
    PROVENANCE = {
        "created": datetime.datetime.now().isoformat(timespec="seconds"),
        "role": "FINAL DELIVERY MODEL (BEST: the globally best val_QWK epoch of the experiment)",
        "model_file": ckpt.MODEL_WEIGHTS_FILENAME,
        "model_sha256": ARTIFACT_VERIFICATION["sha256"],
        "experiment_id": EVAL_MANIFEST["experiment_id"],
        "experiment_dir": EVAL_EXPERIMENT_DIR,
        "source_checkpoint": BEST_DIR,
        "source_generation": EVAL_MANIFEST["checkpoint_source_generation"],
        "last_checkpoint_not_archived": LAST_DIR,      # pointer only: LAST is never the delivery model
        "best_epoch": best_state.best_epoch + 1,
        "best_val_qwk_recorded": best_state.best_metric,
        "best_val_qwk_recomputed": BEST_METRICS["qwk"],
        "best_val_loss_recorded": ARTIFACT_VERIFICATION["best_val_loss_recorded"],
        "best_val_loss_recomputed": BEST_METRICS["loss"],
        "evaluated_validation_images": len(EVALUATED_IDS),
        "skipped_validation_images": SKIPPED_VAL_IDS,
        "config_hash": EXPECTED_CONFIG_HASH,
        "git_commit": ARTIFACT_VERIFICATION["git_commit_hash"],
        "notebook_commit_at_archival": SESSION_COMMIT,
        "trainable_parameters": ARTIFACT_VERIFICATION["trainable_parameters"],
        "all_weights_including_batchnorm_statistics": int(joint_model.count_params()),
        "split_manifest": EVAL_MANIFEST["split_manifest"],
        "split_sha256": EXPECTED_SPLIT_SHA256,
        "split_seed": EVAL_MANIFEST["split_seed"],
        "split_counts": {"train": len(train_entries), "validation": len(val_entries)},
        "dataset_version": EVAL_MANIFEST["dataset_version"],
        "preprocessing": {"image_size": list(jtd.STAGE5_IMAGE_SIZE),
                          "stage5_input_shape": list(jtm.STAGE5_INPUT_SHAPE),
                          "stage6_input_shape": list(jtm.STAGE6_INPUT_SHAPE),
                          "augmentation_at_evaluation": False, "shuffle_at_evaluation": False},
        "corn": {"num_grades": corn.NUM_GRADES, "num_thresholds": corn.NUM_THRESHOLDS,
                 "d_model": corn.D_MODEL, "loss": "corn.corn_loss (via joint_corn_loss)",
                 "decode": "sigmoid -> cumulative product -> count(P(y>k) > 0.5)",
                 "grade_names": list(corn.GRADE_NAMES)},
        "architecture": {"identifier": joint_model.name, "components": _components,
                         "stage05_local_features": "trainable (AdaptiveBranchFusion included)",
                         "stage06_global_features": "trainable (dual-scale Swin)",
                         "stage07_adaptive_cross_attention": "trainable",
                         "racaf": "trainable (w_g, b_g, W_r, b_r)",
                         "stage03_vessel_stage04_lesion": "frozen, outside this graph (cache layer)"},
        "precision_policy": ARTIFACT_VERIFICATION["precision_policy"],
        "optimizer": {"type": ARTIFACT_VERIFICATION["optimizer_type"],
                      "wrapper": ARTIFACT_VERIFICATION["optimizer_wrapper"],
                      "initial_learning_rate": LEARNING_RATE,
                      "learning_rate_at_last_checkpoint": last_state.learning_rate},
        "training": {"epochs_completed": last_state.completed_epoch, "epoch_cap": EPOCHS,
                     "batch_size": BATCH_SIZE, "monitor": MONITOR_METRIC, "mode": MONITOR_MODE,
                     "stop_reason": EVAL_FINISHED,
                     "early_stopping": {"patience": EARLY_STOPPING_PATIENCE, "restore_best_weights": True},
                     "reduce_lr_on_plateau": {"patience": REDUCE_LR_PATIENCE, "factor": REDUCE_LR_FACTOR,
                                              "min_lr": MIN_LR}},
        "evaluation_directory": DIAG_DIR,
        "disclaimer": "Metrics come from this project's single APTOS 2019 validation split. Not an "
                      "external test set; no clinical or generalization claim is made.",
    }

    # Build the archive LOCALLY first, verify it, then copy to Drive and re-verify there; the READY
    # marker is written last, so an interrupted copy leaves an obviously incomplete archive.
    shutil.rmtree(ARCHIVE_STAGING, ignore_errors=True)      # local scratch only
    os.makedirs(ARCHIVE_STAGING)
    shutil.copyfile(BEST_WEIGHTS_LOCAL, posixpath.join(ARCHIVE_STAGING, ckpt.MODEL_WEIGHTS_FILENAME))
    for source, name in ((posixpath.join(BEST_DIR, ckpt.STATE_FILENAME), "checkpoint_state.json"),
                         (posixpath.join(BEST_DIR, ckpt.MANIFEST_FILENAME), "checkpoint_manifest.json"),
                         (posixpath.join(EVAL_EXPERIMENT_DIR, "metadata.json"), "experiment_metadata.json"),
                         (EVAL_MANIFEST["split_manifest"], "aptos2019_train_val_split.csv")):
        shutil.copyfile(source, posixpath.join(ARCHIVE_STAGING, name))
    with open(posixpath.join(ARCHIVE_STAGING, "provenance.json"), "w") as handle:
        json.dump(PROVENANCE, handle, indent=2, default=_json_default)
    with open(posixpath.join(ARCHIVE_STAGING, "evaluation_manifest.json"), "w") as handle:
        json.dump(EVAL_MANIFEST, handle, indent=2, default=_json_default)

    _staged = sorted(os.listdir(ARCHIVE_STAGING))
    _checksums = {name: ckpt.sha256_file(posixpath.join(ARCHIVE_STAGING, name)) for name in _staged}
    if _checksums[ckpt.MODEL_WEIGHTS_FILENAME] != ARTIFACT_VERIFICATION["sha256"]:
        raise RuntimeError("staged weights do not match the verified BEST SHA-256; nothing was copied.")

    os.makedirs(FINAL_MODEL_DIR)                            # exist_ok=False: never overwrites
    for name in _staged:
        shutil.copyfile(posixpath.join(ARCHIVE_STAGING, name), posixpath.join(FINAL_MODEL_DIR, name))
    _destination_checksums = {name: ckpt.sha256_file(posixpath.join(FINAL_MODEL_DIR, name)) for name in _staged}
    if _destination_checksums != _checksums:
        raise RuntimeError(f"the archive did not survive the copy to {FINAL_MODEL_DIR}; it is left "
                           "without a READY marker. Nothing else was changed.")
    with open(posixpath.join(FINAL_MODEL_DIR, ARCHIVE_READY_FILENAME), "w") as handle:
        json.dump({"sealed": datetime.datetime.now().isoformat(timespec="seconds"),
                   "files": _checksums, "model_sha256": ARTIFACT_VERIFICATION["sha256"],
                   "experiment_id": EVAL_MANIFEST["experiment_id"]}, handle, indent=2)
    shutil.rmtree(ARCHIVE_STAGING, ignore_errors=True)

    print("FINAL MODEL ARCHIVED")
    print("  destination:", FINAL_MODEL_DIR)
    for name in sorted(os.listdir(FINAL_MODEL_DIR)):
        print(f"    {name:32} {os.path.getsize(posixpath.join(FINAL_MODEL_DIR, name)):>12,} bytes")
    print("  model SHA-256:", ARTIFACT_VERIFICATION["sha256"])
    print(f"  BEST epoch {PROVENANCE['best_epoch']} | val_QWK {BEST_METRICS['qwk']:.6f} | "
          f"val_loss {BEST_METRICS['loss']:.6f}")
    print("  LAST was NOT archived as the delivery model; provenance.json only records its path.")
    diag_check("the archive contains exactly the staged files plus the READY marker",
               sorted(os.listdir(FINAL_MODEL_DIR)) == sorted(_staged + [ARCHIVE_READY_FILENAME]))
    diag_check("the original BEST checkpoint is unchanged after archival",
               tree_fingerprint(EVAL_CHECKPOINT_DIR) == CHECKPOINTS_FINGERPRINT_BEFORE)
else:
    print("RUN_POST_RUN_EVALUATION is False -- nothing was archived.")


In [ ]:
# ==== [D12] POST-RUN: final diagnostic report (machine-readable JSON + human-readable Markdown) ====
if RUN_POST_RUN_EVALUATION:
    FINAL_REPORT = {
        "generated": datetime.datetime.now().isoformat(timespec="seconds"),
        "experiment": {
            "experiment_id": EVAL_MANIFEST["experiment_id"], "experiment_dir": EVAL_EXPERIMENT_DIR,
            "created": eval_metadata.get("timestamp"), "git_commit": EVAL_MANIFEST["git_commit_hash"],
            "config_hash": EXPECTED_CONFIG_HASH, "notebook_commit_at_report": SESSION_COMMIT,
            "stop_reason": EVAL_FINISHED, "epochs_completed": last_state.completed_epoch,
            "epoch_cap": EPOCHS,
        },
        "data": {
            "split_manifest": EVAL_MANIFEST["split_manifest"], "split_sha256": EXPECTED_SPLIT_SHA256,
            "split_seed": EVAL_MANIFEST["split_seed"], "total_entries": len(train_entries) + len(val_entries),
            "train": len(train_entries), "validation": len(val_entries),
            "evaluated": len(EVALUATED_IDS), "skipped": len(SKIPPED_VAL_IDS),
            "skipped_ids": SKIPPED_VAL_IDS, "skipped_reason": EVAL_MANIFEST["skipped_reason"],
            "class_counts_evaluated": EVAL_MANIFEST["class_counts_evaluated"],
        },
        "model": {
            "architecture": joint_model.name, "trainable_parameters": ARTIFACT_VERIFICATION["trainable_parameters"],
            "all_weights": int(joint_model.count_params()), "best_epoch": best_state.best_epoch + 1,
            "last_epoch": last_state.completed_epoch, "precision_policy": ARTIFACT_VERIFICATION["precision_policy"],
            "optimizer": ARTIFACT_VERIFICATION["optimizer_type"],
            "optimizer_wrapper": ARTIFACT_VERIFICATION["optimizer_wrapper"],
            "initial_learning_rate": LEARNING_RATE,
            "corn": {"num_grades": corn.NUM_GRADES, "num_thresholds": corn.NUM_THRESHOLDS,
                     "decode": "count(P(y>k) > 0.5)"},
            "racaf": "trainable, in the joint graph", "adaptive_branch_fusion": "trainable (Stage 05)",
            "swin": "trainable (Stage 06, dual scale)",
        },
        "performance": {
            "best": {k: BEST_METRICS[k] for k in ("loss", "qwk", "qwk_sklearn", "accuracy", "balanced_accuracy",
                                                  "mae", "macro_f1", "weighted_f1", "macro_precision",
                                                  "weighted_precision", "macro_recall", "weighted_recall")},
            "qwk_bootstrap_ci_95": [globals().get("CI_LOW"), globals().get("CI_HIGH")],
            "best_p8": globals().get("BEST_P8"), "best_full": globals().get("BEST_FULL"),
            "per_class": BEST_METRICS["per_class"],
            "prior_only_reference_loss_on_these_batches": BEST_RESULT["prior_loss"],
            "baseline_0_best_val_qwk": globals().get("BASELINE_0_BEST_VAL_QWK"),
        },
        "error_analysis": {
            "confusion_matrix": BEST_METRICS["confusion"], "ordinal_errors": ERROR_SUMMARY,
            "off_diagonal_pairs": OFF_DIAGONAL,
            "prediction_histogram": BEST_METRICS["prediction_histogram"],
            "true_histogram": BEST_METRICS["true_histogram"],
            "per_class_weakest_recall": min(GRADES, key=lambda g: BEST_METRICS["per_class"][g]["recall"]),
            "per_class_weakest_f1": min(GRADES, key=lambda g: BEST_METRICS["per_class"][g]["f1"]),
        },
        "corn_diagnostics": CALIBRATION_DIAGNOSTICS,
        "best_vs_last": {"comparison": BEST_VS_LAST["comparison"],
                         "best_epoch": best_state.best_epoch + 1, "last_epoch": last_state.completed_epoch},
        "checkpoint": {
            "best_sha256": ARTIFACT_VERIFICATION["sha256"], "best_checkpoint_path": BEST_DIR,
            "last_checkpoint_path": LAST_DIR, "archived_final_model": globals().get("FINAL_MODEL_DIR"),
            "artifact_verification_passed": D10_PASSED,
        },
        "checks": {"post_run_R1_R3": {name: {"ok": ok, "detail": str(detail)}
                                      for name, (ok, detail) in POST_RUN_CHECKS.items()},
                   "diagnostics_D2_D11": {name: {"ok": ok, "detail": detail}
                                          for name, (ok, detail) in DIAG_CHECKS.items()}},
        "outputs": sorted(set(DIAG_OUTPUTS)),
    }
    _all_ok = (all(ok for ok, _ in POST_RUN_CHECKS.values()) and all(ok for ok, _ in DIAG_CHECKS.values()))
    FINAL_REPORT["verdict"] = {
        "training_completed": bool(EVAL_FINISHED),
        "technical_validation_passed": _all_ok,
        "model_selection": f"BEST = epoch {best_state.best_epoch + 1} by maximum val_QWK "
                           f"({BEST_METRICS['qwk']:.6f} recomputed); LAST = epoch "
                           f"{last_state.completed_epoch}, kept only as the resume trajectory",
        "pre_registered_comparison": globals().get("VERDICT"),
        "diagnostic_observations": [
            f"accuracy {BEST_METRICS['accuracy']:.4f}, balanced accuracy {BEST_METRICS['balanced_accuracy']:.4f}, "
            f"MAE {BEST_METRICS['mae']:.4f} over {len(EVALUATED_IDS)} images",
            f"{ERROR_SUMMARY['adjacent_share_of_errors']:.1f}% of errors are adjacent-grade; "
            f"{ERROR_SUMMARY['non_adjacent_errors']} error(s) are two or more grades away",
            f"weakest recall: grade {min(GRADES, key=lambda g: BEST_METRICS['per_class'][g]['recall'])}; "
            f"weakest F1: grade {min(GRADES, key=lambda g: BEST_METRICS['per_class'][g]['f1'])}",
            f"Brier {CALIBRATION_DIAGNOSTICS['brier_score']:.4f}, ECE (decode convention) "
            f"{CALIBRATION_DIAGNOSTICS['ece_decode_convention']:.4f}",
        ],
        "limitations": [
            "One APTOS 2019 validation split from this project's own manifest: not an external or "
            "held-out test set, and not evidence of generalization to other cameras, sites or populations.",
            "No clinical claim of any kind is supported by these numbers.",
            f"{len(SKIPPED_VAL_IDS)} validation image(s) are excluded because Stage 03 finds no field of "
            "view; the metrics describe the remaining population.",
            "Class support is very uneven, so per-class metrics for the rarer grades rest on few images.",
            "Validation ran under mixed_float16: logits within ~0.1% of a decode boundary can flip.",
            "Model selection used this same validation split, so these figures are optimistic for it.",
        ],
        "future_work": [
            "Evaluate on a genuinely held-out or external test set before any performance claim.",
            "Repeat the run with a different seed to separate the learning-rate effect from run-to-run variation.",
            "Revisit the known augmentation defect (JOINT_TRAINING_ARCHITECTURE.md Sec 49) before further tuning.",
            "Consider class-imbalance handling if the rarer grades matter for the intended use.",
        ],
    }
    REPORT_JSON = save_json("final_diagnostic_report.json", FINAL_REPORT)

    _v = FINAL_REPORT["verdict"]
    _lines = [
        f"# Final diagnostic report -- {EVAL_MANIFEST['experiment_id']}", "",
        f"Generated {FINAL_REPORT['generated']} | config hash `{EXPECTED_CONFIG_HASH}` | "
        f"git `{str(EVAL_MANIFEST['git_commit_hash'])[:12]}`", "",
        "## Experiment", "",
        f"- Directory: `{EVAL_EXPERIMENT_DIR}`",
        f"- Epochs completed: {last_state.completed_epoch} of a {EPOCHS}-epoch cap ({EVAL_FINISHED})",
        f"- BEST epoch {best_state.best_epoch + 1}; LAST epoch {last_state.completed_epoch}", "",
        "## Data", "",
        f"- Split: {len(train_entries)} train / {len(val_entries)} validation "
        f"(`{posixpath.basename(EVAL_MANIFEST['split_manifest'])}`, sha256 `{EXPECTED_SPLIT_SHA256[:16]}...`)",
        f"- Evaluated: {len(EVALUATED_IDS)}; skipped (empty field of view): {len(SKIPPED_VAL_IDS)} "
        f"{SKIPPED_VAL_IDS}", "",
        "## Model", "",
        f"- `{joint_model.name}`, {ARTIFACT_VERIFICATION['trainable_parameters']:,} trainable parameters, "
        f"{ARTIFACT_VERIFICATION['precision_policy']}",
        f"- Stage 05 AdaptiveBranchFusion, Stage 06 dual-scale Swin, Stage 07, RACAF and CORN all trainable; "
        "Stage 03/04 frozen outside the graph", "",
        "## Performance (BEST, validation split)", "",
        "| metric | value |", "|---|---|",
    ]
    for name in ("qwk", "accuracy", "balanced_accuracy", "macro_f1", "weighted_f1", "mae", "loss"):
        _lines.append(f"| {name} | {BEST_METRICS[name]:.6f} |")
    if FINAL_REPORT["performance"]["qwk_bootstrap_ci_95"][0] is not None:
        _lines.append(f"| QWK bootstrap 95% CI | [{CI_LOW:.4f}, {CI_HIGH:.4f}] |")
    _lines += ["", "| grade | support | precision | recall | F1 |", "|---|---|---|---|---|"]
    for grade in GRADES:
        row = BEST_METRICS["per_class"][grade]
        _lines.append(f"| {grade} | {row['support']} | {row['precision']:.4f} | {row['recall']:.4f} | "
                      f"{row['f1']:.4f} |")
    _lines += ["", "## Error analysis", "",
               f"- Correct {ERROR_SUMMARY['correct']} / {ERROR_SUMMARY['evaluated']}; adjacent-grade errors "
               f"{ERROR_SUMMARY['adjacent_errors']}; two or more grades away {ERROR_SUMMARY['non_adjacent_errors']}; "
               f"maximum distance {ERROR_SUMMARY['max_error_distance']}",
               f"- Prediction histogram {list(map(int, BEST_METRICS['prediction_histogram']))} against true "
               f"{list(map(int, BEST_METRICS['true_histogram']))}", "",
               "## CORN diagnostics", "",
               f"- Brier {CALIBRATION_DIAGNOSTICS['brier_score']:.4f}; ECE (decode) "
               f"{CALIBRATION_DIAGNOSTICS['ece_decode_convention']:.4f}; ECE (argmax) "
               f"{CALIBRATION_DIAGNOSTICS['ece_argmax_convention']:.4f}",
               "- Threshold calibration, predicted vs observed: "
               + "; ".join(f"P(y>{row['threshold']}) {row['mean_predicted']:.3f} vs {row['empirical_rate']:.3f}"
                           for row in CALIBRATION_DIAGNOSTICS["threshold_calibration"]), "",
               "## Checkpoint", "",
               f"- BEST `{BEST_DIR}` (sha256 `{ARTIFACT_VERIFICATION['sha256'][:16]}...`)",
               f"- Archived final model: `{globals().get('FINAL_MODEL_DIR')}`",
               f"- Artifact verification: {'PASS' if D10_PASSED else 'FAIL'}", "",
               "## Verdict", "",
               f"- Training completed: {_v['training_completed']} ({EVAL_FINISHED})",
               f"- Technical validation: {'PASS' if _v['technical_validation_passed'] else 'FAIL'}",
               f"- Model selection: {_v['model_selection']}",
               f"- Pre-registered comparison ([R4]): {_v['pre_registered_comparison']}", "",
               "### Observations", ""] + [f"- {line}" for line in _v["diagnostic_observations"]] + [
               "", "### Limitations", ""] + [f"- {line}" for line in _v["limitations"]] + [
               "", "### Future work", ""] + [f"- {line}" for line in _v["future_work"]] + [""]
    REPORT_MARKDOWN = diag_path("final_diagnostic_report.md")
    with open(REPORT_MARKDOWN, "w") as handle:
        handle.write("\n".join(_lines))
    DIAG_OUTPUTS.append(REPORT_MARKDOWN)
    print("\n".join(_lines))
    print(f"\nreport written: {REPORT_JSON}\n                {REPORT_MARKDOWN}")
    try:
        from IPython.display import Markdown, display
        display(Markdown("\n".join(_lines)))
    except Exception as _error:   # noqa: BLE001 -- the plain text above is the report
        print("(rendered Markdown unavailable:", _error, ")")
else:
    print("RUN_POST_RUN_EVALUATION is False -- no report was generated.")


In [ ]:
# ==== [D13] POST-RUN: final archived-artifact integrity check ====
if RUN_POST_RUN_EVALUATION:
    import joint_cache_staging as jcs

    ARCHIVED_WEIGHTS = posixpath.join(FINAL_MODEL_DIR, ckpt.MODEL_WEIGHTS_FILENAME)
    REQUIRED_ARCHIVE_FILES = {ckpt.MODEL_WEIGHTS_FILENAME, "provenance.json", "checkpoint_state.json",
                              "checkpoint_manifest.json", "experiment_metadata.json",
                              "evaluation_manifest.json", "aptos2019_train_val_split.csv",
                              ARCHIVE_READY_FILENAME}
    _archived_provenance = read_json(posixpath.join(FINAL_MODEL_DIR, "provenance.json"))
    _ready = read_json(posixpath.join(FINAL_MODEL_DIR, ARCHIVE_READY_FILENAME))
    _present = set(os.listdir(FINAL_MODEL_DIR))

    # Rebuild the architecture from scratch and load ONLY the archived weights: this proves the
    # archive alone reproduces the delivery model. Marked diagnostic-dirty so it can never be trained.
    print("rebuilding the production architecture and loading the archived weights ...")
    archived_model = jtm.build_and_compile_joint_model(
        mixed_precision=MIXED_PRECISION,
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE), verbose=0)
    setattr(archived_model, DIAGNOSTIC_DIRTY_ATTRIBUTE, True)
    ckpt.load_model_weights_only(archived_model, ARCHIVED_WEIGHTS)

    _metric = corn.CORNQuadraticWeightedKappa()
    _loss_sum, _samples, _predictions = 0.0, 0, []
    for (stage5, stage6, reliability), grades in POST_RUN_VAL_DS:
        logits = archived_model.predict_on_batch([stage5, stage6, tf.reshape(reliability, (-1, 1))])
        _loss_sum += float(jtm.joint_corn_loss(grades, logits)) * int(grades.shape[0])
        _metric.update_state(grades, logits)
        _predictions.append(corn.decode_logits(np.asarray(logits, dtype=np.float32))["predicted_grade"])
        _samples += int(grades.shape[0])
    ARCHIVED_QWK = float(_metric.result())
    ARCHIVED_LOSS = _loss_sum / _samples
    ARCHIVED_PREDICTIONS = np.concatenate(_predictions).astype(int)

    _cache_now = jcs.verify_local_cache(train_entries + val_entries, cache_dir=LOCAL_CACHE_DIR,
                                        racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
                                        persistent_cache_dir=config.LOCAL_FEATURE_RESULTS_DIR,
                                        persistent_racaf_cache_dir=config.RACAF_RESULTS_DIR)
    _split_now = hashlib.sha256(json.dumps(
        [[list(e) for e in jtd.split_train_val_ids()[0]], [list(e) for e in jtd.split_train_val_ids()[1]]],
        separators=(",", ":")).encode("utf-8")).hexdigest()
    _baseline_failures = verify_baseline_0_unchanged()
    _checkpoints_unchanged = tree_fingerprint(EVAL_CHECKPOINT_DIR) == CHECKPOINTS_FINGERPRINT_BEFORE
    _sha_ok = ckpt.sha256_file(ARCHIVED_WEIGHTS) == ARTIFACT_VERIFICATION["sha256"]
    _unexpected = sorted(_present - REQUIRED_ARCHIVE_FILES)

    diag_check("archived weights reproduce the verified BEST val_QWK (|diff| < 1e-6)",
               abs(ARCHIVED_QWK - BEST_METRICS["qwk"]) < 1e-6,
               f"{ARCHIVED_QWK:.8f} vs {BEST_METRICS['qwk']:.8f}")
    diag_check("archived weights reproduce the verified BEST val_loss (|diff| < 1e-4)",
               abs(ARCHIVED_LOSS - BEST_METRICS["loss"]) < 1e-4,
               f"{ARCHIVED_LOSS:.8f} vs {BEST_METRICS['loss']:.8f}")
    diag_check("archived model reproduces the per-sample predictions",
               np.array_equal(ARCHIVED_PREDICTIONS, BEST_DIAG["y_pred"]))
    diag_check("archived weights SHA-256 matches the verified BEST", _sha_ok,
               ARTIFACT_VERIFICATION["sha256"][:16] + "...")
    diag_check("the READY marker's recorded model SHA-256 matches",
               _ready.get("model_sha256") == ARTIFACT_VERIFICATION["sha256"])
    diag_check("every required provenance file is present",
               REQUIRED_ARCHIVE_FILES <= _present, sorted(REQUIRED_ARCHIVE_FILES - _present))
    diag_check("the archive holds no unexpected or temporary file", not _unexpected, _unexpected)
    diag_check("provenance records this experiment and BEST epoch",
               _archived_provenance["experiment_id"] == EVAL_MANIFEST["experiment_id"]
               and _archived_provenance["best_epoch"] == best_state.best_epoch + 1)
    diag_check("the original experiment checkpoints are unchanged", _checkpoints_unchanged)
    diag_check("baseline-0 is unchanged", not _baseline_failures, " | ".join(_baseline_failures))
    diag_check("the authoritative split is unchanged", _split_now == EXPECTED_SPLIT_SHA256)
    diag_check("the cache inventory is unchanged since [6]",
               (_cache_now["fully_local"], _cache_now["drive_fallback_required"],
                _cache_now["missing_everywhere"]) == (LOCAL_CACHE_REPORT["fully_local"],
                                                      LOCAL_CACHE_REPORT["drive_fallback_required"],
                                                      LOCAL_CACHE_REPORT["missing_everywhere"]),
               f"fully local {_cache_now['fully_local']}, missing {_cache_now['missing_everywhere']}")

    _failed = [name for name, (ok, _) in DIAG_CHECKS.items() if not ok]
    print("\n" + "=" * 78)
    print(f"FINAL MODEL ARCHIVE:          {'PASS' if not _failed else 'FAIL'}")
    print(f"ARCHIVED BEST QWK:            {ARCHIVED_QWK:.6f} (recomputed from the archive)")
    print(f"ARCHIVED BEST EPOCH:          {best_state.best_epoch + 1}")
    print(f"SHA256:                       {ARTIFACT_VERIFICATION['sha256']}")
    print(f"ORIGINAL CHECKPOINT UNCHANGED: {'PASS' if _checkpoints_unchanged else 'FAIL'}")
    print(f"BASELINE-0 UNCHANGED:         {'PASS' if not _baseline_failures else 'FAIL'}")
    print(f"DATASET UNCHANGED:            {'PASS' if _split_now == EXPECTED_SPLIT_SHA256 else 'FAIL'}")
    print(f"DIAGNOSTIC REPORT:            {REPORT_JSON}")
    print(f"                              {REPORT_MARKDOWN}")
    print(f"FINAL MODEL DIRECTORY:        {FINAL_MODEL_DIR}")
    print("=" * 78)
    if _failed:
        raise RuntimeError(f"FINAL INTEGRITY CHECK FAILED ({len(_failed)}): {_failed}. The archive and "
                           "the experiment were left exactly as they are -- resolve these before "
                           "treating the archived model as final.")
    print(f"All {len(DIAG_CHECKS)} diagnostic checks passed. Run [P3] before disconnecting.")
else:
    print("RUN_POST_RUN_EVALUATION is False -- no archived-artifact check was run.")


## MAINTENANCE (off by default; one task per session; never in a training runtime)

Not part of any training or evaluation session. Set `MAINTENANCE_TASK` in [S], run [S] to [8], then run only that task's cell. Each task records itself in the runtime, so [P2] refuses to train in the same runtime afterwards. [6] still defines the local cache locations these cells use, but does not extract anything in a maintenance session.

| `MAINTENANCE_TASK` | Cell | When to use it |
|---|---|---|
| `"verify_raw_dataset"` | [M1] | After re-uploading APTOS 2019 to Drive, or before [M2]/[M4]: `train.csv`'s row count plus `verify_dataset.verify_image_folder()`'s 50-image decode spot check. Nothing in the cache-backed path reads raw images otherwise. |
| `"precompute_cache"` | [M2] | Only if the Drive cache must be (re)built. Stages the raw dataset, runs Stage 03/04 + RACAF for every uncached split entry into LOCAL SSD (a cheap existence check against the Drive cache skips entries already there -- never a bulk copy, which previously crashed the FUSE mount, §35), then flushes the new entries to Drive. Safe to interrupt and re-run; `CACHE_DIAGNOSTIC_MAX_IMAGES` limits it to a few images first. |
| `"flush_cache_to_drive"` | [M3] | Only right after an interrupted or partially failed [M2]: copies the local entries that are missing on Drive. With the archive extracted it would be ~14,595 Drive existence checks for a guaranteed-zero result, so never otherwise. |
| `"build_cache_archive"` | [M4] | Once, after the Drive cache changes: packs it into the 8 `.tar` shards that [6] stream-extracts. Pays ~1 s per Drive file (per-file staging is latency-bound, not bandwidth-bound); resumable per shard via `ARCHIVE_MAX_SHARDS`. |

The 11 entries with no vessel/lesion/reliability anywhere are the known empty-FOV images. They are reported, never regenerated or fabricated.


In [ ]:
# ==== [M1] MAINTENANCE: verify the raw APTOS 2019 copy on Drive (reads Drive; writes nothing) ====
if MAINTENANCE_TASK == "verify_raw_dataset":
    import csv

    import verify_dataset

    _RUNTIME_NON_TRAINING_WORK.add("maintenance: verify_raw_dataset")
    aptos_raw_dir = config.dataset_raw_dir("APTOS2019")
    train_csv_path = os.path.join(aptos_raw_dir, "train.csv")
    train_image_dir = os.path.join(aptos_raw_dir, "train_images")

    if not os.path.exists(train_csv_path):
        raise FileNotFoundError(
            f"APTOS2019 train.csv not found at {train_csv_path} -- verify Drive is mounted and "
            "APTOS2019_RAW_DIR resolved correctly (see colab_config.APTOS2019_RAW_DIR)."
        )
    with open(train_csv_path, newline="", encoding="utf-8") as handle:
        labeled_row_count = sum(1 for _ in csv.DictReader(handle))
    print(f"APTOS2019 train.csv: {labeled_row_count} labeled rows (expected 3662).")

    verify_dataset.verify_image_folder(train_image_dir, min_images=labeled_row_count)
    print("APTOS2019 train_images/ verified.")
else:
    print("MAINTENANCE_TASK is not 'verify_raw_dataset' -- Drive was not listed and no image was decoded.")


In [ ]:
# ==== [M2] MAINTENANCE: Phase 1 cache precomputation (safe to interrupt and re-run) ====
CACHE_DIAGNOSTIC_MAX_IMAGES = None  # e.g. 5/10/25/50 -- run Phase 1 on only this many images first
FLUSH_TO_DRIVE_AFTER_PHASE1 = True  # persist what Phase 1 wrote as soon as it finishes

if MAINTENANCE_TASK == "precompute_cache":
    import dataset_staging

    _RUNTIME_NON_TRAINING_WORK.add("maintenance: precompute_cache")
    staged_aptos = dataset_staging.stage_dataset(colab_config.APTOS2019_RAW_DIR, "APTOS2019")
    dataset_staging.verify_staged_copy(staged_aptos)
    staged_train_image_dir = os.path.join(staged_aptos.local_dir, "train_images")

    # Canonical RGB shares LOCAL_CACHE_DIR/persistent_cache_dir with Stage 03/04 -- no separate
    # location. Regenerating it (raw read + Stage 02 + a skimage resize) for all 3662 images costs
    # tens of minutes of CPU on every fresh runtime when kept local-only, far more than the marginal
    # Drive cost of persisting it like every other cache entry.
    #
    # Cache hits are checked directly against the persistent Drive cache (existence-only, a
    # cheap stat per file) -- NEVER a bulk content copy. Newly computed entries are written to
    # local SSD here, then flushed to Drive by the block at the end of this cell.
    cache_stats = jtd.precompute_authoritative_joint_caches(
        image_dir=staged_train_image_dir,
        cache_dir=LOCAL_CACHE_DIR,
        racaf_cache_dir=LOCAL_RACAF_CACHE_DIR,
        persistent_cache_dir=config.LOCAL_FEATURE_RESULTS_DIR,
        persistent_racaf_cache_dir=config.RACAF_RESULTS_DIR,
        processed_dir=NO_PRECOMPUTED_STAGE02_DIR,
        max_images=CACHE_DIAGNOSTIC_MAX_IMAGES,
        verbose_diagnostics=CACHE_DIAGNOSTIC_MAX_IMAGES is not None,
    )
    print("Cache precomputation complete:", cache_stats)

    # Phase 1 writes to local SSD only. Persist it now, in the same cell, rather than leaving it
    # to a separate cell that is easy to forget -- /content is wiped on every disconnect, so an
    # unflushed canonical RGB cache means the next fresh runtime regenerates all 3662 entries
    # (raw read + Stage 02 + resize each), which is exactly the cost this cache exists to avoid.
    #
    # Uses the SAME unmodified dataset_staging.sync_missing_files() as [M3], over the SAME
    # directories: vessel, lesion, canonical RGB (all sharing LOCAL_CACHE_DIR) and RACAF
    # reliability. Every copy is atomic -- temp file, size check, rename -- and retries a
    # transient Drive FUSE error with backoff. Only files MISSING on Drive are copied, so existing
    # vessel/lesion/reliability entries are never re-uploaded, rewritten or invalidated, and a
    # re-run after an interruption simply copies whatever is still missing.
    if FLUSH_TO_DRIVE_AFTER_PHASE1:
        flushed_lf, kept_lf, failed_lf = dataset_staging.sync_missing_files(
            LOCAL_CACHE_DIR, config.LOCAL_FEATURE_RESULTS_DIR,
        )
        flushed_racaf, kept_racaf, failed_racaf = dataset_staging.sync_missing_files(
            LOCAL_RACAF_CACHE_DIR, config.RACAF_RESULTS_DIR,
        )
        print(f"Flushed {flushed_lf} Stage 03/04/RGB cache file(s) to Drive "
              f"({kept_lf} were already there) and {flushed_racaf} RACAF file(s) "
              f"({kept_racaf} already there).")
        if failed_lf or failed_racaf:
            print(f"{len(failed_lf) + len(failed_racaf)} file(s) failed to flush and remain "
                  "local-only -- run [M3] (MAINTENANCE_TASK = 'flush_cache_to_drive') to retry just those.")
    else:
        print("FLUSH_TO_DRIVE_AFTER_PHASE1 is False -- the new cache entries are LOCAL ONLY and "
              "will be lost when this runtime is recycled. Run [M3] to persist them.")
else:
    print("MAINTENANCE_TASK is not 'precompute_cache' -- no dataset was staged, no Stage 03/04 "
          "inference was run, no cache was written.")


In [ ]:
# ==== [M3] MAINTENANCE: flush local cache entries to Drive (only after an interrupted/failed [M2]) ====
if MAINTENANCE_TASK == "flush_cache_to_drive":
    import dataset_staging

    _RUNTIME_NON_TRAINING_WORK.add("maintenance: flush_cache_to_drive")
    # Flushes everything Phase 1 wrote locally -- vessel/lesion/RACAF caches AND canonical RGB,
    # since all four share LOCAL_CACHE_DIR/config.LOCAL_FEATURE_RESULTS_DIR. Only genuinely new
    # local entries are copied; anything already on Drive is left alone
    # (JOINT_TRAINING_ARCHITECTURE.md Sec 38).
    flushed_lf, _kept_lf, failed_lf = dataset_staging.sync_missing_files(
        LOCAL_CACHE_DIR, config.LOCAL_FEATURE_RESULTS_DIR)
    flushed_racaf, _kept_racaf, failed_racaf = dataset_staging.sync_missing_files(
        LOCAL_RACAF_CACHE_DIR, config.RACAF_RESULTS_DIR)
    print(f"Flushed {flushed_lf} Stage 03/04/RGB cache files and {flushed_racaf} RACAF cache "
          "files to Drive.")
    if failed_lf or failed_racaf:
        print(f"{len(failed_lf) + len(failed_racaf)} file(s) failed to flush and remain "
              "local-only -- re-run this cell to retry them.")
else:
    print("MAINTENANCE_TASK is not 'flush_cache_to_drive' -- Drive was not walked, stat'ed or written.")


In [ ]:
# ==== [M4] MAINTENANCE: build the cache archive on Drive (once; resumable per shard) ====
ARCHIVE_MAX_SHARDS = None   # limit shards per runtime when building (resumable; finished shards are skipped)
ARCHIVE_COMPRESS = False    # off: the measured ratios came from synthetic data, while gzip (7.7-22 MB/s on
                            # Colab's 2 cores) is a real cost against a transfer that is already latency-bound

if MAINTENANCE_TASK == "build_cache_archive":
    import joint_cache_archive as jca

    _RUNTIME_NON_TRAINING_WORK.add("maintenance: build_cache_archive")
    print("archive dir:", ARCHIVE_DIR)
    build_result = jca.build_archive(
        train_entries + val_entries,
        persistent_cache_dir=config.LOCAL_FEATURE_RESULTS_DIR,
        persistent_racaf_cache_dir=config.RACAF_RESULTS_DIR,
        archive_dir=ARCHIVE_DIR,
        compress=ARCHIVE_COMPRESS,
        max_shards=ARCHIVE_MAX_SHARDS,
    )
    jca.print_build(build_result)
else:
    print("MAINTENANCE_TASK is not 'build_cache_archive' -- nothing was read, written or modified.")


## OPTIONAL DIAGNOSTIC: tiny-subset overfit test (off by default)

Not part of the production path. It has already been run: at LR 1e-3 the model collapsed to a single grade; at LR 1e-4 it PASSED, memorizing the 50-image subset (final inference loss 0.0163, QWK 1.0000, diagonal confusion matrix). It is kept to re-check the pipeline after a future code change.

In a **fresh runtime**, set `RUN_OVERFIT_DIAGNOSTIC = True` in [S], run [S] to [8], then [D1]. It builds and trains its own fresh model (`overfit_model`, seed 1234, LR 1e-4, no augmentation) on 10 TRAIN images per grade with plain `model.fit` -- no `Trainer`, no checkpoint, no Drive write -- marks that model diagnostic-dirty, and proves `experiments/FinalClassification` is unchanged afterwards. It refuses to run in any other kind of session, and [P2] refuses to train in a runtime that ran it.


In [ ]:
# ==== [D1] OPTIONAL DIAGNOSTIC: tiny-subset overfit test -- its own fresh model; writes nothing ====
OVERFIT_PER_GRADE = 10          # 5 grades x 10 = 50 TRAIN images
OVERFIT_SEED = 1234             # the recorded runs' initial weights
OVERFIT_LEARNING_RATE = 1e-4    # the production learning rate (the recorded PASS)
OVERFIT_EPOCHS = 150            # 25 steps per epoch; stops early once the subset is memorized
LOSS_ZERO_THRESHOLD = 0.05      # "loss approached zero" (inference mode)
QWK_OVERFIT_THRESHOLD = 0.95    # "QWK approached 1" (inference mode)
REFERENCE_INITIAL = {"loss": 0.7779, "QWK": 0.0704}   # the recorded untrained starting point
EXPECTED_SUBSET_IDS = None      # optionally paste a previous run's printed id list to enforce it

if RUN_OVERFIT_DIAGNOSTIC:
    import time

    import corn
    import joint_cache_staging as jcs

    if SESSION_ROLE != "overfit diagnostic" or globals().get("_PRODUCTION_TRAINING_LAUNCHED"):
        raise RuntimeError("The overfit diagnostic runs only in its own fresh runtime (RUN_OVERFIT_DIAGNOSTIC alone).")
    if not globals().get("LOCAL_CACHE_VERIFIED"):
        raise RuntimeError("Run [6] first: the local cache has not been verified in this runtime.")
    _RUNTIME_NON_TRAINING_WORK.add("overfit diagnostic")     # [P2] refuses to train in this runtime from now on
    _experiments_before = tree_fingerprint(FINAL_CLASSIFICATION_EXPERIMENTS_DIR)

    # The recorded runs' deterministic subset: the first 10 fully cached TRAIN entries of each grade
    # (train_entries is sorted by id_code).
    _incomplete = {id_code for id_code, _ in jcs.entries_missing_local_cache(
        train_entries, LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR, jtd.STAGE5_IMAGE_SIZE)}
    _usable = [entry for entry in train_entries if entry[0] not in _incomplete]
    overfit_entries = []
    for _grade in range(corn.NUM_GRADES):
        overfit_entries += [entry for entry in _usable if entry[1] == _grade][:OVERFIT_PER_GRADE]
    _subset_ids = [id_code for id_code, _ in overfit_entries]
    assert len(overfit_entries) == OVERFIT_PER_GRADE * corn.NUM_GRADES, "local cache incomplete"
    assert not ({id_code for id_code, _ in val_entries} & set(_subset_ids)), "a validation image is in the subset"
    if EXPECTED_SUBSET_IDS is not None:
        assert _subset_ids == list(EXPECTED_SUBSET_IDS), "the subset differs from EXPECTED_SUBSET_IDS"
    print(f"subset: {len(overfit_entries)} TRAIN images ({OVERFIT_PER_GRADE} per grade), fingerprint "
          f"{hashlib.sha256(','.join(_subset_ids).encode()).hexdigest()[:16]}")
    print("subset ids:", _subset_ids)

    tf.keras.utils.set_random_seed(OVERFIT_SEED)
    overfit_model = jtm.build_and_compile_joint_model(
        mixed_precision=True,
        optimizer=tf.keras.optimizers.Adam(learning_rate=OVERFIT_LEARNING_RATE),  # compile wraps it in a LossScaleOptimizer
    )
    setattr(overfit_model, DIAGNOSTIC_DIRTY_ATTRIBUTE, True)   # training.Trainer.fit() refuses this model
    jtd.check_gpu()                                            # "Memory growth could not be enabled" is expected here
    _vessel_model = jtd.load_vessel_model(jtd.DEFAULT_VESSEL_MODEL_PATH)
    _stage4_model = jtd.racaf.load_frozen_stage4_model()
    _subset_batches = -(-len(overfit_entries) // BATCH_SIZE)

    def _subset_pipeline(shuffle):
        ds = jtd._make_joint_dataset(
            overfit_entries, LOCAL_RAW_IMAGE_DIR, LOCAL_CACHE_DIR, LOCAL_RACAF_CACHE_DIR,
            _vessel_model, _stage4_model, BATCH_SIZE,
            shuffle=shuffle, augment=False, seed=jtd.DEFAULT_SEED,
            processed_dir=NO_PRECOMPUTED_STAGE02_DIR,
            persistent_cache_dir=config.LOCAL_FEATURE_RESULTS_DIR,
            persistent_racaf_cache_dir=config.RACAF_RESULTS_DIR,
        )
        return ds.apply(tf.data.experimental.assert_cardinality(_subset_batches))

    overfit_train_ds = _subset_pipeline(shuffle=True)    # production shuffling, NO augmentation
    overfit_eval_ds = _subset_pipeline(shuffle=False)    # the SAME 50 images, fixed order, inference mode

    initial = overfit_model.evaluate(overfit_eval_ds, return_dict=True, verbose=0)
    print(f"initial (fresh weights, inference mode): loss={initial['loss']:.4f} QWK={initial['QWK']:.4f} | "
          "recorded starting point reproduced: "
          f"{all(abs(initial[k] - REFERENCE_INITIAL[k]) < 2e-3 for k in ('loss', 'QWK'))}")

    class _OverfitProbe(tf.keras.callbacks.Callback):
        def __init__(self):
            super().__init__()
            self.streak = 0

        def on_epoch_begin(self, epoch, logs=None):
            self.batches, self.started = 0, time.time()

        def on_train_batch_end(self, batch, logs=None):
            self.batches += 1

        def on_epoch_end(self, epoch, logs=None):
            print(f"epoch {epoch + 1:3d}/{OVERFIT_EPOCHS} | batches {self.batches} | "
                  f"train-mode loss {logs['loss']:.4f} QWK {logs['QWK']:.4f} | "
                  f"inference-mode (same 50) loss {logs['val_loss']:.4f} QWK {logs['val_QWK']:.4f} | "
                  f"{time.time() - self.started:.0f}s")
            memorized = logs["val_loss"] < LOSS_ZERO_THRESHOLD and logs["val_QWK"] >= 0.999
            self.streak = self.streak + 1 if memorized else 0
            if self.streak >= 3:
                print("Memorized for 3 consecutive epochs -- stopping early.")
                self.model.stop_training = True

    history = overfit_model.fit(
        overfit_train_ds,
        validation_data=overfit_eval_ds,   # the SAME 50 training images in inference mode -- NOT the validation split
        epochs=OVERFIT_EPOCHS, callbacks=[_OverfitProbe()], verbose=0,
    )
    final = overfit_model.evaluate(overfit_eval_ds, return_dict=True, verbose=0)
    _metric = corn.CORNQuadraticWeightedKappa()
    for (stage5, stage6, reliability), grades in overfit_eval_ds:
        _metric.update_state(grades, overfit_model.predict_on_batch(
            [stage5, stage6, tf.reshape(reliability, (-1, 1))]))
    _confusion = _metric.confusion.numpy().astype(np.int64)
    _passed = final["loss"] < LOSS_ZERO_THRESHOLD and final["QWK"] >= QWK_OVERFIT_THRESHOLD

    print(f"\n=== OVERFIT DIAGNOSTIC (lr {OVERFIT_LEARNING_RATE:g}, {len(history.history['loss'])} epochs run) ===")
    print(f"inference loss: initial {initial['loss']:.4f} -> final {final['loss']:.4f} | "
          f"inference QWK: initial {initial['QWK']:.4f} -> final {final['QWK']:.4f}")
    print("final confusion matrix on the subset (rows = true grade, columns = predicted grade):")
    for g in range(corn.NUM_GRADES):
        print(f"true {g} " + "".join(f"{v:>6d}" for v in _confusion[g]))
    print(f"loss approached zero (< {LOSS_ZERO_THRESHOLD}): {final['loss'] < LOSS_ZERO_THRESHOLD} | "
          f"QWK approached 1 (>= {QWK_OVERFIT_THRESHOLD}): {final['QWK'] >= QWK_OVERFIT_THRESHOLD}")
    print("RESULT:", "PASS -- the subset was memorized." if _passed else "FAIL -- the subset was not memorized.")

    _experiments_after = tree_fingerprint(FINAL_CLASSIFICATION_EXPERIMENTS_DIR)
    assert _experiments_after == _experiments_before, (
        "experiments/FinalClassification CHANGED: "
        + describe_fingerprint_change(_experiments_before, _experiments_after))
    print("Isolation: experiments/FinalClassification unchanged. Disconnect this runtime; never train in it.")
else:
    print("RUN_OVERFIT_DIAGNOSTIC is False -- nothing was built, trained or read.")
